# 🏍️ Triple Crown Motocross - 2025 Report

---

## Table of Contents

- [1. Setup & Data Load](#setup-data-load)
- [2. Demographic Analysis](#demographic-analysis)
- [3. Transition Matrices](#transition-matrices)
- [4. Fatigue Effects](#fatigue-effects)
- [5. Rider Rivalries](#rider-rivalries)
- [6. Points Race](#points-race)
- [7. From the Holeshot to the Podium](#from-the-holeshot-to-the-podium)
- [8. Lap Consistency](#lap-consistency)
- [9. Lap Comparisons to Race Leader](#lap-comparisons-to-race-leader)
- [10. Place by Moto Quartile](#place-by-moto-quartile)
- [11. Standardizing lap times](#standardizing-lap-times)
- [12. Traffic Effects](#traffic-effects)
- [13. Gender Gap](#gender-gap)
- [14. Prize Money](#prize-money)
- [15. Weather Effects](#weather-effects)
- [16. Fall Detection](#fall-detection)
- [17. Win Probability Model](#win-probability-model)
- [18. Rider Profiles](#rider-profiles)
- [19. Extensions](#extensions)


<a id="setup-data-load"></a>
## 1. Setup & Data Load


Check out the [Python scraper file](www.github.com) to see how the raw lap time and place data was pulled from Trackside's website ([2024](https://resultsmx.com/cmrc-archive-2024/) and [2025](https://cmrc.tracksideresults.com/)). Slight edits to rider_location (and its associated rider_city, rider_state, and rider_country) were made to improve accuracy and fill in missing cells.

In [1]:
import pandas as pd
import numpy as np

FILE = "tcmx_laptimes.xlsx"

# Read file, ensure number stays as string to preserve any letter suffixes
df = pd.read_excel(FILE, dtype={"number": str})

# ── Categories ────────────────────────────────────────────────────────────────
cat_cols = ["race_id", "class", "track", "manufacturer",
            "rider_state", "rider_country"]
for col in cat_cols:
    df[col] = df[col].astype("category")

# ── Strings ───────────────────────────────────────────────────────────────────
str_cols = ["track_location", "name", "number", "rider_location", "rider_city"]
for col in str_cols:
    df[col] = df[col].astype(str).str.strip()

# ── Integers (nullable Int64 for columns with NaN) ───────────────────────────
nullable_int_cols = ["finish_position", "place", "lap"]
for col in nullable_int_cols:
    df[col] = pd.array(df[col], dtype="Int64")

# ── Integers (standard int64, no nulls) ──────────────────────────────────────
int_cols = ["year", "round", "moto"]
for col in int_cols:
    df[col] = df[col].astype("int64")

# ── Floats ────────────────────────────────────────────────────────────────────
float_cols = ["lap_time", "behind_time"]
for col in float_cols:
    df[col] = df[col].astype(float)
df["behind_time"] = df["behind_time"].fillna(0)

# ── Date ──────────────────────────────────────────────────────────────────────
df["date"] = pd.to_datetime(df["date"])

# ── Normalise class label for display ─────────────────────────────────────────
CLASS_LABELS = {450: "450", 250: "250", "WMX": "WMX"}
df["class_label"] = df["class"].map(CLASS_LABELS).fillna(df["class"].astype(str)).astype("category")

MASTER_OUTPUT_PATH = "tcmx_24-25_master_dataset.csv"

TCMX_POINTS = {1: 25, 2: 22, 3: 20, 4: 18, 5: 16, 6: 15, 7: 14, 8: 13, 9: 12, 10: 11,
              11: 10, 12: 9, 13: 8, 14: 7, 15: 6, 16: 5, 17: 4, 18: 3, 19: 2, 20: 1}

# ── State abbreviation mapping (applied before export) ───────────────────────
STATE_ABBREV = {
    "Alberta": "AB", "British Columbia": "BC", "Manitoba": "MB",
    "New Brunswick": "NB", "Nova Scotia": "NS", "Ontario": "ON",
    "Quebec": "QC", "Saskatchewan": "SK",
    "California": "CA", "Colorado": "CO", "Connecticut": "CT",
    "Florida": "FL", "Georgia": "GA", "Idaho": "ID", "Illinois": "IL",
    "Iowa": "IA", "Maine": "ME", "Maryland": "MD", "Massachusetts": "MA",
    "Michigan": "MI", "Minnesota": "MN", "Nebraska": "NE", "Nevada": "NV",
    "New Hampshire": "NH", "New Jersey": "NJ", "New York": "NY",
    "North Carolina": "NC", "North Dakota": "ND", "Ohio": "OH",
    "Pennsylvania": "PA", "Rhode Island": "RI", "South Carolina": "SC",
    "Tennessee": "TN", "Texas": "TX", "Utah": "UT", "Vermont": "VT",
    "Washington": "WA", "West Virginia": "WV", "Wisconsin": "WI",
    "Wyoming": "WY", "Bay of Plenty": "BAY", "Queensland": "QLD",
}
df["rider_state"] = df["rider_state"].astype(str).map(STATE_ABBREV)

# ── Build derived columns directly on df ─────────────────────────────────────
moto_max = (
    df.groupby("race_id", observed=True)["lap"]
    .max()
    .rename("max_lap")
    .reset_index()
)
df = df.merge(moto_max, on="race_id", how="left")

# ── % of moto completed ───────────────────────────────────────────────────────
df["frac_completed"] = df["lap"] / df["max_lap"]
df["frac_remaining"] = 1 - df["frac_completed"]

# ── Cumulative lap time per rider per moto ────────────────────────────────────
df = df.sort_values(["race_id", "name", "lap"])
df["cumulative_time"] = df.groupby(
    ["race_id", "name"], observed=True
)["lap_time"].cumsum()

# ── Prev / next lap place & place gain/loss ───────────────────────────────────
df["prev_place"] = df.groupby(
    ["race_id", "name"], observed=True
)["place"].shift(1)
df["next_place"] = df.groupby(
    ["race_id", "name"], observed=True
)["place"].shift(-1)
df["place_change"] = df["prev_place"] - df["place"]  # positive = gained places

# ── Pairwise gaps (time to place ahead / behind) ──────────────────────────────
df = df.sort_values(["race_id", "lap", "place"])

df["prev_cumulative_time"] = df.groupby(
    ["race_id", "lap"], observed=True
)["cumulative_time"].shift(1)
df["next_cumulative_time"] = df.groupby(
    ["race_id", "lap"], observed=True
)["cumulative_time"].shift(-1)

df["gap_ahead"] = df["cumulative_time"] - df["prev_cumulative_time"]
df["gap_behind"] = df["next_cumulative_time"] - df["cumulative_time"]

df = df.drop(columns=["prev_cumulative_time", "next_cumulative_time"])

# ── Within-class z-score per lap per moto ────────────────────────────────────
lap_stats = (
    df.groupby(["class_label", "race_id", "lap"], observed=True)["lap_time"]
    .agg(mean_lt="mean", std_lt="std")
    .reset_index()
)
df = df.merge(lap_stats, on=["class_label", "race_id", "lap"], how="left")
df["z_score"] = np.where(
    df["std_lt"].notna() & (df["std_lt"] > 0),
    (df["lap_time"] - df["mean_lt"]) / df["std_lt"],
    np.nan
)
df = df.drop(columns=["mean_lt", "std_lt"])

# ── % off best lap (vs fastest lap of that class/moto/lap) ───────────────────
lap_best = (
    df.groupby(["class_label", "race_id", "lap"], observed=True)["lap_time"]
    .min()
    .rename("best_lap_time")
    .reset_index()
)
df = df.merge(lap_best, on=["class_label", "race_id", "lap"], how="left")
df["pct_off_best"] = np.where(
    df["best_lap_time"] > 0,
    (df["lap_time"] - df["best_lap_time"]) / df["best_lap_time"] * 100,
    np.nan
)
df = df.drop(columns=["best_lap_time"])

# ── Points ────────────────────────────────────────────────────────────────────
df["points"] = df["finish_position"].apply(
    lambda x: TCMX_POINTS.get(int(x), 0) if pd.notna(x) and x <= 20 else 0
)

# ── Traffic state ─────────────────────────────────────────────────────────────
TRAFFIC_THRESHOLD = 2.0

close_ahead = (
        df["gap_ahead"].notna() &
        (df["gap_ahead"] <= TRAFFIC_THRESHOLD) &
        (df["gap_ahead"] >= 0)
)
free_ahead = (
        (df["place"] == 1) |
        (df["gap_ahead"].notna() & (df["gap_ahead"] > TRAFFIC_THRESHOLD))
)
close_behind = (
        df["gap_behind"].notna() &
        (df["gap_behind"] <= TRAFFIC_THRESHOLD) &
        (df["gap_behind"] >= 0)
)
free_behind = (
        df["gap_behind"].isna() |
        (df["gap_behind"] > TRAFFIC_THRESHOLD)
)

df["traffic_state"] = "Unknown"
df.loc[close_ahead & close_behind, "traffic_state"] = "Sandwiched"
df.loc[close_ahead & free_behind, "traffic_state"] = "Chasing"
df.loc[free_ahead & close_behind, "traffic_state"] = "Being Chased"
df.loc[free_ahead & free_behind, "traffic_state"] = "Clear Air"

# ── Round numeric columns ─────────────────────────────────────────────────────
for col in ["z_score", "pct_off_best", "gap_ahead", "gap_behind",
            "cumulative_time", "frac_completed", "frac_remaining",
            "place_change"]:
    if col in df.columns:
        df[col] = df[col].round(4)

# ── Select and order final columns ────────────────────────────────────────────
FINAL_COLS = [
    # Identity
    "race_id", "year", "round", "moto", "date",
    "track", "track_location", "class_label", "name", "number", "manufacturer",
    # Demographics
    "rider_state", "rider_country", "rider_location",
    # Lap data
    "lap", "lap_time", "place", "behind_time", "finish_position", "points",
    # Place movement
    "prev_place", "next_place", "place_change",
    # Race position
    "frac_completed", "frac_remaining",
    # Pace metrics
    "z_score", "pct_off_best",
    # Time gaps
    "cumulative_time", "gap_ahead", "gap_behind",
    # Traffic
    "traffic_state",
]

FINAL_COLS = [c for c in FINAL_COLS if c in df.columns]
df = df[FINAL_COLS].copy()

# ── Export ────────────────────────────────────────────────────────────────────
df.to_csv(MASTER_OUTPUT_PATH, index=False)

print(f"Loaded {len(df):,} rows")
print(df.dtypes)

Loaded 31,997 rows
race_id                  category
year                        int64
round                       int64
moto                        int64
date               datetime64[us]
track                    category
track_location                str
class_label              category
name                          str
number                        str
manufacturer             category
rider_state                   str
rider_country            category
rider_location                str
lap                         Int64
lap_time                  float64
place                       Int64
behind_time               float64
finish_position             Int64
points                      int64
prev_place                  Int64
next_place                  Int64
place_change                Int64
frac_completed            Float64
frac_remaining            Float64
z_score                   float64
pct_off_best              float64
cumulative_time           float64
gap_ahead                 flo

In [2]:
overview = {
    "Total lap rows": f"{len(df):,}",
    "Unique races": df["race_id"].nunique(),
    "Unique riders": df["name"].nunique(),
    "Classes": sorted(df["class_label"].unique()),
    "Years": sorted(int(x) for x in df["year"].unique()),
    "Tracks": df["track"].nunique(),
}
for k, v in overview.items():
    print(f"  {k:<22}: {v}")

  Total lap rows        : 31,997
  Unique races          : 98
  Unique riders         : 364
  Classes               : ['250', '450', 'WMX']
  Years                 : [2024, 2025]
  Tracks                : 10


<a id="demographic-analysis"></a>
## 2. Demographic Analysis
- Breakdown by province/state
- Percent of returning riders
- Manufacturer success

In [3]:
import pandas as pd

PROV_ABBREV_TO_NAME = {
    "AB": "Alberta", "BC": "British Columbia", "MB": "Manitoba",
    "NB": "New Brunswick", "NL": "Newfoundland and Labrador",
    "NT": "Northwest Territories", "NS": "Nova Scotia", "NU": "Nunavut",
    "ON": "Ontario", "PE": "Prince Edward Island", "QC": "Quebec",
    "SK": "Saskatchewan", "YT": "Yukon"
}
US_ABBREV_TO_NAME = {
    "AL": "Alabama", "AK": "Alaska", "AZ": "Arizona", "AR": "Arkansas",
    "CA": "California", "CO": "Colorado", "CT": "Connecticut", "DE": "Delaware",
    "FL": "Florida", "GA": "Georgia", "HI": "Hawaii", "ID": "Idaho",
    "IL": "Illinois", "IN": "Indiana", "IA": "Iowa", "KS": "Kansas",
    "KY": "Kentucky", "LA": "Louisiana", "ME": "Maine", "MD": "Maryland",
    "MA": "Massachusetts", "MI": "Michigan", "MN": "Minnesota",
    "MS": "Mississippi", "MO": "Missouri", "MT": "Montana", "NE": "Nebraska",
    "NV": "Nevada", "NH": "New Hampshire", "NJ": "New Jersey",
    "NM": "New Mexico", "NY": "New York", "NC": "North Carolina",
    "ND": "North Dakota", "OH": "Ohio", "OK": "Oklahoma", "OR": "Oregon",
    "PA": "Pennsylvania", "RI": "Rhode Island", "SC": "South Carolina",
    "SD": "South Dakota", "TN": "Tennessee", "TX": "Texas", "UT": "Utah",
    "VT": "Vermont", "VA": "Virginia", "WA": "Washington",
    "WV": "West Virginia", "WI": "Wisconsin", "WY": "Wyoming",
    "DC": "District of Columbia"
}
ABBREV_TO_NAME = {**PROV_ABBREV_TO_NAME, **US_ABBREV_TO_NAME}
NA_STATES = set(ABBREV_TO_NAME.keys())

# ── Riders geo base ───────────────────────────────────────────────────────────
riders_geo = (
    df.drop_duplicates(subset=["name", "class_label"])
    [["name", "class_label", "rider_state", "rider_country"]]
    .copy()
)

class_totals = riders_geo.groupby("class_label", observed=True)["name"].nunique()

# ── Per-class summary tables ──────────────────────────────────────────────────
for cls in ["450", "250", "WMX"]:
    cls_riders = riders_geo[riders_geo["class_label"] == cls]
    total = int(class_totals[cls])

    na_riders = cls_riders[cls_riders["rider_state"].isin(NA_STATES)]
    intl_riders = cls_riders[~cls_riders["rider_state"].isin(NA_STATES)]

    state_counts = (
        na_riders.groupby("rider_state", observed=True)
        .size()
        .reset_index(name="Riders")
        .sort_values("Riders", ascending=False)
    )
    state_counts["Pct"] = (state_counts["Riders"] / total * 100).round(1)
    state_counts["Pct"] = state_counts["Pct"].astype(str) + "%"
    state_counts["State/Province"] = state_counts["rider_state"].map(ABBREV_TO_NAME)
    state_counts = state_counts[["State/Province", "Riders", "Pct"]]

    print(f"\n{'=' * 40}")
    print(f"  {cls} CLASS  —  N = {total} unique riders")
    print(f"{'=' * 40}")
    print(state_counts.head(15).to_string(index=False))

    if not intl_riders.empty:
        intl_summary = (
            intl_riders.groupby("rider_country", observed=True)
            .size()
            .reset_index(name="Riders")
            .sort_values("Riders", ascending=False)
            .rename(columns={"rider_country": "Country"})
        )
        if not intl_summary.empty:
            print(f"\n  {cls} — Riders outside North America:")
            print(intl_summary.to_string(index=False))
        else:
            print(f"\n  {cls} — All riders are based in North America")
    else:
        print(f"\n  {cls} — All riders are based in North America")


  450 CLASS  —  N = 130 unique riders
  State/Province  Riders   Pct
         Ontario      23 17.7%
          Quebec      13 10.0%
British Columbia      12  9.2%
         Alberta      10  7.7%
        Michigan       6  4.6%
      Washington       5  3.8%
       Wisconsin       4  3.1%
      California       4  3.1%
   Massachusetts       4  3.1%
     Connecticut       3  2.3%
        Nebraska       3  2.3%
        New York       3  2.3%
        Maryland       3  2.3%
     Nova Scotia       3  2.3%
   New Brunswick       3  2.3%

  450 — Riders outside North America:
    Country  Riders
  Australia       1
    Denmark       1
     France       1
New Zealand       1

  250 CLASS  —  N = 167 unique riders
  State/Province  Riders   Pct
         Ontario      35 21.0%
          Quebec      19 11.4%
         Alberta      18 10.8%
British Columbia      15  9.0%
        Manitoba      10  6.0%
      Washington       7  4.2%
     Connecticut       6  3.6%
    Pennsylvania       5  3.0%
   Massa

In [4]:
rider_motos = (
    df.drop_duplicates(subset=["race_id", "name"])
    .groupby("name", observed=True)
    .agg(
        motos_entered=("race_id", "count"),
        total_points=("points", "sum"),
    )
    .reset_index()
    .sort_values(["motos_entered", "total_points"], ascending=[False, False])
)
rider_motos["total_points"] = rider_motos["total_points"].astype(int)

print(f"\n{'='*50}")
print(f"  Top 25 by Motos Entered — All Classes")
print(f"{'='*50}")
top25 = rider_motos.head(25).reset_index(drop=True)
top25.index += 1
display(
    top25[["name", "motos_entered", "total_points"]]
    .rename(columns={"name": "Rider", "motos_entered": "Motos Entered", "total_points": "Total Points"})
)


  Top 25 by Motos Entered — All Classes


,Rider,Motos Entered,Total Points
1,Kaylie Kayer,33,675
2,Preston Kilroy,32,661
3,Katrine Ferguson,32,602
4,Tanner Ward,32,537
5,Quinn Amyotte,32,470
6,Daniel Elmore,32,369
7,Tanner Scott,32,287
8,Clayton Schmucki,32,239
9,Sebastien Racine,31,492
10,Evan Stice,31,247


In [5]:
for cls in ["450", "250", "WMX"]:
    riders_by_year = (
        df.drop_duplicates(subset=["name", "class_label", "year"])
    )
    y2024 = set(riders_by_year[(riders_by_year["year"] == 2024) & (riders_by_year["class_label"] == cls)]["name"])
    y2025 = set(riders_by_year[(riders_by_year["year"] == 2025) & (riders_by_year["class_label"] == cls)]["name"])

    returned = y2024 & y2025
    pct      = len(returned) / len(y2024) * 100 if y2024 else 0

    print(f"\n  {cls}")
    print(f"    2024 riders      : {len(y2024)}")
    print(f"    2025 riders      : {len(y2025)}")
    print(f"    Returned in 2025 : {len(returned)}  ({pct:.1f}% of 2024 field)")
    print(f"    New in 2025      : {len(y2025 - y2024)}")
    print(f"    Didn't return    : {len(y2024 - y2025)}")


  450
    2024 riders      : 83
    2025 riders      : 78
    Returned in 2025 : 31  (37.3% of 2024 field)
    New in 2025      : 47
    Didn't return    : 52

  250
    2024 riders      : 102
    2025 riders      : 106
    Returned in 2025 : 41  (40.2% of 2024 field)
    New in 2025      : 65
    Didn't return    : 61

  WMX
    2024 riders      : 66
    2025 riders      : 53
    Returned in 2025 : 30  (45.5% of 2024 field)
    New in 2025      : 23
    Didn't return    : 36


In [6]:
all_mfr = sorted(df["manufacturer"].dropna().astype(str).unique())
print(f"All manufacturers in dataset: {', '.join(all_mfr)}\n")

# ── 2025 only base ────────────────────────────────────────────────────────────
mfr_base_2025 = (
    df[df["year"] == 2025]
    .drop_duplicates(subset=["name", "class_label"])
    .dropna(subset=["manufacturer"])
)

# ── Total riders by manufacturer by class (2025) ──────────────────────────────
mfr_riders = (
    mfr_base_2025.groupby(["class_label", "manufacturer"], observed=True)
    .agg(riders=("name", "nunique"))
    .reset_index()
)
mfr_riders = mfr_riders[mfr_riders["riders"] >= 3].copy()

for cls in ["450", "250", "WMX"]:
    print(f"\n{'=' * 45}")
    print(f"  Riders by Manufacturer (2025) — {cls}")
    print(f"{'=' * 45}")
    display(
        mfr_riders[mfr_riders["class_label"] == cls]
        .drop(columns="class_label")
        .sort_values("riders", ascending=False)
        .reset_index(drop=True)
        .rename(columns={"manufacturer": "Manufacturer", "riders": "Riders"})
    )

# ── Top 10 by cumulative points in 2025, manufacturer breakdown ───────────────
mfr_2025 = (
    df[df["year"] == 2025]
    .drop_duplicates(subset=["name", "class_label"])
    [["name", "class_label", "manufacturer"]]
    .dropna(subset=["manufacturer"])
)
points_2025 = (
    df[df["year"] == 2025]
    .drop_duplicates(subset=["race_id", "name", "class_label"])
    .groupby(["name", "class_label"], observed=True)
    .agg(total_points=("points", "sum"))
    .reset_index()
    .sort_values(["class_label", "total_points"], ascending=[True, False])
)
top10_mfr = (
    points_2025.groupby("class_label", observed=True)
    .head(10)
    .merge(mfr_2025, on=["name", "class_label"], how="left")
)

for cls in ["450", "250", "WMX"]:
    subset = top10_mfr[top10_mfr["class_label"] == cls]
    mfr_counts = (
        subset.groupby("manufacturer", observed=True)
        .size()
        .reset_index(name="riders_in_top10")
    )
    mfr_counts["pct"] = (mfr_counts["riders_in_top10"] / len(subset) * 100).round(1)
    print(f"\n{'=' * 50}")
    print(f"  Manufacturer % of Top 10 (2025 Points) — {cls}")
    print(f"{'=' * 50}")
    display(
        mfr_counts.sort_values("riders_in_top10", ascending=False)
        .reset_index(drop=True)
        .rename(columns={
            "manufacturer": "Manufacturer",
            "riders_in_top10": "Riders in Top 10",
            "pct": "% of Top 10",
        })
    )

All manufacturers in dataset: Gas Gas, Honda, Husqvarna, KTM, Kawasaki, Stark, Suzuki, Triumph, Yamaha


  Riders by Manufacturer (2025) — 450


,Manufacturer,Riders
0,Yamaha,24
1,KTM,19
2,Honda,12
3,Kawasaki,11
4,Gas Gas,5
5,Husqvarna,3



  Riders by Manufacturer (2025) — 250


,Manufacturer,Riders
0,KTM,31
1,Yamaha,31
2,Husqvarna,11
3,Gas Gas,9
4,Honda,9
5,Kawasaki,9
6,Triumph,6



  Riders by Manufacturer (2025) — WMX


...

,Manufacturer,Riders in Top 10,% of Top 10
0,Honda,4,40.0
1,KTM,2,20.0
2,Gas Gas,1,10.0
3,Husqvarna,1,10.0
4,Kawasaki,1,10.0
5,Yamaha,1,10.0



  Manufacturer % of Top 10 (2025 Points) — 250


,Manufacturer,Riders in Top 10,% of Top 10
0,Gas Gas,2,20.0
1,Honda,2,20.0
2,KTM,2,20.0
3,Yamaha,2,20.0
4,Kawasaki,1,10.0
5,Triumph,1,10.0



  Manufacturer % of Top 10 (2025 Points) — WMX


,Manufacturer,Riders in Top 10,% of Top 10
0,Husqvarna,4,40.0
1,KTM,3,30.0
2,Kawasaki,2,20.0
3,Triumph,1,10.0


<a id="transition-matrices"></a>
## 3. Transition Matrices
- Passing probabilities on any given lap
- Start-finish probabilities
- Assessing entertainment value by class

In [7]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Position buckets ──────────────────────────────────────────────────────────
def assign_bucket(pos):
    if pd.isna(pos): return None
    if pos <= 3:     return "1-3"
    if pos <= 6:     return "4-6"
    if pos <= 10:    return "7-10"
    if pos <= 15:    return "11-15"
    return "16+"

BUCKETS = ["1-3", "4-6", "7-10", "11-15", "16+"]

# ── Build transitions ────────────────────────────────────────────────────
lap_data = (
    df.dropna(subset=["lap", "place"])
    .sort_values(["race_id", "name", "lap"])
    .copy()
)
lap_data["bucket"]      = lap_data["place"].apply(assign_bucket)
lap_data["next_bucket"] = lap_data.groupby(["race_id", "name"], observed=True)["bucket"].shift(-1)
lap_data["next_lap"]    = lap_data.groupby(["race_id", "name"], observed=True)["lap"].shift(-1)

transitions = lap_data[
    lap_data["next_bucket"].notna() &
    lap_data["bucket"].notna() &
    (lap_data["next_lap"] == lap_data["lap"] + 1)
].copy()

# ── Filter options ────────────────────────────────────────────────────────────
all_years   = sorted(transitions["year"].astype(str).unique())
all_classes = ["450", "250", "WMX"]
all_tracks  = sorted(transitions["track"].astype(str).unique())
all_riders  = sorted(transitions["name"].astype(str).unique())

# ── Widget factory ────────────────────────────────────────────────────────────
def get_eligible_riders(year_sel, class_sel, track_sel):
    mask = (
        transitions["year"].astype(str).isin(year_sel) &
        transitions["class_label"].isin(class_sel) &
        transitions["track"].astype(str).isin(track_sel)
    )
    return sorted(transitions[mask]["name"].astype(str).unique())

def make_filters(label):
    title = widgets.HTML(f"<b style='font-size:14px'>Matrix {label}</b>")
    year_sel = widgets.SelectMultiple(
        options=all_years, value=all_years,
        description="Year:", rows=len(all_years)
    )
    class_sel = widgets.SelectMultiple(
        options=all_classes, value=all_classes,
        description="Class:", rows=len(all_classes)
    )
    track_sel = widgets.SelectMultiple(
        options=all_tracks, value=all_tracks,
        description="Track:", rows=min(6, len(all_tracks))
    )
    rider_sel = widgets.Dropdown(
        options=all_riders, value=all_riders[0] if all_riders else None,
        description="Rider:",
        layout=widgets.Layout(width="250px")
    )

    def refresh_riders(*args):
        eligible = get_eligible_riders(year_sel.value, class_sel.value, track_sel.value)
        current  = rider_sel.value
        rider_sel.options = eligible
        rider_sel.value   = current if current in eligible else (eligible[0] if eligible else None)

    for w in [year_sel, class_sel, track_sel]:
        w.observe(refresh_riders, names="value")

    return title, year_sel, class_sel, track_sel, rider_sel

title_a, year_a, class_a, track_a, rider_a = make_filters("A")
title_b, year_b, class_b, track_b, rider_b = make_filters("B")

btn    = widgets.Button(description="Update", button_style="primary")
output = widgets.Output()

# ── Filter label builder ──────────────────────────────────────────────────────
def make_filter_label(year_sel, class_sel, track_sel, rider_sel):
    years   = ", ".join(sorted(year_sel))
    classes = ", ".join(sorted(class_sel))

    if set(track_sel) == set(all_tracks):
        tracks = "All Tracks"
    elif len(track_sel) <= 2:
        tracks = ", ".join(sorted(track_sel))
    else:
        tracks = f"{len(track_sel)} Tracks"

    riders = rider_sel if rider_sel else "No Rider Selected"

    return f"{classes} | {years} | {tracks} | {riders}"

# ── Heatmap builder ───────────────────────────────────────────────────────────
def build_heatmap(subset, label):
    matrix = (
        subset.groupby(["bucket", "next_bucket"], observed=True)
        .size()
        .reset_index(name="count")
    )
    if matrix.empty:
        return go.Heatmap(
            z=[[0]*5]*5, x=BUCKETS, y=BUCKETS,
            colorscale="Oranges", zmin=0, zmax=1,
            showscale=False,
        )
    matrix["prob"] = (
        matrix["count"] /
        matrix.groupby("bucket", observed=True)["count"].transform("sum")
    )
    pivot = (
        matrix.pivot(index="bucket", columns="next_bucket", values="prob")
        .reindex(index=BUCKETS, columns=BUCKETS)
        .fillna(0)
    )
    text_labels = pivot.map(lambda v: f"{v:.0%}" if v > 0 else "")
    return go.Heatmap(
        z=pivot.values,
        x=BUCKETS,
        y=BUCKETS,
        text=text_labels.values,
        texttemplate="%{text}",
        textfont=dict(size=11),
        colorscale="Oranges",
        zmin=0, zmax=1,
        showscale=(label == "B"),
        colorbar=dict(title="Prob", tickformat=".0%", x=1.02) if label == "B" else None,
        hovertemplate="From: %{y}<br>To: %{x}<br>Probability: %{z:.1%}<extra></extra>",
        name=label,
    )

# ── Update callback ───────────────────────────────────────────────────────────
def update(_):
    output.clear_output(wait=True)
    with output:

        def filter_transitions(year_sel, class_sel, track_sel, rider_sel):
            return transitions[
                transitions["year"].astype(str).isin(year_sel) &
                transitions["class_label"].isin(class_sel) &
                transitions["track"].astype(str).isin(track_sel) &
                (transitions["name"].astype(str) == rider_sel)
            ]

        sub_a = filter_transitions(year_a.value, class_a.value, track_a.value, rider_a.value)
        sub_b = filter_transitions(year_b.value, class_b.value, track_b.value, rider_b.value)

        label_a = make_filter_label(year_a.value, class_a.value, track_a.value, rider_a.value)
        label_b = make_filter_label(year_b.value, class_b.value, track_b.value, rider_b.value)

        fig = make_subplots(
            rows=1, cols=2,
            subplot_titles=[label_a, label_b],
            horizontal_spacing=0.15,
        )
        fig.add_trace(build_heatmap(sub_a, "A"), row=1, col=1)
        fig.add_trace(build_heatmap(sub_b, "B"), row=1, col=2)

        for col in [1, 2]:
            fig.update_xaxes(title_text="Position (Lap N+1)", row=1, col=col)
            fig.update_yaxes(title_text="Position (Lap N)",   row=1, col=col)

        fig.update_layout(
            height=500, width=1100,
            margin=dict(l=60, r=80, t=80, b=60),
        )
        display(fig)

btn.on_click(update)

# ── Layout ────────────────────────────────────────────────────────────────────
panel_a = widgets.VBox([title_a, year_a, class_a, track_a, rider_a])
panel_b = widgets.VBox([title_b, year_b, class_b, track_b, rider_b])

display(widgets.VBox([
    widgets.HBox([panel_a, panel_b]),
    btn,
    output,
]))

update(None)

In [8]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Position buckets ──────────────────────────────────────────────────────────
def assign_bucket(pos):
    if pd.isna(pos): return None
    if pos <= 3:     return "1-3"
    if pos <= 6:     return "4-6"
    if pos <= 10:    return "7-10"
    if pos <= 15:    return "11-15"
    return "16+"

BUCKETS = ["1-3", "4-6", "7-10", "11-15", "16+"]

# ── Build start-finish transitions ────────────────────────────────────────────
lap1 = (
    df[df["lap"] == 1]
    .dropna(subset=["place"])
    [["race_id", "name", "class_label", "year", "track", "place"]]
    .rename(columns={"place": "lap1_place"})
    .copy()
)

finish = (
    df.drop_duplicates(subset=["race_id", "name"])
    .dropna(subset=["finish_position"])
    [["race_id", "name", "finish_position"]]
    .copy()
)

sf_transitions = lap1.merge(finish, on=["race_id", "name"], how="inner")
sf_transitions["start_bucket"]  = sf_transitions["lap1_place"].apply(assign_bucket)
sf_transitions["finish_bucket"] = sf_transitions["finish_position"].apply(assign_bucket)
sf_transitions = sf_transitions.dropna(subset=["start_bucket", "finish_bucket"])

# ── Filter options ────────────────────────────────────────────────────────────
all_years   = sorted(sf_transitions["year"].astype(str).unique())
all_classes = ["450", "250", "WMX"]
all_tracks  = sorted(sf_transitions["track"].astype(str).unique())
all_riders  = sorted(sf_transitions["name"].astype(str).unique())

# ── Widget factory ────────────────────────────────────────────────────────────
def get_eligible_riders(year_sel, class_sel, track_sel):
    mask = (
        sf_transitions["year"].astype(str).isin(year_sel) &
        sf_transitions["class_label"].isin(class_sel) &
        sf_transitions["track"].astype(str).isin(track_sel)
    )
    return sorted(sf_transitions[mask]["name"].astype(str).unique())

def make_sf_filters(label):
    title = widgets.HTML(f"<b style='font-size:14px'>Matrix {label}</b>")
    year_sel = widgets.SelectMultiple(
        options=all_years, value=all_years,
        description="Year:", rows=len(all_years)
    )
    class_sel = widgets.SelectMultiple(
        options=all_classes, value=all_classes,
        description="Class:", rows=len(all_classes)
    )
    track_sel = widgets.SelectMultiple(
        options=all_tracks, value=all_tracks,
        description="Track:", rows=min(6, len(all_tracks))
    )
    rider_sel = widgets.Dropdown(
        options=all_riders, value=all_riders[0] if all_riders else None,
        description="Rider:",
        layout=widgets.Layout(width="250px")
    )

    def refresh_riders(*args):
        eligible = get_eligible_riders(year_sel.value, class_sel.value, track_sel.value)
        current  = rider_sel.value
        rider_sel.options = eligible
        rider_sel.value   = current if current in eligible else (eligible[0] if eligible else None)

    for w in [year_sel, class_sel, track_sel]:
        w.observe(refresh_riders, names="value")

    return title, year_sel, class_sel, track_sel, rider_sel

title_a, year_a, class_a, track_a, rider_a = make_sf_filters("A")
title_b, year_b, class_b, track_b, rider_b = make_sf_filters("B")

btn    = widgets.Button(description="Update", button_style="primary")
output = widgets.Output()

# ── Filter label builder ──────────────────────────────────────────────────────
def make_sf_filter_label(year_sel, class_sel, track_sel, rider_sel):
    years   = ", ".join(sorted(year_sel))
    classes = ", ".join(sorted(class_sel))

    if set(track_sel) == set(all_tracks):
        tracks = "All Tracks"
    elif len(track_sel) <= 2:
        tracks = ", ".join(sorted(track_sel))
    else:
        tracks = f"{len(track_sel)} Tracks"

    riders = rider_sel if rider_sel else "No Rider Selected"

    return f"{classes} | {years} | {tracks} | {riders}"

# ── Heatmap builder ───────────────────────────────────────────────────────────
def build_sf_heatmap(subset, label):
    matrix = (
        subset.groupby(["start_bucket", "finish_bucket"], observed=True)
        .size()
        .reset_index(name="count")
    )
    if matrix.empty:
        return go.Heatmap(
            z=[[0]*5]*5, x=BUCKETS, y=BUCKETS,
            colorscale="Oranges", zmin=0, zmax=1,
            showscale=False,
        )
    matrix["prob"] = (
        matrix["count"] /
        matrix.groupby("start_bucket", observed=True)["count"].transform("sum")
    )
    pivot = (
        matrix.pivot(index="start_bucket", columns="finish_bucket", values="prob")
        .reindex(index=BUCKETS, columns=BUCKETS)
        .fillna(0)
    )
    text_labels = pivot.map(lambda v: f"{v:.0%}" if v > 0 else "")
    return go.Heatmap(
        z=pivot.values,
        x=BUCKETS,
        y=BUCKETS,
        text=text_labels.values,
        texttemplate="%{text}",
        textfont=dict(size=11),
        colorscale="Oranges",
        zmin=0, zmax=1,
        showscale=(label == "B"),
        colorbar=dict(title="Prob", tickformat=".0%", x=1.02) if label == "B" else None,
        hovertemplate="Start: %{y}<br>Finish: %{x}<br>Probability: %{z:.1%}<extra></extra>",
        name=label,
    )

# ── Update callback ───────────────────────────────────────────────────────────
def update(_):
    output.clear_output(wait=True)
    with output:

        def filter_sf(year_sel, class_sel, track_sel, rider_sel):
            return sf_transitions[
                sf_transitions["year"].astype(str).isin(year_sel) &
                sf_transitions["class_label"].isin(class_sel) &
                sf_transitions["track"].astype(str).isin(track_sel) &
                (sf_transitions["name"].astype(str) == rider_sel)
            ]

        sub_a = filter_sf(year_a.value, class_a.value, track_a.value, rider_a.value)
        sub_b = filter_sf(year_b.value, class_b.value, track_b.value, rider_b.value)

        label_a = make_sf_filter_label(year_a.value, class_a.value, track_a.value, rider_a.value)
        label_b = make_sf_filter_label(year_b.value, class_b.value, track_b.value, rider_b.value)

        fig = make_subplots(
            rows=1, cols=2,
            subplot_titles=[label_a, label_b],
            horizontal_spacing=0.15,
        )
        fig.add_trace(build_sf_heatmap(sub_a, "A"), row=1, col=1)
        fig.add_trace(build_sf_heatmap(sub_b, "B"), row=1, col=2)

        for col in [1, 2]:
            fig.update_xaxes(title_text="Finish Position", row=1, col=col)
            fig.update_yaxes(title_text="Start Position",  row=1, col=col)

        fig.update_layout(
            height=500, width=1100,
            margin=dict(l=60, r=80, t=80, b=60),
        )
        display(fig)

btn.on_click(update)

# ── Layout ────────────────────────────────────────────────────────────────────
panel_a = widgets.VBox([title_a, year_a, class_a, track_a, rider_a])
panel_b = widgets.VBox([title_b, year_b, class_b, track_b, rider_b])

display(widgets.VBox([
    widgets.HBox([panel_a, panel_b]),
    btn,
    output,
]))

update(None)

In [9]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import plotly.graph_objects as go
import numpy as np

# ── Transitions setup ─────────
def assign_bucket(pos):
    if pd.isna(pos): return None
    if pos <= 3:     return "1-3"
    if pos <= 6:     return "4-6"
    if pos <= 10:    return "7-10"
    if pos <= 15:    return "11-15"
    return "16+"

BUCKETS = ["1-3", "4-6", "7-10", "11-15", "16+"]

lap_data = (
    df.dropna(subset=["lap", "place"])
    .sort_values(["race_id", "name", "lap"])
    .copy()
)
lap_data["bucket"]      = lap_data["place"].apply(assign_bucket)
lap_data["next_bucket"] = lap_data.groupby(["race_id", "name"], observed=True)["bucket"].shift(-1)
lap_data["next_lap"]    = lap_data.groupby(["race_id", "name"], observed=True)["lap"].shift(-1)

transitions = lap_data[
    lap_data["next_bucket"].notna() &
    lap_data["bucket"].notna() &
    (lap_data["next_lap"] == lap_data["lap"] + 1)
].copy()


# ── Compute pass flag per rider per lap transition ────────────────────────────
lap_pass = transitions.copy()
lap_pass["place"] = lap_pass["place"].astype(float)
lap_pass["next_place"] = lap_pass.groupby(["race_id", "name"], observed=True)["place"].shift(-1)
lap_pass["made_pass"] = lap_pass["next_place"] < lap_pass["place"]

# ── Filter options ────────────────────────────────────────────────────────────
all_years = sorted(lap_pass["year"].astype(str).unique())
all_classes = ["450", "250", "WMX"]
all_tracks = sorted(lap_pass["track"].astype(str).unique())


# ── Widget factory ────────────────────────────────────────────────────────────
def make_pass_filters(label):
    title = widgets.HTML(f"<b style='font-size:14px'>Series {label}</b>")
    year_sel = widgets.SelectMultiple(
        options=all_years, value=all_years,
        description="Year:", rows=len(all_years)
    )
    class_sel = widgets.SelectMultiple(
        options=all_classes, value=all_classes,
        description="Class:", rows=len(all_classes)
    )
    track_sel = widgets.SelectMultiple(
        options=all_tracks, value=all_tracks,
        description="Track:", rows=min(6, len(all_tracks))
    )
    return title, year_sel, class_sel, track_sel


title_a, year_a, class_a, track_a = make_pass_filters("A")
title_b, year_b, class_b, track_b = make_pass_filters("B")

btn = widgets.Button(description="Update", button_style="primary")
output = widgets.Output()


# ── Filter label builder ──────────────────────────────────────────────────────
def make_filter_label(year_sel, class_sel, track_sel):
    years = ", ".join(sorted(year_sel))
    classes = ", ".join(sorted(class_sel))

    if set(track_sel) == set(all_tracks):
        tracks = "All Tracks"
    elif len(track_sel) <= 2:
        tracks = ", ".join(sorted(track_sel))
    else:
        tracks = f"{len(track_sel)} Tracks"

    return f"{classes} | {years} | {tracks}"


# ── Pass rate calc ────────────────────────────────────────────────────────────
def compute_pass_rate(subset):
    if subset.empty:
        return pd.DataFrame(columns=["lap", "pass_rate", "riders"])
    grouped = (
        subset.groupby("lap", observed=True)
        .agg(
            riders=("name", "count"),
            passers=("made_pass", "sum"),
        )
        .reset_index()
    )
    grouped["pass_rate"] = (grouped["passers"] / grouped["riders"] * 100).round(1)
    grouped["lap"] = grouped["lap"].astype(float)
    return grouped.sort_values("lap")


# ── Line chart builder ────────────────────────────────────────────────────────
def build_pass_line(subset, color, title_str):
    pass_by_lap = compute_pass_rate(subset)
    if pass_by_lap.empty:
        return [go.Scatter(x=[], y=[], name=title_str)]

    x = pass_by_lap["lap"].values
    y = pass_by_lap["pass_rate"].values

    main_trace = go.Scatter(
        x=x,
        y=y,
        mode="lines+markers",
        name=title_str,
        line=dict(color=color, width=2),
        marker=dict(size=6),
        hovertemplate=(
            f"<b>{title_str}</b><br>"
            "Lap: %{x}<br>"
            "Pass rate: %{y}<br>"
            "Across %{customdata} rider-laps<extra></extra>"
        ),
        customdata=pass_by_lap["riders"],
    )

    # Linear trend line
    coeffs  = np.polyfit(x, y, 1)
    trend_y = np.polyval(coeffs, x)
    ss_res  = np.sum((y - trend_y) ** 2)
    ss_tot  = np.sum((y - np.mean(y)) ** 2)
    r2      = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
    slope_sign = "↑" if coeffs[0] >= 0 else "↓"

    trend_trace = go.Scatter(
        x=x,
        y=trend_y,
        mode="lines",
        name=f"{title_str} trend {slope_sign} (R²={r2:.2f})",
        line=dict(color=color, width=1.5, dash="dash"),
        hovertemplate=(
            f"<b>{title_str} trend</b><br>"
            "Lap: %{x}<br>"
            "Trend: %{y:.1f}%<extra></extra>"
        ),
    )

    return [main_trace, trend_trace]

# ── Update callback ───────────────────────────────────────────────────────────
def update(_):
    output.clear_output(wait=True)
    with output:

        def filter_pass(year_sel, class_sel, track_sel):
            return lap_pass[
                lap_pass["year"].astype(str).isin(year_sel) &
                lap_pass["class_label"].isin(class_sel) &
                lap_pass["track"].astype(str).isin(track_sel) &
                lap_pass["next_place"].notna()
                ]

        sub_a = filter_pass(year_a.value, class_a.value, track_a.value)
        sub_b = filter_pass(year_b.value, class_b.value, track_b.value)

        label_a = make_filter_label(year_a.value, class_a.value, track_a.value)
        label_b = make_filter_label(year_b.value, class_b.value, track_b.value)

        # Dynamic y-axis range across both series
        all_rates = pd.concat([
            compute_pass_rate(sub_a)["pass_rate"],
            compute_pass_rate(sub_b)["pass_rate"],
        ]).dropna()
        if all_rates.empty:
            y_min, y_max = 0, 100
        else:
            padding = (all_rates.max() - all_rates.min()) * 0.15 or 5
            y_min = max(0, all_rates.min() - padding)
            y_max = min(100, all_rates.max() + padding)

        fig = go.Figure()
        for trace in build_pass_line(sub_a, "#E8641A", label_a):
            fig.add_trace(trace)
        for trace in build_pass_line(sub_b, "#1A7FE8", label_b):
            fig.add_trace(trace)

        fig.update_layout(
            title="% of riders who made a pass by lap number",
            xaxis=dict(
                title="Lap Number",
                tickmode="linear",
                dtick=1,
            ),
            yaxis=dict(
                title="% of riders who made a pass",
                range=[y_min, y_max],
                ticksuffix="%",
            ),
            height=500,
            width=900,
            margin=dict(l=60, r=60, t=60, b=100),
            legend=dict(
                title="",
                orientation="h",
                yanchor="bottom",
                y=-0.45,
                xanchor="left",
                x=0,
            ),
            hovermode="x unified",
        )
        display(fig)


btn.on_click(update)

# ── Layout ────────────────────────────────────────────────────────────────────
panel_a = widgets.VBox([title_a, year_a, class_a, track_a])
panel_b = widgets.VBox([title_b, year_b, class_b, track_b])

display(widgets.VBox([
    widgets.HBox([panel_a, panel_b]),
    btn,
    output,
]))

update(None)

In [10]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import plotly.graph_objects as go
import pandas as pd
import numpy as np
from scipy.stats import gaussian_kde

# ── Build gap to leader base ──────────────────────────────────────────────────
gap_base = (
    df.dropna(subset=["lap", "behind_time", "place"])
    .copy()
)
gap_base["lap"]         = gap_base["lap"].astype(float)
gap_base["place"]       = gap_base["place"].astype(float)
gap_base["behind_time"] = gap_base["behind_time"].astype(float)
gap_base["year"]        = gap_base["year"].astype(str)

# Keep only 2nd place riders
gap_base = gap_base[gap_base["place"] == 2].copy()

# ── Filter options ─────────────────────────────────────────────────────────────
gap_years   = sorted(gap_base["year"].unique())

CLASS_COLORS = {
    "450": "#1A7FE8",
    "250": "#E8641A",
    "WMX": "#1AE87F",
}

# ── Helper ─────────────────────────────────────────────────────────────────────
def hex_to_rgba(hex_color, opacity=0.08):
    hex_color = hex_color.lstrip("#")
    r, g, b   = int(hex_color[0:2], 16), int(hex_color[2:4], 16), int(hex_color[4:6], 16)
    return f"rgba({r},{g},{b},{opacity})"

# ── Widgets ────────────────────────────────────────────────────────────────────
year_sel_gap = widgets.Dropdown(
    options=gap_years, value=gap_years[-1],
    description="Year:", layout=widgets.Layout(width="200px")
)

# ── Plot function ──────────────────────────────────────────────────────────────
def plot_gap(year_val):
    fig      = go.Figure()
    all_vals = []

    for cls, color in CLASS_COLORS.items():
        mask = (
            (gap_base["year"]        == year_val) &
            (gap_base["class_label"] == cls)
        )
        vals = gap_base[mask]["behind_time"].dropna().values

        if len(vals) < 2:
            continue

        all_vals.extend(vals)

        p95     = np.percentile(vals, 95)
        kde     = gaussian_kde(vals, bw_method=0.3)
        x_range = np.linspace(0, p95, 500)
        density        = kde(x_range)
        median         = np.median(vals)
        median_density = kde(np.array([median]))[0]

        fig.add_trace(go.Scatter(
            x=x_range,
            y=density,
            mode="lines",
            name=f"{cls} (n={len(vals)}, median={median:.2f}s)",
            line=dict(color=color, width=2),
            fill="tozeroy",
            fillcolor=hex_to_rgba(color, 0.08),
            hovertemplate=(
                f"<b>{cls}</b><br>"
                "Gap: %{x:.2f}s<br>"
                "Density: %{y:.4f}"
                "<extra></extra>"
            ),
        ))

        fig.add_trace(go.Scatter(
            x=[median, median],
            y=[0, median_density],
            mode="lines",
            line=dict(color=color, width=1.5, dash="dot"),
            showlegend=False,
            hoverinfo="skip",
        ))

    if not all_vals:
        print("No data for selected filters.")
        return

    global_p95 = np.percentile(all_vals, 95)

    fig.update_layout(
        title=f"Gap to Leader (1st to 2nd) — All Classes | {year_val} | Capped at 95th Percentile",
        xaxis=dict(
            title="Gap to Leader (seconds)",
            range=[0, global_p95],
        ),
        yaxis=dict(title="Density", showticklabels=False),
        height=500,
        width=950,
        margin=dict(l=60, r=60, t=60, b=60),
        legend=dict(title="Class"),
        hovermode="x unified",
    )
    display(fig)

btn_gap = widgets.Button(description="Update", button_style="primary")
output_gap = widgets.Output()

def update_gap(_):
    output_gap.clear_output(wait=True)
    with output_gap:
        plot_gap(year_sel_gap.value)

btn_gap.on_click(update_gap)

display(widgets.VBox([
    year_sel_gap,
    btn_gap,
    output_gap,
]))

update_gap(None)

<a id="fatigue-effects"></a>
## 4. Fatigue Effects
- How does pace change throughout a moto?
- How does pace change from moto 1 to moto 2?

In [11]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import plotly.graph_objects as go
import numpy as np
from scipy import stats

# ── Build fatigue data ────────────────────────────────────────────────────────
fatigue_base = (
    df.dropna(subset=["lap", "lap_time"])
    .copy()
)
fatigue_base["lap"] = fatigue_base["lap"].astype(float)

# Max lap per race (across all riders)
max_lap_per_race = (
    fatigue_base.groupby("race_id", observed=True)["lap"]
    .max()
    .rename("max_lap")
    .reset_index()
)
fatigue_base = fatigue_base.merge(max_lap_per_race, on="race_id", how="left")
fatigue_base["threshold"] = (fatigue_base["max_lap"] * 0.75).apply(np.floor)

# Laps completed per rider per race
laps_completed = (
    fatigue_base.groupby(["race_id", "name"], observed=True)["lap"]
    .max()
    .rename("laps_completed")
    .reset_index()
)
fatigue_base = fatigue_base.merge(laps_completed, on=["race_id", "name"], how="left")

# Keep only riders who met the 75% threshold
fatigue_base = fatigue_base[fatigue_base["laps_completed"] >= fatigue_base["threshold"]].copy()

# Exclude lap 1
fatigue_base = fatigue_base[fatigue_base["lap"] > 1].copy()

# Personal best lap time within each moto
personal_best = (
    fatigue_base.groupby(["race_id", "name"], observed=True)["lap_time"]
    .min()
    .rename("personal_best")
    .reset_index()
)
fatigue_base = fatigue_base.merge(personal_best, on=["race_id", "name"], how="left")

# Lap time delta as percentage above personal best
fatigue_base["delta_pct"] = (
    (fatigue_base["lap_time"] - fatigue_base["personal_best"]) /
    fatigue_base["personal_best"] * 100
)

# % of race completed per lap
fatigue_base["pct_completed"] = (fatigue_base["lap"] / fatigue_base["max_lap"] * 100)

# Bin into 10% intervals, labeled by upper bound
bin_edges  = list(range(0, 101, 10))
bin_labels = [f"{b+10}" for b in range(0, 100, 10)]
fatigue_base["pct_bin"] = pd.cut(
    fatigue_base["pct_completed"],
    bins=bin_edges,
    labels=bin_labels,
    include_lowest=True
)

# ── Filter options ────────────────────────────────────────────────────────────
all_years   = sorted(fatigue_base["year"].astype(str).unique())
all_classes = ["450", "250", "WMX"]
all_tracks  = sorted(fatigue_base["track"].astype(str).unique())
all_motos   = sorted(fatigue_base["moto"].astype(str).unique())
all_riders  = sorted(fatigue_base["name"].astype(str).unique())

# ── Widget factory ────────────────────────────────────────────────────────────
def get_eligible_riders(year_sel, class_sel, track_sel, moto_sel):
    mask = (
        fatigue_base["year"].astype(str).isin(year_sel) &
        fatigue_base["class_label"].isin(class_sel) &
        fatigue_base["track"].astype(str).isin(track_sel) &
        fatigue_base["moto"].astype(str).isin(moto_sel)
    )
    return sorted(fatigue_base[mask]["name"].astype(str).unique())

def make_fatigue_filters(label):
    title = widgets.HTML(f"<b style='font-size:14px'>Series {label}</b>")
    year_sel = widgets.SelectMultiple(
        options=all_years, value=all_years,
        description="Year:", rows=len(all_years)
    )
    class_sel = widgets.SelectMultiple(
        options=all_classes, value=all_classes,
        description="Class:", rows=len(all_classes)
    )
    track_sel = widgets.SelectMultiple(
        options=all_tracks, value=all_tracks,
        description="Track:", rows=min(6, len(all_tracks))
    )
    moto_sel = widgets.SelectMultiple(
        options=all_motos, value=all_motos,
        description="Moto:", rows=len(all_motos)
    )
    rider_sel = widgets.Dropdown(
        options=all_riders, value=all_riders[0] if all_riders else None,
        description="Rider:",
        layout=widgets.Layout(width="250px")
    )

    def refresh_riders(*args):
        eligible = get_eligible_riders(year_sel.value, class_sel.value, track_sel.value, moto_sel.value)
        current  = rider_sel.value
        rider_sel.options = eligible
        rider_sel.value   = current if current in eligible else (eligible[0] if eligible else None)

    for w in [year_sel, class_sel, track_sel, moto_sel]:
        w.observe(refresh_riders, names="value")

    return title, year_sel, class_sel, track_sel, moto_sel, rider_sel

title_a, year_a, class_a, track_a, moto_a, rider_a = make_fatigue_filters("A")
title_b, year_b, class_b, track_b, moto_b, rider_b = make_fatigue_filters("B")

btn    = widgets.Button(description="Update", button_style="primary")
output = widgets.Output()

# ── Filter label builder ──────────────────────────────────────────────────────
def make_fatigue_label(year_sel, class_sel, track_sel, moto_sel, rider_sel):
    years   = ", ".join(sorted(year_sel))
    classes = ", ".join(sorted(class_sel))

    if set(track_sel) == set(all_tracks):
        tracks = "All Tracks"
    elif len(track_sel) <= 2:
        tracks = ", ".join(sorted(track_sel))
    else:
        tracks = f"{len(track_sel)} Tracks"

    if set(moto_sel) == set(all_motos):
        motos = "All Motos"
    else:
        motos = "Moto " + ", ".join(sorted(moto_sel))

    riders = rider_sel if rider_sel else "No Rider Selected"

    return f"{classes} | {years} | {tracks} | {motos} | {riders}"

# ── Line builder ──────────────────────────────────────────────────────────────
def build_fatigue_line(subset, color, label):
    if subset.empty:
        return None, None

    avg_delta = (
        subset.groupby("pct_bin", observed=True)["delta_pct"]
        .agg(avg_delta="mean", n="count")
        .reset_index()
    )
    avg_delta["pct_bin_num"] = avg_delta["pct_bin"].astype(float)
    avg_delta = avg_delta.sort_values("pct_bin_num")

    slope, intercept, r, p, _ = stats.linregress(
        avg_delta["pct_bin_num"], avg_delta["avg_delta"]
    )
    trendline_y = slope * avg_delta["pct_bin_num"] + intercept

    scatter = go.Scatter(
        x=avg_delta["pct_bin_num"],
        y=avg_delta["avg_delta"],
        mode="lines+markers",
        name=label,
        line=dict(color=color, width=2),
        marker=dict(size=6),
        hovertemplate=(
            f"<b>{label}</b><br>"
            "Race Completed: %{x}<br>"
            "Avg Delta: +%{y:.2f}%<br>"
            "Sample Size: %{customdata}<extra></extra>"
        ),
        customdata=avg_delta["n"],
    )

    trend = go.Scatter(
        x=avg_delta["pct_bin_num"],
        y=trendline_y,
        mode="lines",
        name=f"Trend {label}: {slope:+.3f}% per 10% completed  (R²={r**2:.3f})",
        line=dict(color=color, width=2, dash="dash"),
        hoverinfo="skip",
    )

    return scatter, trend, avg_delta, trendline_y

# ── Update callback ───────────────────────────────────────────────────────────
def update(_):
    output.clear_output(wait=True)
    with output:

        def filter_fatigue(year_sel, class_sel, track_sel, moto_sel, rider_sel):
            return fatigue_base[
                fatigue_base["year"].astype(str).isin(year_sel) &
                fatigue_base["class_label"].isin(class_sel) &
                fatigue_base["track"].astype(str).isin(track_sel) &
                fatigue_base["moto"].astype(str).isin(moto_sel) &
                (fatigue_base["name"].astype(str) == rider_sel)
            ]

        sub_a = filter_fatigue(year_a.value, class_a.value, track_a.value, moto_a.value, rider_a.value)
        sub_b = filter_fatigue(year_b.value, class_b.value, track_b.value, moto_b.value, rider_b.value)

        label_a = make_fatigue_label(year_a.value, class_a.value, track_a.value, moto_a.value, rider_a.value)
        label_b = make_fatigue_label(year_b.value, class_b.value, track_b.value, moto_b.value, rider_b.value)

        result_a = build_fatigue_line(sub_a, "#E8641A", label_a)
        result_b = build_fatigue_line(sub_b, "#1A7FE8", label_b)

        if result_a[0] is None and result_b[0] is None:
            print("No data for selected filters.")
            return

        fig = go.Figure()

        all_y = []
        all_x = set()

        for result, label in [(result_a, label_a), (result_b, label_b)]:
            if result[0] is None:
                continue
            scatter, trend, avg_delta, trendline_y = result
            fig.add_trace(scatter)
            fig.add_trace(trend)
            all_y += list(avg_delta["avg_delta"]) + list(trendline_y)
            all_x.update(avg_delta["pct_bin_num"].tolist())

        padding = (max(all_y) - min(all_y)) * 0.15 or 0.5
        y_min   = max(0, min(all_y) - padding)
        y_max   = max(all_y) + padding + 1.5

        sorted_x = sorted(all_x)

        fig.update_layout(
            title="Fatigue Effect",
            xaxis=dict(
                title="% of Race Completed",
                tickmode="array",
                tickvals=sorted_x,
                ticktext=[f"{int(v)}%" for v in sorted_x],
            ),
            yaxis=dict(
                title="Lap Time Delta from Personal Best (%)",
                range=[y_min, y_max],
                ticksuffix="%",
            ),
            height=550,
            width=1000,
            margin=dict(l=60, r=60, t=60, b=80),
            legend=dict(
                orientation="h",
                yanchor="bottom",
                y=-0.35,
                xanchor="left",
                x=0,
            ),
            hovermode="x unified",
        )
        display(fig)

btn.on_click(update)

# ── Layout ────────────────────────────────────────────────────────────────────
panel_a = widgets.VBox([title_a, year_a, class_a, track_a, moto_a, rider_a])
panel_b = widgets.VBox([title_b, year_b, class_b, track_b, moto_b, rider_b])

display(widgets.VBox([
    widgets.HBox([panel_a, panel_b]),
    btn,
    output,
]))

print("\nNotes")
print("─────────────────")
print("• Y-axis is the % a lap is slower than that rider's fastest lap")
print("  in the same moto (personal best per moto, not per season).")
print("• Riders must complete ≥75% of the moto's laps to be included;")
print("  lap 1 is excluded as starts have shorter lap times and are noisy.")
print("• Binned by % of race completed rather than lap number to enable")
print("  fair comparison across tracks with different total lap counts.")
print("• The trendline is linear — a steeper slope means more degradation")
print("  per 10% of race completed. R² shows how well the linear fit")
print("  describes the pattern; low R² means the pattern isn't straight-line.")

update(None)


Notes
─────────────────
• Y-axis is the % a lap is slower than that rider's fastest lap
  in the same moto (personal best per moto, not per season).
• Riders must complete ≥75% of the moto's laps to be included;
  lap 1 is excluded as starts have shorter lap times and are noisy.
• Binned by % of race completed rather than lap number to enable
  fair comparison across tracks with different total lap counts.
• The trendline is linear — a steeper slope means more degradation
  per 10% of race completed. R² shows how well the linear fit
  describes the pattern; low R² means the pattern isn't straight-line.


In [12]:
import pandas as pd
import numpy as np

# ── Build moto comparison data ────────────────────────────────────────────────
moto_comp_base = (
    df.dropna(subset=["lap", "lap_time"])
    .copy()
)
moto_comp_base["lap"]  = moto_comp_base["lap"].astype(float)
moto_comp_base["moto"] = moto_comp_base["moto"].astype(str)

# Apply 75% threshold
max_lap_mc = (
    moto_comp_base.groupby("race_id", observed=True)["lap"]
    .max()
    .rename("max_lap")
    .reset_index()
)
moto_comp_base = moto_comp_base.merge(max_lap_mc, on="race_id", how="left")
moto_comp_base["threshold"] = (moto_comp_base["max_lap"] * 0.75).apply(np.floor)

laps_comp = (
    moto_comp_base.groupby(["race_id", "name"], observed=True)["lap"]
    .max()
    .rename("laps_completed")
    .reset_index()
)
moto_comp_base = moto_comp_base.merge(laps_comp, on=["race_id", "name"], how="left")
moto_comp_base = moto_comp_base[
    moto_comp_base["laps_completed"] >= moto_comp_base["threshold"]
].copy()

# Average lap time per rider per race (excluding lap 1)
avg_lap = (
    moto_comp_base[moto_comp_base["lap"] > 1]
    .groupby(["race_id", "name", "class_label", "year", "round", "moto"], observed=True)
    ["lap_time"]
    .mean()
    .reset_index(name="avg_lap_time")
)
avg_lap["round"] = avg_lap["round"].astype(str)
avg_lap["moto"]  = avg_lap["moto"].astype(str)
avg_lap["year"]  = avg_lap["year"].astype(str)

# ── WMX moto remapping ────────────────────────────────────────────────────────
wmx_round2 = (
    (avg_lap["class_label"] == "WMX") &
    (avg_lap["round"] == "2")
)
avg_lap = avg_lap[~(wmx_round2 & (avg_lap["moto"] == "1"))].copy()
avg_lap.loc[wmx_round2 & (avg_lap["moto"] == "2"), "moto"] = "1"
avg_lap.loc[wmx_round2 & (avg_lap["moto"] == "3"), "moto"] = "2"
avg_lap = avg_lap[avg_lap["moto"].isin(["1", "2"])].copy()

# Only include riders who completed both motos at a round
rider_motos_per_round = (
    avg_lap.groupby(["name", "class_label", "year", "round"], observed=True)["moto"]
    .nunique()
    .reset_index(name="motos_completed")
)
both_motos = rider_motos_per_round[rider_motos_per_round["motos_completed"] == 2]
avg_lap = avg_lap.merge(
    both_motos[["name", "class_label", "year", "round"]],
    on=["name", "class_label", "year", "round"],
    how="inner"
)

# ── Within-round moto difference ──────────────────────────────────────────────
moto_pivot = avg_lap.pivot_table(
    index=["name", "class_label", "year", "round"],
    columns="moto",
    values="avg_lap_time",
    observed=True
).reset_index()
moto_pivot.columns.name = None
moto_pivot = moto_pivot.rename(columns={"1": "moto1_avg", "2": "moto2_avg"})
moto_pivot = moto_pivot.dropna(subset=["moto1_avg", "moto2_avg"])

moto_pivot["diff"]           = moto_pivot["moto2_avg"] - moto_pivot["moto1_avg"]
moto_pivot["slower_in_moto2"] = moto_pivot["diff"] > 0

# ── Summary table by class and year ───────────────────────────────────────────
summary = (
    moto_pivot.groupby(["class_label", "year"], observed=True)
    .agg(
        mean_diff=("diff", "mean"),
        median_diff=("diff", "median"),
        pct_slower=("slower_in_moto2", "mean"),
        n=("diff", "count"),
    )
    .reset_index()
)
summary["mean_diff"]   = summary["mean_diff"].round(2)
summary["median_diff"] = summary["median_diff"].round(2)
summary["pct_slower"]  = (summary["pct_slower"] * 100).round(1).astype(str) + "%"
summary = summary.rename(columns={
    "class_label":  "Class",
    "year":         "Year",
    "mean_diff":    "Mean Diff (s)",
    "median_diff":  "Median Diff (s)",
    "pct_slower":   "% Slower in Moto 2",
    "n":            "N",
})
summary = summary.sort_values(["Class", "Year"]).reset_index(drop=True)

print("\nMoto 1 → Moto 2 Average Lap Time Change")
print("Positive = slower in Moto 2, Negative = faster in Moto 2\n")
display(summary)

# ── Overall summary (both years combined) ─────────────────────────────────────
overall = (
    moto_pivot.groupby("class_label", observed=True)
    .agg(
        mean_diff=("diff", "mean"),
        median_diff=("diff", "median"),
        pct_slower=("slower_in_moto2", "mean"),
        n=("diff", "count"),
    )
    .reset_index()
)
overall["mean_diff"]   = overall["mean_diff"].round(2)
overall["median_diff"] = overall["median_diff"].round(2)
overall["pct_slower"]  = (overall["pct_slower"] * 100).round(1).astype(str) + "%"
overall = overall.rename(columns={
    "class_label":  "Class",
    "mean_diff":    "Mean Diff (s)",
    "median_diff":  "Median Diff (s)",
    "pct_slower":   "% Slower in Moto 2",
    "n":            "N",
})
overall = overall.sort_values("Class").reset_index(drop=True)

print("\nOverall (2024 + 2025 Combined)")
display(overall)

print("\nNotes")
print("─────────────────")
print("• Riders are required to complete ≥75% of laps in each moto; lap 1 is")
print("  excluded as starts have shorter lap times and are noisy.")
print("• Only riders who completed both motos in a round are included")
print("  (paired comparison).")
print("• WMX Round 2 weekends ran three motos. Moto 1 was a separate day, so")
print("  we use motos 2 and 3 (same-day motos) for the fatigue comparison and")
print("  remap them to 'moto 1' and 'moto 2'.")
print("• '% Slower in Moto 2' is a binary count; mean and median capture")
print("  magnitude. When mean > median, a few large degradations are pulling")
print("  the average.")


Moto 1 → Moto 2 Average Lap Time Change
Positive = slower in Moto 2, Negative = faster in Moto 2



,Class,Year,Mean Diff (s),Median Diff (s),% Slower in Moto 2,N
0,250,2024,0.01,0.39,56.1%,264
1,250,2025,7.46,2.95,67.2%,241
2,450,2024,1.04,0.64,60.3%,199
3,450,2025,5.93,0.94,57.6%,170
4,WMX,2024,0.44,0.45,53.5%,129
5,WMX,2025,-6.36,-0.30,48.6%,148



Overall (2024 + 2025 Combined)


,Class,Mean Diff (s),Median Diff (s),% Slower in Moto 2,N
0,250,3.57,1.15,61.4%,505
1,450,3.29,0.67,59.1%,369
2,WMX,-3.19,0.20,50.9%,277



Notes
─────────────────
• Riders are required to complete ≥75% of laps in each moto; lap 1 is
  excluded as starts have shorter lap times and are noisy.
• Only riders who completed both motos in a round are included
  (paired comparison).
• WMX Round 2 weekends ran three motos. Moto 1 was a separate day, so
  we use motos 2 and 3 (same-day motos) for the fatigue comparison and
  remap them to 'moto 1' and 'moto 2'.
• '% Slower in Moto 2' is a binary count; mean and median capture
  magnitude. When mean > median, a few large degradations are pulling
  the average.


<a id="rider-rivalries"></a>
## 5. Rider Rivalries


In [13]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import plotly.graph_objects as go
import pandas as pd
import numpy as np

# ── Build adjacency data ──────────────────────────────────────────────────────
rivalry_base = (
    df.dropna(subset=["lap", "place", "behind_time"])
    .copy()
)
rivalry_base["lap"] = rivalry_base["lap"].astype(float)
rivalry_base["place"] = rivalry_base["place"].astype(float)
rivalry_base["behind_time"] = rivalry_base["behind_time"].astype(float)
rivalry_base["moto"] = rivalry_base["moto"].astype(str)
rivalry_base["year"] = rivalry_base["year"].astype(str)
rivalry_base["track"] = rivalry_base["track"].astype(str)

rivalry_base = rivalry_base.sort_values(["race_id", "lap", "place"])

# Build rider-ahead lookup with their cumulative behind_time
ahead_lookup = (
    rivalry_base[["race_id", "lap", "place", "name", "behind_time"]]
    .copy()
    .rename(columns={
        "name": "rider_ahead",
        "behind_time": "behind_time_ahead",
        "place": "place_ahead_lookup"  # avoid name collision on merge
    })
)

# Each rider merges to the rider at place - 1
rivalry_base["place_ahead"] = rivalry_base["place"] - 1

rivalry_laps = rivalry_base.merge(
    ahead_lookup,
    left_on=["race_id", "lap", "place_ahead"],
    right_on=["race_id", "lap", "place_ahead_lookup"],
    how="inner"
)

rivalry_laps = rivalry_laps.drop(columns=["place_ahead", "place_ahead_lookup"])

# Actual gap to rider directly ahead
rivalry_laps["gap_to_ahead"] = rivalry_laps["behind_time"] - rivalry_laps["behind_time_ahead"]

# Standardise pair so Rider 1 is always alphabetically first
rivalry_laps["rider_1"] = np.where(
    rivalry_laps["name"] < rivalry_laps["rider_ahead"],
    rivalry_laps["name"], rivalry_laps["rider_ahead"]
)
rivalry_laps["rider_2"] = np.where(
    rivalry_laps["name"] < rivalry_laps["rider_ahead"],
    rivalry_laps["rider_ahead"], rivalry_laps["name"]
)

# ── Proximity tiers ───────────────────────────────────────────────────────────
rivalry_laps["within_1s"] = rivalry_laps["gap_to_ahead"] < 1
rivalry_laps["within_2s"] = rivalry_laps["gap_to_ahead"] < 2
rivalry_laps["within_5s"] = rivalry_laps["gap_to_ahead"] < 5

# ── Consecutive streak within 2s within a single moto ────────────────────────
rivalry_laps = rivalry_laps.sort_values(["race_id", "rider_1", "rider_2", "lap"])


def calc_max_streak(group):
    within = group["within_2s"].astype(int).values
    max_streak = streak = 0
    for v in within:
        if v:
            streak += 1
            max_streak = max(max_streak, streak)
        else:
            streak = 0
    return max_streak


streak_df = (
    rivalry_laps.groupby(["race_id", "rider_1", "rider_2"], observed=True)
    .apply(calc_max_streak, include_groups=False)
    .reset_index(name="streak")
)

# Max streak across all motos matching the filter
max_streak = (
    streak_df.merge(
        rivalry_base[["race_id", "class_label", "year", "track"]]
        .drop_duplicates(),
        on="race_id", how="left"
    )
    .groupby(["rider_1", "rider_2", "class_label", "year", "track"], observed=True)
    ["streak"]
    .max()
    .reset_index(name="max_streak")
)

# ── Aggregate proximity tiers ─────────────────────────────────────────────────
rivalry_agg = (
    rivalry_laps.groupby(
        ["rider_1", "rider_2", "class_label", "year", "track"],
        observed=True
    )
    .agg(
        laps_within_1s=("within_1s", "sum"),
        laps_within_2s=("within_2s", "sum"),
        laps_within_5s=("within_5s", "sum"),
    )
    .reset_index()
)

# Join streak
rivalry_agg = rivalry_agg.merge(
    max_streak,
    on=["rider_1", "rider_2", "class_label", "year", "track"],
    how="left"
)
rivalry_agg["max_streak"] = rivalry_agg["max_streak"].fillna(0).astype(int)
rivalry_agg["laps_within_1s"] = rivalry_agg["laps_within_1s"].astype(int)
rivalry_agg["laps_within_2s"] = rivalry_agg["laps_within_2s"].astype(int)
rivalry_agg["laps_within_5s"] = rivalry_agg["laps_within_5s"].astype(int)

# ── Filter options ────────────────────────────────────────────────────────────
all_years = sorted(rivalry_agg["year"].astype(str).unique())
all_classes = ["450", "250", "WMX"]
all_tracks = sorted(rivalry_agg["track"].astype(str).unique())

# ── Widgets ───────────────────────────────────────────────────────────────────
year_sel = widgets.SelectMultiple(
    options=all_years, value=all_years,
    description="Year:", rows=len(all_years)
)
class_sel = widgets.SelectMultiple(
    options=all_classes, value=all_classes,
    description="Class:", rows=len(all_classes)
)
track_sel = widgets.SelectMultiple(
    options=all_tracks, value=all_tracks,
    description="Track:", rows=min(6, len(all_tracks))
)
sort_sel = widgets.Dropdown(
    options=[
        ("Laps within 1s", "laps_within_1s"),
        ("Laps within 2s", "laps_within_2s"),
        ("Laps within 5s", "laps_within_5s"),
        ("Max Streak (within 2s)", "max_streak"),
    ],
    value="laps_within_2s",
    description="Sort by:",
    layout=widgets.Layout(width="300px")
)
top_n_sel = widgets.IntSlider(
    value=20, min=5, max=50, step=5,
    description="Top N:",
    layout=widgets.Layout(width="300px")
)

btn = widgets.Button(description="Update", button_style="primary")
output = widgets.Output()


# ── Update callback ───────────────────────────────────────────────────────────
def update(_):
    output.clear_output(wait=True)
    with output:

        filtered = rivalry_agg[
            rivalry_agg["year"].astype(str).isin(year_sel.value) &
            rivalry_agg["class_label"].isin(class_sel.value) &
            rivalry_agg["track"].astype(str).isin(track_sel.value)
            ]

        # Reaggregate after filtering
        filtered_agg = (
            filtered.groupby(["rider_1", "rider_2", "class_label"], observed=True)
            .agg(
                laps_within_1s=("laps_within_1s", "sum"),
                laps_within_2s=("laps_within_2s", "sum"),
                laps_within_5s=("laps_within_5s", "sum"),
                max_streak=("max_streak", "max"),
            )
            .reset_index()
        )

        # Apply 10-lap minimum within 5s
        filtered_agg = filtered_agg[filtered_agg["laps_within_5s"] >= 10].copy()

        if filtered_agg.empty:
            print("No rivalries found for selected filters.")
            return

        result = (
            filtered_agg
            .sort_values(sort_sel.value, ascending=False)
            .head(top_n_sel.value)
            .reset_index(drop=True)
        )
        result.index += 1
        result = result.rename(columns={
            "rider_1": "Rider 1",
            "rider_2": "Rider 2",
            "class_label": "Class",
            "laps_within_1s": "Laps < 1s",
            "laps_within_2s": "Laps < 2s",
            "laps_within_5s": "Laps < 5s",
            "max_streak": "Max Streak (< 2s)",
        })

        display(result[[
            "Rider 1", "Rider 2", "Class",
            "Laps < 1s", "Laps < 2s", "Laps < 5s", "Max Streak (< 2s)"
        ]])


btn.on_click(update)

# ── Layout ────────────────────────────────────────────────────────────────────
filters = widgets.VBox([
    widgets.HBox([year_sel, class_sel, track_sel]),
    widgets.HBox([sort_sel, top_n_sel]),
    btn,
])

display(widgets.VBox([filters, output]))

print("\nNotes")
print("─────────────────")
print("• 'Rivalry' here means on-track proximity, not off-track narrative.")
print("  Pairs may top the list because they were consistently fast and")
print("  near each other rather than because of any specific battle.")
print("• Gaps are computed only between consecutively-placed riders. If a")
print("  third rider was sandwiched between two rivals, those laps don't")
print("  count toward the pair's proximity totals.")
print("• Pairs are normalized alphabetically — A vs B and B vs A are the")
print("  same pair regardless of who led on a given lap.")
print("• 'Max streak' is the longest consecutive run of laps within 2s in")
print("  a single moto, taken as the max across all matching motos.")
print("• Pairs must have ≥10 laps within 5s across the filter to appear,")
print("  filtering out one-off close encounters.")
print("• WMX has shorter motos and smaller fields — absolute lap counts")
print("  are lower than 450/250 by structure, not because rivalries are")
print("  less intense.")

update(None)


Notes
─────────────────
• 'Rivalry' here means on-track proximity, not off-track narrative.
  Pairs may top the list because they were consistently fast and
  near each other rather than because of any specific battle.
• Gaps are computed only between consecutively-placed riders. If a
  third rider was sandwiched between two rivals, those laps don't
  count toward the pair's proximity totals.
• Pairs are normalized alphabetically — A vs B and B vs A are the
  same pair regardless of who led on a given lap.
• 'Max streak' is the longest consecutive run of laps within 2s in
  a single moto, taken as the max across all matching motos.
• Pairs must have ≥10 laps within 5s across the filter to appear,
  filtering out one-off close encounters.
• WMX has shorter motos and smaller fields — absolute lap counts
  are lower than 450/250 by structure, not because rivalries are
  less intense.


<a id="points-race"></a>
## 6. Points Race


In [14]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import plotly.graph_objects as go
import pandas as pd
import numpy as np

# ── Build points race data ────────────────────────────────────────────────────
points_race_base = (
    df.drop_duplicates(subset=["race_id", "name", "class_label"])
    .dropna(subset=["finish_position"])
    .copy()
)
points_race_base["year"]  = points_race_base["year"].astype(str)
points_race_base["round"] = points_race_base["round"].astype(int)
points_race_base["moto"]  = points_race_base["moto"].astype(str)

# Sum points per rider per round (both motos combined)
points_per_round = (
    points_race_base.groupby(["name", "class_label", "year", "round"], observed=True)
    .agg(round_points=("points", "sum"))
    .reset_index()
)

# Cumulative points across rounds in order
points_per_round = points_per_round.sort_values(["name", "class_label", "year", "round"])
points_per_round["cumulative_points"] = (
    points_per_round.groupby(["name", "class_label", "year"], observed=True)
    ["round_points"]
    .cumsum()
)

# Final points standing per rider per class per year
final_standings = (
    points_per_round.groupby(["name", "class_label", "year"], observed=True)
    ["cumulative_points"]
    .max()
    .reset_index(name="final_points")
    .sort_values(["class_label", "year", "final_points"], ascending=[True, True, False])
)

# Top 10 per class per year
top10_riders = (
    final_standings.groupby(["class_label", "year"], observed=True)
    .head(10)
    [["name", "class_label", "year"]]
)

# Filter points race to top 10 only
points_race = points_per_round.merge(
    top10_riders, on=["name", "class_label", "year"], how="inner"
)

# Fill missing rounds without using apply
all_rounds = sorted(points_race["round"].unique())
rider_index = points_race[["name", "class_label", "year"]].drop_duplicates()

full_grid = rider_index.merge(
    pd.DataFrame({"round": all_rounds}), how="cross"
)

points_race_filled = full_grid.merge(
    points_race[["name", "class_label", "year", "round", "cumulative_points", "round_points"]],
    on=["name", "class_label", "year", "round"],
    how="left"
)

points_race_filled = points_race_filled.sort_values(
    ["name", "class_label", "year", "round"]
)
points_race_filled["cumulative_points"] = (
    points_race_filled.groupby(["name", "class_label", "year"], observed=False)
    ["cumulative_points"]
    .ffill()
    .fillna(0)
)
points_race_filled["round_points"] = points_race_filled["round_points"].fillna(0).astype(int)

# ── Filter options ────────────────────────────────────────────────────────────
all_years   = sorted(points_race_filled["year"].astype(str).unique())
all_classes = ["450", "250", "WMX"]

# ── Widgets ───────────────────────────────────────────────────────────────────
year_sel = widgets.Dropdown(
    options=all_years,
    value=all_years[-1],
    description="Year:",
    layout=widgets.Layout(width="200px")
)
class_sel = widgets.Dropdown(
    options=all_classes,
    value="450",
    description="Class:",
    layout=widgets.Layout(width="200px")
)

btn    = widgets.Button(description="Update", button_style="primary")
output = widgets.Output()

# ── Update callback ───────────────────────────────────────────────────────────
def update(_):
    output.clear_output(wait=True)
    with output:

        subset = points_race_filled[
            (points_race_filled["year"] == year_sel.value) &
            (points_race_filled["class_label"] == class_sel.value)
        ]

        if subset.empty:
            print("No data for selected filters.")
            return

        # Get final standings to order legend
        final = (
            subset.groupby("name")["cumulative_points"]
            .max()
            .reset_index()
            .sort_values("cumulative_points", ascending=False)
        )
        rider_order = final["name"].tolist()

        fig = go.Figure()

        for rider in rider_order:
            rider_data = subset[subset["name"] == rider].sort_values("round")
            final_pts  = int(rider_data["cumulative_points"].max())

            fig.add_trace(go.Scatter(
                x=rider_data["round"],
                y=rider_data["cumulative_points"],
                mode="lines+markers",
                name=f"{rider} ({final_pts} pts)",
                line=dict(width=2),
                marker=dict(size=7),
                customdata=rider_data["round_points"],
                hovertemplate=(
                    f"<b>{rider}</b><br>"
                    "Points: %{y}<br>"
                    "Change: +%{customdata}<extra></extra>"
                ),
            ))

        fig.update_layout(
            title=f"Points Race — {class_sel.value} | {year_sel.value}",
            xaxis=dict(
                title="Round",
                tickmode="linear",
                dtick=1,
                range=[0.8, len(all_rounds) + 0.2],
            ),
            yaxis=dict(
                title="Cumulative Points",
            ),
            height=550,
            width=1000,
            margin=dict(l=60, r=60, t=60, b=60),
            legend=dict(
                title="Rider (Final Points)",
                traceorder="normal",
            ),
            hovermode="x unified",
        )
        fig.show()

btn.on_click(update)

# ── Layout ────────────────────────────────────────────────────────────────────
display(widgets.VBox([
    widgets.HBox([year_sel, class_sel, btn]),
    output,
]))

print("\nNotes")
print("─────────────────")
print("• WMX class is split into a West and East series so two championships")
print("  take place. For simplicity, we merged cumulative points into one standing.")

update(None)


Notes
─────────────────
• WMX class is split into a West and East series so two championships
  take place. For simplicity, we merged cumulative points into one standing.


<a id="from-the-holeshot-to-the-podium"></a>
## 7. From the Holeshot to the Podium
- Average starts, finishes, positions gained and lost

In [15]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
import numpy as np

# ── Build positions gained/lost data ─────────────────────────────────────────
pos_base = (
    df.dropna(subset=["lap", "place"])
    .copy()
)
pos_base["lap"]   = pos_base["lap"].astype(float)
pos_base["place"] = pos_base["place"].astype(float)
pos_base["year"]  = pos_base["year"].astype(str)
pos_base["moto"]  = pos_base["moto"].astype(str)

pos_base = pos_base.sort_values(["race_id", "name", "lap"])

# Lap-to-lap position change
pos_base["prev_place"] = (
    pos_base.groupby(["race_id", "name"], observed=True)["place"]
    .shift(1)
)
pos_base["next_lap"] = (
    pos_base.groupby(["race_id", "name"], observed=True)["lap"]
    .shift(1)
)

# Only consecutive laps
pos_base = pos_base[
    pos_base["prev_place"].notna() &
    (pos_base["lap"] == pos_base["next_lap"] + 1)
].copy()

# Positions gained (moved forward) and lost (moved backward)
pos_base["pos_gained"] = (pos_base["prev_place"] - pos_base["place"]).clip(lower=0)
pos_base["pos_lost"]   = (pos_base["place"] - pos_base["prev_place"]).clip(lower=0)

# ── Avg start position (place at end of lap 1) ────────────────────────────────
lap1_place = (
    df[df["lap"] == 1]
    .dropna(subset=["place"])
    .groupby(["race_id", "name"], observed=True)["place"]
    .first()
    .reset_index(name="lap1_place")
)

# ── Aggregate per rider per race ──────────────────────────────────────────────
race_agg = (
    pos_base.groupby(["race_id", "name", "class_label", "year"], observed=True)
    .agg(
        total_pos_gained=("pos_gained", "sum"),
        total_pos_lost=("pos_lost", "sum"),
    )
    .reset_index()
)

# Join lap 1 place and finish position
race_agg = race_agg.merge(lap1_place, on=["race_id", "name"], how="left")
race_agg = race_agg.merge(
    df.drop_duplicates(subset=["race_id", "name"])[["race_id", "name", "finish_position"]],
    on=["race_id", "name"], how="left"
)

# ── Aggregate to rider season level ──────────────────────────────────────────
rider_agg = (
    race_agg.groupby(["name", "class_label", "year"], observed=True)
    .agg(
        total_motos=("race_id", "count"),
        avg_start=("lap1_place", "mean"),
        total_pos_gained=("total_pos_gained", "sum"),
        total_pos_lost=("total_pos_lost", "sum"),
        avg_finish=("finish_position", "mean"),
    )
    .reset_index()
)

rider_agg["pos_gained_per_moto"] = (rider_agg["total_pos_gained"] / rider_agg["total_motos"]).round(2)
rider_agg["pos_lost_per_moto"]   = (rider_agg["total_pos_lost"]   / rider_agg["total_motos"]).round(2)
rider_agg["gain_loss_ratio"]     = (
    rider_agg["pos_gained_per_moto"] /
    rider_agg["pos_lost_per_moto"].replace(0, np.nan)
).round(2)
rider_agg["avg_start"]           = rider_agg["avg_start"].round(1)
rider_agg["avg_finish"]          = rider_agg["avg_finish"].round(1)
rider_agg["total_pos_gained"]    = rider_agg["total_pos_gained"].astype(int)
rider_agg["total_pos_lost"]      = rider_agg["total_pos_lost"].astype(int)

# ── Filter options ────────────────────────────────────────────────────────────
all_years   = sorted(rider_agg["year"].astype(str).unique())
all_classes = ["450", "250", "WMX"]

# ── Widgets ───────────────────────────────────────────────────────────────────
year_sel = widgets.Dropdown(
    options=all_years,
    value=all_years[-1],
    description="Year:",
    layout=widgets.Layout(width="200px")
)
class_sel = widgets.Dropdown(
    options=all_classes,
    value="450",
    description="Class:",
    layout=widgets.Layout(width="200px")
)
top_n_sel = widgets.IntSlider(
    value=20, min=5, max=50, step=5,
    description="Top N:",
    layout=widgets.Layout(width="300px")
)
sort_sel = widgets.Dropdown(
    options=[
        ("Avg Finish",       "avg_finish"),
        ("Avg Start",        "avg_start"),
        ("Total Motos",      "total_motos"),
        ("Total Pos Gained", "total_pos_gained"),
        ("Total Pos Lost",   "total_pos_lost"),
        ("Pos Gained/Moto",  "pos_gained_per_moto"),
        ("Pos Lost/Moto",    "pos_lost_per_moto"),
        ("Gain/Loss Ratio",  "gain_loss_ratio"),
    ],
    value="avg_finish",
    description="Sort by:",
    layout=widgets.Layout(width="280px")
)

# Default sort direction per column
SORT_ASCENDING = {
    "avg_finish":          True,
    "avg_start":           True,
    "total_motos":         False,
    "total_pos_gained":    False,
    "total_pos_lost":      False,
    "pos_gained_per_moto": False,
    "pos_lost_per_moto":   False,
    "gain_loss_ratio":     False,
}

btn    = widgets.Button(description="Update", button_style="primary")
output = widgets.Output()

# ── Update callback ───────────────────────────────────────────────────────────
def update(_):
    output.clear_output(wait=True)
    with output:

        subset = rider_agg[
            (rider_agg["year"] == year_sel.value) &
            (rider_agg["class_label"] == class_sel.value)
        ]

        if subset.empty:
            print("No data for selected filters.")
            return

        ascending = SORT_ASCENDING.get(sort_sel.value, True)

        result = (
            subset
            .sort_values(sort_sel.value, ascending=ascending, na_position="last")
            .head(top_n_sel.value)
            .reset_index(drop=True)
        )
        result.index += 1
        result = result.rename(columns={
            "name":                "Rider",
            "total_motos":         "Total Motos",
            "avg_start":           "Avg Start",
            "total_pos_gained":    "Total Pos Gained",
            "total_pos_lost":      "Total Pos Lost",
            "avg_finish":          "Avg Finish",
            "pos_gained_per_moto": "Pos Gained/Moto",
            "pos_lost_per_moto":   "Pos Lost/Moto",
            "gain_loss_ratio":     "Gain/Loss Ratio",
        })

        print(f"\n  Positions Gained/Lost — {class_sel.value} | {year_sel.value}  |  Sorted by: {sort_sel.label}\n")
        display(result[[
            "Rider", "Total Motos", "Avg Start",
            "Total Pos Gained", "Total Pos Lost",
            "Avg Finish", "Pos Gained/Moto", "Pos Lost/Moto",
            "Gain/Loss Ratio"
        ]])

btn.on_click(update)

# ── Layout ────────────────────────────────────────────────────────────────────
display(widgets.VBox([
    widgets.HBox([year_sel, class_sel, top_n_sel]),
    sort_sel,
    btn,
    output,
]))

print("\nNotes")
print("─────────────────")
print("• 'Avg Start' is position at the end of lap 1, not at the gate.")
print("  A rider who got the holeshot but got passed before lap 1 ended")
print("  will show a worse start than they actually had off the gate.")
print("• Positions gained/lost are summed lap-to-lap across the moto, not")
print("  start-to-finish. A rider who dropped 10 spots then recovered 8")
print("  registers as 10 lost AND 8 gained — total on-track action, not")
print("  net result.")
print("• Gain/Loss Ratio: positions gained per moto / positions lost per")
print("  moto. >1 = net gainer, <1 = net loser. Riders with zero positions")
print("  lost return NaN.")
print("• Top riders often have low gain totals — they start near the front")
print("  with less room to move forward. Ratio is most informative for")
print("  mid-pack riders.")

update(None)


Notes
─────────────────
• 'Avg Start' is position at the end of lap 1, not at the gate.
  A rider who got the holeshot but got passed before lap 1 ended
  will show a worse start than they actually had off the gate.
• Positions gained/lost are summed lap-to-lap across the moto, not
  start-to-finish. A rider who dropped 10 spots then recovered 8
  registers as 10 lost AND 8 gained — total on-track action, not
  net result.
• Gain/Loss Ratio: positions gained per moto / positions lost per
  moto. >1 = net gainer, <1 = net loser. Riders with zero positions
  lost return NaN.
• Top riders often have low gain totals — they start near the front
  with less room to move forward. Ratio is most informative for
  mid-pack riders.


In [17]:
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np
from scipy import stats
from statsmodels.nonparametric.smoothers_lowess import lowess

# ── Positions / rider_agg setup (self-contained) ─────────────────────────────
pos_base = df.dropna(subset=["lap", "place"]).copy()
pos_base["lap"]   = pos_base["lap"].astype(float)
pos_base["place"] = pos_base["place"].astype(float)
pos_base["year"]  = pos_base["year"].astype(str)
pos_base["moto"]  = pos_base["moto"].astype(str)

pos_base = pos_base.sort_values(["race_id", "name", "lap"])
pos_base["prev_place"] = pos_base.groupby(["race_id", "name"], observed=True)["place"].shift(1)
pos_base["next_lap"]   = pos_base.groupby(["race_id", "name"], observed=True)["lap"].shift(1)
pos_base = pos_base[
    pos_base["prev_place"].notna() & (pos_base["lap"] == pos_base["next_lap"] + 1)
].copy()
pos_base["pos_gained"] = (pos_base["prev_place"] - pos_base["place"]).clip(lower=0)
pos_base["pos_lost"]   = (pos_base["place"] - pos_base["prev_place"]).clip(lower=0)

lap1_place = (
    df[df["lap"] == 1].dropna(subset=["place"])
    .groupby(["race_id", "name"], observed=True)["place"].first()
    .reset_index(name="lap1_place")
)

race_agg = (
    pos_base.groupby(["race_id", "name", "class_label", "year"], observed=True)
    .agg(total_pos_gained=("pos_gained", "sum"), total_pos_lost=("pos_lost", "sum"))
    .reset_index()
)
race_agg = race_agg.merge(lap1_place, on=["race_id", "name"], how="left")
race_agg = race_agg.merge(
    df.drop_duplicates(subset=["race_id", "name"])[["race_id", "name", "finish_position"]],
    on=["race_id", "name"], how="left"
)

rider_agg = (
    race_agg.groupby(["name", "class_label", "year"], observed=True)
    .agg(
        total_motos=("race_id", "count"),
        avg_start=("lap1_place", "mean"),
        total_pos_gained=("total_pos_gained", "sum"),
        total_pos_lost=("total_pos_lost", "sum"),
        avg_finish=("finish_position", "mean"),
    )
    .reset_index()
)
rider_agg["pos_gained_per_moto"] = (rider_agg["total_pos_gained"] / rider_agg["total_motos"]).round(2)
rider_agg["pos_lost_per_moto"]   = (rider_agg["total_pos_lost"]   / rider_agg["total_motos"]).round(2)
rider_agg["gain_loss_ratio"]     = (
    rider_agg["pos_gained_per_moto"] / rider_agg["pos_lost_per_moto"].replace(0, np.nan)
).round(2)
rider_agg["avg_start"]        = rider_agg["avg_start"].round(1)
rider_agg["avg_finish"]       = rider_agg["avg_finish"].round(1)
rider_agg["total_pos_gained"] = rider_agg["total_pos_gained"].astype(int)
rider_agg["total_pos_lost"]   = rider_agg["total_pos_lost"].astype(int)

all_years_pg = sorted(rider_agg["year"].astype(str).unique())

year_sel_pg = widgets.Dropdown(
    options=all_years_pg,
    value=all_years_pg[-1],
    description="Year:",
    layout=widgets.Layout(width="200px")
)

btn_pg    = widgets.Button(description="Update", button_style="primary")
output_pg = widgets.Output()

CLASS_COLORS = {"450": "#E8641A", "250": "#1A7FE8", "WMX": "#2ECC71"}

def update_pg(_):
    output_pg.clear_output(wait=True)
    with output_pg:

        fig = go.Figure()

        for cls in ["450", "250", "WMX"]:
            subset = rider_agg[
                (rider_agg["year"] == year_sel_pg.value) &
                (rider_agg["class_label"] == cls)
            ].dropna(subset=["avg_finish", "pos_gained_per_moto"])

            if subset.empty:
                continue

            color = CLASS_COLORS[cls]

            # Scatter points
            fig.add_trace(go.Scatter(
                x=subset["avg_finish"],
                y=subset["pos_gained_per_moto"],
                mode="markers",
                name=cls,
                marker=dict(color=color, size=7, opacity=0.6),
                legendgroup=cls,
                hovertemplate=(
                    "<b>%{customdata}</b><br>"
                    "Avg Finish: %{x}<br>"
                    "Pos Gained/Moto: %{y}<extra></extra>"
                ),
                customdata=subset["name"],
            ))

            # LOWESS smooth line
            sorted_subset = subset.sort_values("avg_finish")
            smoothed = lowess(
                sorted_subset["pos_gained_per_moto"],
                sorted_subset["avg_finish"],
                frac=0.5,
                return_sorted=True
            )

            fig.add_trace(go.Scatter(
                x=smoothed[:, 0],
                y=smoothed[:, 1],
                mode="lines",
                name=f"{cls} trend",
                line=dict(color=color, width=2.5, dash="solid"),
                legendgroup=cls,
                hoverinfo="skip",
            ))

        fig.update_layout(
            title=f"Avg Finish vs Pos Gained/Moto — {year_sel_pg.value}",
            xaxis=dict(
                title="Avg Finish",
                autorange=True,
            ),
            yaxis=dict(
                title="Pos Gained / Moto",
            ),
            height=550,
            width=950,
            margin=dict(l=60, r=60, t=60, b=60),
            legend=dict(title="Class"),
            hovermode="closest",
        )
        fig.show()

btn_pg.on_click(update_pg)

display(widgets.VBox([
    widgets.HBox([year_sel_pg, btn_pg]),
    output_pg,
]))

print("\nNotes")
print("─────────────────")
print("• Both axes are rider-season aggregates from the same data, so")
print("  this is a descriptive relationship, not a causal one — a rider's")
print("  finish position is partly determined by how many positions they")
print("  gained.")
print("• Trend lines are LOWESS-smoothed showing the geometric ceiling")
print("  effect: front-runners can't gain much (nowhere to go) and")
print("  back-runners can't gain much (too slow to pass).")
print("• The peak of the curve isn't a 'sweet spot' — it's the field")
print("  position where the most movement is geometrically possible.")


update_pg(None)


Notes
─────────────────
• Both axes are rider-season aggregates from the same data, so
  this is a descriptive relationship, not a causal one — a rider's
  finish position is partly determined by how many positions they
  gained.
• Trend lines are LOWESS-smoothed showing the geometric ceiling
  effect: front-runners can't gain much (nowhere to go) and
  back-runners can't gain much (too slow to pass).
• The peak of the curve isn't a 'sweet spot' — it's the field
  position where the most movement is geometrically possible.


In [18]:
import plotly.graph_objects as go
from scipy import stats
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np

# ── Positions / rider_agg setup (self-contained) ─────────────────────────────
pos_base = df.dropna(subset=["lap", "place"]).copy()
pos_base["lap"]   = pos_base["lap"].astype(float)
pos_base["place"] = pos_base["place"].astype(float)
pos_base["year"]  = pos_base["year"].astype(str)
pos_base["moto"]  = pos_base["moto"].astype(str)

pos_base = pos_base.sort_values(["race_id", "name", "lap"])
pos_base["prev_place"] = pos_base.groupby(["race_id", "name"], observed=True)["place"].shift(1)
pos_base["next_lap"]   = pos_base.groupby(["race_id", "name"], observed=True)["lap"].shift(1)
pos_base = pos_base[
    pos_base["prev_place"].notna() & (pos_base["lap"] == pos_base["next_lap"] + 1)
].copy()
pos_base["pos_gained"] = (pos_base["prev_place"] - pos_base["place"]).clip(lower=0)
pos_base["pos_lost"]   = (pos_base["place"] - pos_base["prev_place"]).clip(lower=0)

lap1_place = (
    df[df["lap"] == 1].dropna(subset=["place"])
    .groupby(["race_id", "name"], observed=True)["place"].first()
    .reset_index(name="lap1_place")
)

race_agg = (
    pos_base.groupby(["race_id", "name", "class_label", "year"], observed=True)
    .agg(total_pos_gained=("pos_gained", "sum"), total_pos_lost=("pos_lost", "sum"))
    .reset_index()
)
race_agg = race_agg.merge(lap1_place, on=["race_id", "name"], how="left")
race_agg = race_agg.merge(
    df.drop_duplicates(subset=["race_id", "name"])[["race_id", "name", "finish_position"]],
    on=["race_id", "name"], how="left"
)

rider_agg = (
    race_agg.groupby(["name", "class_label", "year"], observed=True)
    .agg(
        total_motos=("race_id", "count"),
        avg_start=("lap1_place", "mean"),
        total_pos_gained=("total_pos_gained", "sum"),
        total_pos_lost=("total_pos_lost", "sum"),
        avg_finish=("finish_position", "mean"),
    )
    .reset_index()
)
rider_agg["pos_gained_per_moto"] = (rider_agg["total_pos_gained"] / rider_agg["total_motos"]).round(2)
rider_agg["pos_lost_per_moto"]   = (rider_agg["total_pos_lost"]   / rider_agg["total_motos"]).round(2)
rider_agg["gain_loss_ratio"]     = (
    rider_agg["pos_gained_per_moto"] / rider_agg["pos_lost_per_moto"].replace(0, np.nan)
).round(2)
rider_agg["avg_start"]        = rider_agg["avg_start"].round(1)
rider_agg["avg_finish"]       = rider_agg["avg_finish"].round(1)
rider_agg["total_pos_gained"] = rider_agg["total_pos_gained"].astype(int)
rider_agg["total_pos_lost"]   = rider_agg["total_pos_lost"].astype(int)

all_years_sf = sorted(rider_agg["year"].astype(str).unique())

year_sel_sf = widgets.Dropdown(
    options=all_years_sf,
    value=all_years_sf[-1],
    description="Year:",
    layout=widgets.Layout(width="200px")
)

btn_sf    = widgets.Button(description="Update", button_style="primary")
output_sf = widgets.Output()

CLASS_COLORS = {"450": "#E8641A", "250": "#1A7FE8", "WMX": "#2ECC71"}

def update_sf(_):
    output_sf.clear_output(wait=True)
    with output_sf:

        fig = go.Figure()

        all_vals = []

        for cls in ["450", "250", "WMX"]:
            subset = rider_agg[
                (rider_agg["year"] == year_sel_sf.value) &
                (rider_agg["class_label"] == cls)
            ].dropna(subset=["avg_start", "avg_finish"])

            if subset.empty:
                continue

            color = CLASS_COLORS[cls]
            all_vals += list(subset["avg_start"]) + list(subset["avg_finish"])

            # Scatter points
            fig.add_trace(go.Scatter(
                x=subset["avg_start"],
                y=subset["avg_finish"],
                mode="markers",
                name=cls,
                marker=dict(color=color, size=7, opacity=0.6),
                legendgroup=cls,
                hovertemplate=(
                    "<b>%{customdata}</b><br>"
                    "Avg Start: %{x}<br>"
                    "Avg Finish: %{y}<extra></extra>"
                ),
                customdata=subset["name"],
            ))

            # Linear trend line with R²
            slope, intercept, r, p, _ = stats.linregress(
                subset["avg_start"], subset["avg_finish"]
            )
            x_range = np.linspace(subset["avg_start"].min(), subset["avg_start"].max(), 100)
            y_range = slope * x_range + intercept

            fig.add_trace(go.Scatter(
                x=x_range,
                y=y_range,
                mode="lines",
                name=f"{cls} trend (R²={r**2:.2f})",
                line=dict(color=color, width=2, dash="dash"),
                legendgroup=cls,
                hoverinfo="skip",
            ))

        # y=x reference line
        if all_vals:
            ref_min = min(all_vals)
            ref_max = max(all_vals)
            fig.add_trace(go.Scatter(
                x=[ref_min, ref_max],
                y=[ref_min, ref_max],
                mode="lines",
                name="Start = Finish",
                line=dict(color="grey", width=1.5, dash="dot"),
                hoverinfo="skip",
            ))

        fig.update_layout(
            title=f"Avg Start vs Avg Finish — {year_sel_sf.value}",
            xaxis=dict(title="Avg Start"),
            yaxis=dict(title="Avg Finish"),
            height=550,
            width=950,
            margin=dict(l=60, r=60, t=60, b=60),
            legend=dict(title="Class"),
            hovermode="closest",
        )
        fig.show()

btn_sf.on_click(update_sf)

display(widgets.VBox([
    widgets.HBox([year_sel_sf, btn_sf]),
    output_sf,
]))

update_sf(None)

<a id="lap-consistency"></a>
## 8. Lap Consistency
- KDE curves
- Coefficient of variation (CV)

In [ ]:
import plotly.io as pio
import ipywidgets as widgets
from IPython.display import display, clear_output
import plotly.graph_objects as go
import numpy as np
from scipy.stats import gaussian_kde

pio.renderers.default = "notebook"


# ── Helper ────────────────────────────────────────────────────────────────────
def hex_to_rgba(hex_color, opacity=0.08):
    hex_color = hex_color.lstrip("#")
    r, g, b   = int(hex_color[0:2], 16), int(hex_color[2:4], 16), int(hex_color[4:6], 16)
    return f"rgba({r},{g},{b},{opacity})"

# ── Filter options ────────────────────────────────────────────────────────────
kde_years   = sorted(df["year"].astype(str).unique())
kde_classes = ["450", "250", "WMX"]
kde_rounds  = sorted(df["round"].astype(str).unique(), key=lambda x: int(x))
kde_motos   = sorted(df["moto"].astype(str).unique(), key=lambda x: int(x))

year_sel_kde = widgets.Dropdown(
    options=kde_years, value=kde_years[-1],
    description="Year:", layout=widgets.Layout(width="200px")
)
class_sel_kde = widgets.Dropdown(
    options=kde_classes, value="450",
    description="Class:", layout=widgets.Layout(width="200px")
)
round_sel_kde = widgets.Dropdown(
    options=kde_rounds, value=kde_rounds[0],
    description="Round:", layout=widgets.Layout(width="200px")
)
moto_sel_kde = widgets.Dropdown(
    options=kde_motos, value=kde_motos[0],
    description="Moto:", layout=widgets.Layout(width="200px")
)

RIDER_COLORS = ["#E8641A", "#1A7FE8", "#2ECC71", "#9B59B6", "#E74C3C"]

def plot_kde(year_val, class_val, round_val, moto_val):
    subset = df[
        (df["year"].astype(str)  == year_val) &
        (df["class_label"]       == class_val) &
        (df["round"].astype(str) == round_val) &
        (df["moto"].astype(str)  == moto_val)
    ].copy()

    if subset.empty:
        print("No data for selected filters.")
        return

    top5 = (
        subset.drop_duplicates(subset=["name"])
        .dropna(subset=["finish_position"])
        .sort_values("finish_position")
        .head(5)["name"]
        .tolist()
    )

    if not top5:
        print("No finish position data for selected race.")
        return

    track = subset["track"].iloc[0]
    laps  = subset[
        (subset["name"].isin(top5)) &
        (subset["lap"] > 1)
    ].dropna(subset=["lap_time"])

    if laps.empty:
        print("No lap time data for selected race.")
        return

    x_min   = laps["lap_time"].min() * 0.98
    x_max   = laps["lap_time"].max() * 1.02
    x_range = np.linspace(x_min, x_max, 500)

    fig = go.Figure()

    for i, rider in enumerate(top5):
        rider_laps = laps[laps["name"] == rider]["lap_time"].values
        if len(rider_laps) < 2:
            continue

        finish_pos = int(
            subset[subset["name"] == rider]["finish_position"]
            .dropna().iloc[0]
        )
        color   = RIDER_COLORS[i]
        kde     = gaussian_kde(rider_laps, bw_method=0.3)
        density = kde(x_range)
        median  = np.median(rider_laps)
        peak    = kde(np.array([median]))[0]

        fig.add_trace(go.Scatter(
            x=x_range,
            y=density,
            mode="lines",
            name=f"P{finish_pos} — {rider} (med: {median:.2f}s)",
            line=dict(color=color, width=2),
            fill="tozeroy",
            fillcolor=hex_to_rgba(color, 0.08),
            hovertemplate=(
                f"<b>P{finish_pos} — {rider}</b><br>"
                "Lap Time: %{x:.2f}s<br>"
                "Density: %{y:.4f}<extra></extra>"
            ),
        ))

    fig.update_layout(
        title=f"Lap Time Distribution — {class_val} | {track} | Round {round_val} Moto {moto_val}",
        xaxis=dict(title="Lap Time (s)"),
        yaxis=dict(title="Density", showticklabels=False),
        height=500,
        width=950,
        margin=dict(l=60, r=60, t=60, b=60),
        legend=dict(title="Rider"),
        hovermode="x unified",
    )
    display(fig)

btn_kde = widgets.Button(description="Update", button_style="primary")
output_kde = widgets.Output()

def update_kde(_):
    output_kde.clear_output(wait=True)
    with output_kde:
        plot_kde(year_sel_kde.value, class_sel_kde.value, round_sel_kde.value, moto_sel_kde.value)

btn_kde.on_click(update_kde)

display(widgets.VBox([
    widgets.HBox([year_sel_kde, class_sel_kde, round_sel_kde, moto_sel_kde]),
    btn_kde,
    output_kde,
]))

update_kde(None)

print("\nNotes")
print("─────────────────")
print("• Lap 1 is excluded; it's shorter than a full lap and therefore not comparable.")
print("• Curves are KDE (kernel density) estimates with a tight")
print("  bandwidth (0.3), so they track actual lap times closely")
print("  rather than heavily smoothing them.")
print("• Y-axis density values aren't directly interpretable — they")
print("  integrate to 1 across the x-axis. Curve shape and position")
print("  are what matter.")
print("• Lap times are absolute (seconds), so this chart compares")
print("  riders within ONE moto only. For cross-race or cross-track")
print("  comparisons, see the standardized lap time section.")


Notes
─────────────────
• Lap 1 is excluded; it's shorter than a full lap and therefore not comparable.
• Curves are KDE (kernel density) estimates with a tight
  bandwidth (0.3), so they track actual lap times closely
  rather than heavily smoothing them.
• Y-axis density values aren't directly interpretable — they
  integrate to 1 across the x-axis. Curve shape and position
  are what matter.
• Lap times are absolute (seconds), so this chart compares
  riders within ONE moto only. For cross-race or cross-track
  comparisons, see the standardized lap time section.


In [20]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np

# ── Build consistency data ────────────────────────────────────────────────────
cons_base = (
    df.dropna(subset=["lap", "lap_time"])
    .copy()
)
cons_base["lap"]  = cons_base["lap"].astype(float)
cons_base["year"] = cons_base["year"].astype(str)

# Apply 75% threshold
max_lap_cons = (
    cons_base.groupby("race_id", observed=True)["lap"]
    .max()
    .rename("max_lap")
    .reset_index()
)
cons_base = cons_base.merge(max_lap_cons, on="race_id", how="left")
cons_base["threshold"] = (cons_base["max_lap"] * 0.75).apply(np.floor)

laps_cons = (
    cons_base.groupby(["race_id", "name"], observed=True)["lap"]
    .max()
    .rename("laps_completed")
    .reset_index()
)
cons_base = cons_base.merge(laps_cons, on=["race_id", "name"], how="left")
cons_base = cons_base[cons_base["laps_completed"] >= cons_base["threshold"]].copy()

# Exclude lap 1
cons_base = cons_base[cons_base["lap"] > 1].copy()

# CV per rider per race
cv_per_race = (
    cons_base.groupby(["race_id", "name", "class_label", "year"], observed=True)["lap_time"]
    .agg(mean_lap="mean", std_lap="std")
    .reset_index()
)
cv_per_race["cv"] = (cv_per_race["std_lap"] / cv_per_race["mean_lap"] * 100)
cv_per_race = cv_per_race.dropna(subset=["cv"])

# Join finish position
cv_per_race = cv_per_race.merge(
    df.drop_duplicates(subset=["race_id", "name"])[["race_id", "name", "finish_position"]],
    on=["race_id", "name"], how="left"
)

# Average CV and avg finish per rider per year
rider_cv = (
    cv_per_race.groupby(["name", "class_label", "year"], observed=True)
    .agg(
        avg_cv=("cv", "mean"),
        avg_finish=("finish_position", "mean"),
        eligible_motos=("race_id", "count"),
    )
    .reset_index()
)
rider_cv["avg_cv"]     = rider_cv["avg_cv"].round(2)
rider_cv["avg_finish"] = rider_cv["avg_finish"].round(1)

# Join total motos (before 75% threshold)
total_motos = (
    df.drop_duplicates(subset=["race_id", "name"])
    .assign(year=lambda x: x["year"].astype(str))
    .groupby(["name", "class_label", "year"], observed=True)
    .size()
    .reset_index(name="total_motos")
)
rider_cv = rider_cv.merge(total_motos, on=["name", "class_label", "year"], how="left")

# ── Filter options ────────────────────────────────────────────────────────────
all_years_cv   = sorted(rider_cv["year"].astype(str).unique())
all_classes_cv = ["450", "250", "WMX"]

# ── Widgets ───────────────────────────────────────────────────────────────────
year_sel_cv = widgets.Dropdown(
    options=all_years_cv,
    value=all_years_cv[-1],
    description="Year:",
    layout=widgets.Layout(width="200px")
)
class_sel_cv = widgets.Dropdown(
    options=all_classes_cv,
    value="450",
    description="Class:",
    layout=widgets.Layout(width="200px")
)
top_n_sel_cv = widgets.IntSlider(
    value=20, min=5, max=50, step=5,
    description="Top N:",
    layout=widgets.Layout(width="300px")
)
sort_sel_cv = widgets.Dropdown(
    options=[
        ("Avg CV (most consistent first)", "avg_cv_asc"),
        ("Avg CV (least consistent first)", "avg_cv_desc"),
        ("Avg Finish",                      "avg_finish"),
        ("Total Motos",                     "total_motos"),
        ("Eligible Motos",                  "eligible_motos"),
    ],
    value="avg_cv_asc",
    description="Sort by:",
    layout=widgets.Layout(width="320px")
)

btn_cv    = widgets.Button(description="Update", button_style="primary")
output_cv = widgets.Output()

# ── Update callback ───────────────────────────────────────────────────────────
def update_cv(_):
    output_cv.clear_output(wait=True)
    with output_cv:

        subset = rider_cv[
            (rider_cv["year"] == year_sel_cv.value) &
            (rider_cv["class_label"] == class_sel_cv.value)
        ]

        if subset.empty:
            print("No data for selected filters.")
            return

        sort_map = {
            "avg_cv_asc":     ("avg_cv",        True),
            "avg_cv_desc":    ("avg_cv",         False),
            "avg_finish":     ("avg_finish",     True),
            "total_motos":    ("total_motos",    False),
            "eligible_motos": ("eligible_motos", False),
        }
        sort_col, ascending = sort_map[sort_sel_cv.value]

        result = (
            subset
            .sort_values(sort_col, ascending=ascending, na_position="last")
            .head(top_n_sel_cv.value)
            .reset_index(drop=True)
        )
        result.index += 1
        result = result.rename(columns={
            "name":           "Rider",
            "avg_cv":         "Avg CV (%)",
            "avg_finish":     "Avg Finish",
            "total_motos":    "Total Motos",
            "eligible_motos": "Eligible Motos",
        })

        print(f"\n  Lap Consistency — {class_sel_cv.value} | {year_sel_cv.value}  |  Sorted by: {sort_sel_cv.label}\n")
        display(result[["Rider", "Total Motos", "Eligible Motos", "Avg CV (%)", "Avg Finish"]])

btn_cv.on_click(update_cv)

# ── Layout ────────────────────────────────────────────────────────────────────
display(widgets.VBox([
    widgets.HBox([year_sel_cv, class_sel_cv, top_n_sel_cv]),
    sort_sel_cv,
    btn_cv,
    output_cv,
]))

print("\nNotes")
print("─────────────────")
print("• CV = standard deviation of lap times / mean lap time, expressed")
print("  as a %. It normalizes consistency across different pace levels,")
print("  so a 2% CV means the same thing whether the rider's average lap")
print("  is 60s or 100s.")
print("• Computed per rider per moto, then averaged across motos. This")
print("  keeps CV measuring within-moto consistency rather than mixing in")
print("  track-to-track variation.")
print("• Riders must complete ≥75% of a moto's laps to be eligible; lap 1")
print("  is excluded as start laps are slower and noisier.")
print("• 'Total Motos' is how many motos the rider entered. 'Eligible")
print("  Motos' is how many met the 75% threshold. A large gap between")
print("  them indicates an unreliable finisher.")

update_cv(None)


Notes
─────────────────
• CV = standard deviation of lap times / mean lap time, expressed
  as a %. It normalizes consistency across different pace levels,
  so a 2% CV means the same thing whether the rider's average lap
  is 60s or 100s.
• Computed per rider per moto, then averaged across motos. This
  keeps CV measuring within-moto consistency rather than mixing in
  track-to-track variation.
• Riders must complete ≥75% of a moto's laps to be eligible; lap 1
  is excluded as start laps are slower and noisier.
• 'Total Motos' is how many motos the rider entered. 'Eligible
  Motos' is how many met the 75% threshold. A large gap between
  them indicates an unreliable finisher.


<a id="lap-comparisons-to-race-leader"></a>
## 9. Lap Comparisons to Race Leader


In [21]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
import numpy as np

# ── Filter options ────────────────────────────────────────────────────────────
lc_years   = sorted(df["year"].astype(str).unique())
lc_classes = ["450", "250", "WMX"]
lc_rounds  = sorted(df["round"].astype(str).unique(), key=lambda x: int(x))
lc_motos   = sorted(df["moto"].astype(str).unique(), key=lambda x: int(x))

year_sel_lc = widgets.Dropdown(
    options=lc_years, value=lc_years[-1],
    description="Year:", layout=widgets.Layout(width="200px")
)
class_sel_lc = widgets.Dropdown(
    options=lc_classes, value="450",
    description="Class:", layout=widgets.Layout(width="200px")
)
round_sel_lc = widgets.Dropdown(
    options=lc_rounds, value=lc_rounds[0],
    description="Round:", layout=widgets.Layout(width="200px")
)
moto_sel_lc = widgets.Dropdown(
    options=lc_motos, value=lc_motos[0],
    description="Moto:", layout=widgets.Layout(width="200px")
)
rider_sel_lc = widgets.Dropdown(
    options=[], value=None,
    description="Rider:", layout=widgets.Layout(width="250px")
)

btn_lc    = widgets.Button(description="Update", button_style="primary")
output_lc = widgets.Output()

# ── Update rider dropdown when race filters change ────────────────────────────
def update_rider_options(_=None):
    subset = df[
        (df["year"].astype(str)  == year_sel_lc.value) &
        (df["class_label"]       == class_sel_lc.value) &
        (df["round"].astype(str) == round_sel_lc.value) &
        (df["moto"].astype(str)  == moto_sel_lc.value)
    ]
    riders        = sorted(subset["name"].dropna().unique())
    current_rider = rider_sel_lc.value
    rider_sel_lc.options = riders
    if current_rider in riders:
        rider_sel_lc.value = current_rider
    else:
        rider_sel_lc.value = riders[0] if riders else None

for sel in [year_sel_lc, class_sel_lc, round_sel_lc, moto_sel_lc]:
    sel.observe(lambda change: update_rider_options(), names="value")

update_rider_options()

# ── Update callback ───────────────────────────────────────────────────────────
def update_lc(_):
    output_lc.clear_output(wait=True)
    with output_lc:

        if not rider_sel_lc.value:
            print("No rider selected.")
            return

        subset = df[
            (df["year"].astype(str)  == year_sel_lc.value) &
            (df["class_label"]       == class_sel_lc.value) &
            (df["round"].astype(str) == round_sel_lc.value) &
            (df["moto"].astype(str)  == moto_sel_lc.value)
        ].copy()

        if subset.empty:
            print("No data for selected filters.")
            return

        # Get leader lap times (place == 1 on each lap)
        leader_laps = (
            subset[subset["place"] == 1]
            .dropna(subset=["lap", "lap_time"])
            [["lap", "lap_time"]]
            .rename(columns={"lap_time": "leader_lap_time"})
            .drop_duplicates(subset=["lap"])
        )

        # Get selected rider lap times
        rider_laps = (
            subset[subset["name"] == rider_sel_lc.value]
            .dropna(subset=["lap", "lap_time"])
            [["lap", "lap_time", "place"]]
            .sort_values("lap")
            .copy()
        )

        if rider_laps.empty:
            print(f"No lap time data for {rider_sel_lc.value}.")
            return

        # Difference to previous lap
        rider_laps["diff_to_last"] = rider_laps["lap_time"].diff()

        # Join leader lap times
        rider_laps = rider_laps.merge(leader_laps, on="lap", how="left")
        rider_laps["diff_to_leader"] = rider_laps["lap_time"] - rider_laps["leader_lap_time"]

        # Format
        result = rider_laps[["lap", "place", "lap_time", "diff_to_last", "diff_to_leader"]].copy()
        result["lap"]            = result["lap"].astype(int)
        result["place"]          = result["place"].astype("Int64")
        result["lap_time"]       = result["lap_time"].round(3)
        result["diff_to_last"]   = result["diff_to_last"].round(3)
        result["diff_to_leader"] = result["diff_to_leader"].round(3)

        result = result.rename(columns={
            "lap":             "Lap",
            "place":           "Place",
            "lap_time":        "Time",
            "diff_to_last":    "Difference to Last Lap",
            "diff_to_leader":  "Difference to Leader Lap",
        })

        # Get track and finish position for title
        track = subset["track"].iloc[0]
        fp    = subset[subset["name"] == rider_sel_lc.value]["finish_position"].dropna()
        finish_pos = int(fp.iloc[0]) if not fp.empty else "N/A"

        print(f"\n  {rider_sel_lc.value} — P{finish_pos} | {class_sel_lc.value} | {track} | Round {round_sel_lc.value} Moto {moto_sel_lc.value}\n")
        display(result.reset_index(drop=True))

btn_lc.on_click(update_lc)

# ── Layout ────────────────────────────────────────────────────────────────────
display(widgets.VBox([
    widgets.HBox([year_sel_lc, class_sel_lc, round_sel_lc, moto_sel_lc]),
    rider_sel_lc,
    btn_lc,
    output_lc,
]))

update_lc(None)

<a id="place-by-moto-quartile"></a>
## 10. Place by Moto Quartile


In [22]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
import numpy as np

# ── Build quartile data ───────────────────────────────────────────────────────
quart_base = (
    df.dropna(subset=["lap", "place"])
    .copy()
)
quart_base["lap"] = quart_base["lap"].astype(float)
quart_base["place"] = quart_base["place"].astype(float)
quart_base["year"] = quart_base["year"].astype(str)
quart_base["round"] = quart_base["round"].astype(str)
quart_base["moto"] = quart_base["moto"].astype(str)

# Max lap per race
max_lap_q = (
    quart_base.groupby("race_id", observed=True)["lap"]
    .max()
    .rename("max_lap")
    .reset_index()
)

# Race-level quartile lap numbers
race_laps = max_lap_q.copy()
race_laps["lap_start"] = 1.0
race_laps["lap_25"] = (race_laps["max_lap"] * 0.25 + 0.5).astype(int).astype(float)
race_laps["lap_50"] = (race_laps["max_lap"] * 0.50 + 0.5).astype(int).astype(float)
race_laps["lap_75"] = (race_laps["max_lap"] * 0.75 + 0.5).astype(int).astype(float)

# Join race lap targets onto quart_base
quart_base = quart_base.merge(
    race_laps[["race_id", "lap_start", "lap_25", "lap_50", "lap_75"]],
    on="race_id", how="left"
)


# Extract place at each quartile lap
def extract_quartile(quart_base, lap_col, out_col):
    return (
        quart_base[quart_base["lap"] == quart_base[lap_col]]
        .groupby(["race_id", "name"], observed=True)["place"]
        .first()
        .reset_index(name=out_col)
    )


start_q = extract_quartile(quart_base, "lap_start", "start")
q25 = extract_quartile(quart_base, "lap_25", "q25")
q50 = extract_quartile(quart_base, "lap_50", "q50")
q75 = extract_quartile(quart_base, "lap_75", "q75")

# Merge all quartiles together
quartile_df = start_q.merge(q25, on=["race_id", "name"], how="outer")
quartile_df = quartile_df.merge(q50, on=["race_id", "name"], how="outer")
quartile_df = quartile_df.merge(q75, on=["race_id", "name"], how="outer")

# Join finish position and race metadata
quartile_df = quartile_df.merge(
    df.drop_duplicates(subset=["race_id", "name"])[[
        "race_id", "name", "finish_position",
        "class_label", "year", "round", "moto", "track"
    ]],
    on=["race_id", "name"], how="left"
)
quartile_df["year"] = quartile_df["year"].astype(str)
quartile_df["round"] = quartile_df["round"].astype(str)
quartile_df["moto"] = quartile_df["moto"].astype(str)

# ── Filter options ────────────────────────────────────────────────────────────
all_years_q = sorted(quartile_df["year"].unique())
all_classes_q = ["450", "250", "WMX"]
all_riders_q = sorted(quartile_df["name"].dropna().unique())

# ── Widgets ───────────────────────────────────────────────────────────────────
year_sel_q = widgets.Dropdown(
    options=all_years_q, value=all_years_q[-1],
    description="Year:", layout=widgets.Layout(width="200px")
)
class_sel_q = widgets.Dropdown(
    options=all_classes_q, value="450",
    description="Class:", layout=widgets.Layout(width="200px")
)
rider_sel_q = widgets.Dropdown(
    options=[], value=None,
    description="Rider:", layout=widgets.Layout(width="250px")
)


def get_eligible_riders_q(year_val, class_val):
    mask = (
        (quartile_df["year"] == year_val) &
        (quartile_df["class_label"] == class_val)
    )
    return sorted(quartile_df[mask]["name"].dropna().unique())


def refresh_riders_q(*args):
    eligible = get_eligible_riders_q(year_sel_q.value, class_sel_q.value)
    current = rider_sel_q.value
    rider_sel_q.options = eligible
    rider_sel_q.value = current if current in eligible else (eligible[0] if eligible else None)


for w in [year_sel_q, class_sel_q]:
    w.observe(refresh_riders_q, names="value")

refresh_riders_q()

btn_q = widgets.Button(description="Update", button_style="primary")
output_q = widgets.Output()


# ── Update callback ───────────────────────────────────────────────────────────
def update_q(_):
    output_q.clear_output(wait=True)
    with output_q:

        if not rider_sel_q.value:
            print("No rider selected.")
            return

        subset = quartile_df[
            (quartile_df["year"] == year_sel_q.value) &
            (quartile_df["class_label"] == class_sel_q.value) &
            (quartile_df["name"] == rider_sel_q.value)
            ].copy()

        if subset.empty:
            print("No data for selected filters.")
            return

        subset = subset.sort_values(["round", "moto"], key=lambda x: x.astype(int))

        result = subset[["round", "moto", "track",
                         "start", "q25", "q50", "q75",
                         "finish_position"]].copy()

        for col in ["start", "q25", "q50", "q75", "finish_position"]:
            result[col] = result[col].astype("Int64")

        result = result.rename(columns={
            "round": "Round",
            "moto": "Moto",
            "track": "Track",
            "start": "Start",
            "q25": "25%",
            "q50": "50%",
            "q75": "75%",
            "finish_position": "Finish",
        })

        result = result[["Round", "Moto", "Track", "Start", "25%", "50%", "75%", "Finish"]]
        result = result.reset_index(drop=True)
        result.index += 1

        print(f"\n  Places by Moto Quartile — {rider_sel_q.value} | {class_sel_q.value} | {year_sel_q.value}\n")
        display(result)


btn_q.on_click(update_q)

# ── Layout ────────────────────────────────────────────────────────────────────
display(widgets.VBox([
    widgets.HBox([year_sel_q, class_sel_q]),
    widgets.HBox([rider_sel_q]),
    btn_q,
    output_q,
]))

print("\nNotes")
print("─────────────────")
print("• 'Start' is position at the end of lap 1, not gate position. The")
print("  actual gate-drop or holeshot position isn't in the data, so lap 1")
print("  finish is used as the closest proxy.")
print("• Quartile checkpoints (25%, 50%, 75%) are calculated per moto")
print("  based on each moto's total lap count, then rounded to the nearest")
print("  whole lap. A 12-lap moto's 25% checkpoint is lap 3; an 18-lap")
print("  moto's is lap 5.")
print("• Finish is the final scored position, which may include penalty")
print("  or DNF handling from the scoring system.")
print("• Riders who DNF'd before reaching a checkpoint will show <NA> in")
print("  that column.")

update_q(None)


Notes
─────────────────
• 'Start' is position at the end of lap 1, not gate position. The
  actual gate-drop or holeshot position isn't in the data, so lap 1
  finish is used as the closest proxy.
• Quartile checkpoints (25%, 50%, 75%) are calculated per moto
  based on each moto's total lap count, then rounded to the nearest
  whole lap. A 12-lap moto's 25% checkpoint is lap 3; an 18-lap
  moto's is lap 5.
• Finish is the final scored position, which may include penalty
  or DNF handling from the scoring system.
• Riders who DNF'd before reaching a checkpoint will show <NA> in
  that column.


In [23]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
import numpy as np

# ── Quartile setup (self-contained — reproduces Section 10 Cell 1) ───────────
quart_base = (
    df.dropna(subset=["lap", "place"])
    .copy()
)
quart_base["lap"]   = quart_base["lap"].astype(float)
quart_base["place"] = quart_base["place"].astype(float)
quart_base["year"]  = quart_base["year"].astype(str)
quart_base["round"] = quart_base["round"].astype(str)
quart_base["moto"]  = quart_base["moto"].astype(str)

max_lap_q = (
    quart_base.groupby("race_id", observed=True)["lap"]
    .max().rename("max_lap").reset_index()
)
race_laps = max_lap_q.copy()
race_laps["lap_start"] = 1.0
race_laps["lap_25"] = (race_laps["max_lap"] * 0.25 + 0.5).astype(int).astype(float)
race_laps["lap_50"] = (race_laps["max_lap"] * 0.50 + 0.5).astype(int).astype(float)
race_laps["lap_75"] = (race_laps["max_lap"] * 0.75 + 0.5).astype(int).astype(float)

quart_base = quart_base.merge(
    race_laps[["race_id", "lap_start", "lap_25", "lap_50", "lap_75"]],
    on="race_id", how="left"
)

def extract_quartile(quart_base, lap_col, out_col):
    return (
        quart_base[quart_base["lap"] == quart_base[lap_col]]
        .groupby(["race_id", "name"], observed=True)["place"]
        .first().reset_index(name=out_col)
    )

start_q = extract_quartile(quart_base, "lap_start", "start")
q25     = extract_quartile(quart_base, "lap_25",   "q25")
q50     = extract_quartile(quart_base, "lap_50",   "q50")
q75     = extract_quartile(quart_base, "lap_75",   "q75")

quartile_df = start_q.merge(q25, on=["race_id", "name"], how="outer")
quartile_df = quartile_df.merge(q50, on=["race_id", "name"], how="outer")
quartile_df = quartile_df.merge(q75, on=["race_id", "name"], how="outer")
quartile_df = quartile_df.merge(
    df.drop_duplicates(subset=["race_id", "name"])[[
        "race_id", "name", "finish_position",
        "class_label", "year", "round", "moto", "track"
    ]],
    on=["race_id", "name"], how="left"
)
quartile_df["year"]  = quartile_df["year"].astype(str)
quartile_df["round"] = quartile_df["round"].astype(str)
quartile_df["moto"]  = quartile_df["moto"].astype(str)


# ── Aggregate quartiles to season level ───────────────────────────────────────
season_quartiles = (
    quartile_df.groupby(["name", "class_label", "year"], observed=True)
    .agg(
        motos=("race_id", "count"),
        avg_start=("start", "mean"),
        avg_q25=("q25", "mean"),
        avg_q50=("q50", "mean"),
        avg_q75=("q75", "mean"),
        avg_finish=("finish_position", "mean"),
    )
    .reset_index()
)

for col in ["avg_start", "avg_q25", "avg_q50", "avg_q75", "avg_finish"]:
    season_quartiles[col] = season_quartiles[col].round(1)

season_quartiles["year"] = season_quartiles["year"].astype(str)

# ── Filter options ────────────────────────────────────────────────────────────
all_years_sq   = sorted(season_quartiles["year"].unique())
all_classes_sq = ["450", "250", "WMX"]

# ── Widgets ───────────────────────────────────────────────────────────────────
year_sel_sq = widgets.Dropdown(
    options=all_years_sq, value=all_years_sq[-1],
    description="Year:", layout=widgets.Layout(width="200px")
)
class_sel_sq = widgets.Dropdown(
    options=all_classes_sq, value="450",
    description="Class:", layout=widgets.Layout(width="200px")
)
top_n_sel_sq = widgets.IntSlider(
    value=20, min=5, max=50, step=5,
    description="Top N:", layout=widgets.Layout(width="300px")
)
sort_sel_sq = widgets.Dropdown(
    options=[
        ("Avg Finish",  "avg_finish"),
        ("Avg Start",   "avg_start"),
        ("Avg 25%",     "avg_q25"),
        ("Avg 50%",     "avg_q50"),
        ("Avg 75%",     "avg_q75"),
        ("Total Motos", "motos"),
    ],
    value="avg_finish",
    description="Sort by:", layout=widgets.Layout(width="280px")
)

SORT_ASCENDING = {
    "avg_finish": True,
    "avg_start":  True,
    "avg_q25":    True,
    "avg_q50":    True,
    "avg_q75":    True,
    "motos":      False,
}

btn_sq    = widgets.Button(description="Update", button_style="primary")
output_sq = widgets.Output()

# ── Update callback ───────────────────────────────────────────────────────────
def update_sq(_):
    output_sq.clear_output(wait=True)
    with output_sq:

        subset = season_quartiles[
            (season_quartiles["year"]        == year_sel_sq.value) &
            (season_quartiles["class_label"] == class_sel_sq.value)
        ]

        if subset.empty:
            print("No data for selected filters.")
            return

        ascending = SORT_ASCENDING.get(sort_sel_sq.value, True)

        result = (
            subset
            .sort_values(sort_sel_sq.value, ascending=ascending, na_position="last")
            .head(top_n_sel_sq.value)
            .reset_index(drop=True)
        )
        result.index += 1
        result = result.rename(columns={
            "name":        "Rider",
            "motos":       "Motos",
            "avg_start":   "Avg Start",
            "avg_q25":     "Avg 25%",
            "avg_q50":     "Avg 50%",
            "avg_q75":     "Avg 75%",
            "avg_finish":  "Avg Finish",
        })

        print(f"\n  Season Quartile Averages — {class_sel_sq.value} | {year_sel_sq.value}  |  Sorted by: {sort_sel_sq.label}\n")
        display(result[["Rider", "Motos", "Avg Start", "Avg 25%", "Avg 50%", "Avg 75%", "Avg Finish"]])

btn_sq.on_click(update_sq)

# ── Layout ────────────────────────────────────────────────────────────────────
display(widgets.VBox([
    widgets.HBox([year_sel_sq, class_sel_sq, top_n_sel_sq]),
    sort_sel_sq,
    btn_sq,
    output_sq,
]))

print("\nNotes")
print("─────────────────")
print("• NAs from DNFs are excluded from the average rather than imputed.")
print("  This means the denominator differs column-by-column for the")
print("  same rider: a rider who entered 16 motos but DNF'd before the")
print("  75% checkpoint in 3 of them has Avg 75% averaged over 13 motos")
print("  while Avg Start is averaged over all 16.")
print("• DNF-prone riders can look artificially better at later")
print("  checkpoints — the bad late-race laps that triggered the DNF")
print("  simply don't exist in the data, so they're not averaged in.")
print("• 'Motos' counts every moto where the rider was in the data at")
print("  all (at least one lap recorded). It's not the count of motos")
print("  where they completed all four checkpoints.")

update_sq(None)


Notes
─────────────────
• NAs from DNFs are excluded from the average rather than imputed.
  This means the denominator differs column-by-column for the
  same rider: a rider who entered 16 motos but DNF'd before the
  75% checkpoint in 3 of them has Avg 75% averaged over 13 motos
  while Avg Start is averaged over all 16.
• DNF-prone riders can look artificially better at later
  checkpoints — the bad late-race laps that triggered the DNF
  simply don't exist in the data, so they're not averaged in.
• 'Motos' counts every moto where the rider was in the data at
  all (at least one lap recorded). It's not the count of motos
  where they completed all four checkpoints.


<a id="standardizing-lap-times"></a>
## 11. Standardizing lap times
- % off best
- Z-scores

In [ ]:
import plotly.io as pio
import ipywidgets as widgets
from IPython.display import display, clear_output
import plotly.graph_objects as go
import numpy as np
from scipy.stats import gaussian_kde, kurtosis

pio.renderers.default = "notebook"


# ── Build % off best data ─────────────────────────────────────────────────────
pob_base = (
    df.dropna(subset=["lap", "lap_time"])
    .copy()
)
pob_base["lap"]   = pob_base["lap"].astype(float)
pob_base["year"]  = pob_base["year"].astype(str)
pob_base["round"] = pob_base["round"].astype(str)
pob_base["moto"]  = pob_base["moto"].astype(str)

# Fastest lap per lap number per moto (lap-by-lap benchmark)
lap_best = (
    pob_base.groupby(["race_id", "lap"], observed=True)["lap_time"]
    .min()
    .rename("best_lap_time")
    .reset_index()
)
pob_base = pob_base.merge(lap_best, on=["race_id", "lap"], how="left")

# % off best for that lap number in that moto
pob_base["pct_off_best"] = (
    (pob_base["lap_time"] - pob_base["best_lap_time"]) /
    pob_base["best_lap_time"] * 100
)
pob_base = pob_base[pob_base["pct_off_best"] >= 0].copy()

# ── Helper ────────────────────────────────────────────────────────────────────
def hex_to_rgba(hex_color, opacity=0.08):
    hex_color = hex_color.lstrip("#")
    r, g, b   = int(hex_color[0:2], 16), int(hex_color[2:4], 16), int(hex_color[4:6], 16)
    return f"rgba({r},{g},{b},{opacity})"

# ── Filter options ────────────────────────────────────────────────────────────
pob_years   = sorted(pob_base["year"].unique())
pob_classes = ["450", "250", "WMX"]
pob_rounds  = ["All"] + sorted(pob_base["round"].unique(), key=lambda x: int(x))
pob_motos   = ["All"] + sorted(pob_base["moto"].unique(), key=lambda x: int(x))
pob_riders  = sorted(pob_base["name"].dropna().unique())

RIDER_COLORS = ["#E8641A", "#1A7FE8"]

# ── Widgets ───────────────────────────────────────────────────────────────────
year_sel_pob = widgets.Dropdown(
    options=pob_years, value=pob_years[-1],
    description="Year:", layout=widgets.Layout(width="200px")
)
class_sel_pob = widgets.Dropdown(
    options=pob_classes, value="450",
    description="Class:", layout=widgets.Layout(width="200px")
)
round_sel_pob = widgets.Dropdown(
    options=pob_rounds, value="All",
    description="Round:", layout=widgets.Layout(width="200px")
)
moto_sel_pob = widgets.Dropdown(
    options=pob_motos, value="All",
    description="Moto:", layout=widgets.Layout(width="200px")
)
rider_sel_a = widgets.Dropdown(
    options=[], value=None,
    description="Rider A:", layout=widgets.Layout(width="250px")
)
rider_sel_b = widgets.Dropdown(
    options=[], value=None,
    description="Rider B:", layout=widgets.Layout(width="250px")
)

def get_eligible_riders_pob(year_val, class_val, round_val, moto_val):
    mask = (
        (pob_base["year"]        == year_val) &
        (pob_base["class_label"] == class_val)
    )
    if round_val != "All":
        mask &= pob_base["round"] == round_val
    if moto_val != "All":
        mask &= pob_base["moto"] == moto_val
    return sorted(pob_base[mask]["name"].dropna().unique())

def refresh_riders_pob(*args):
    eligible = get_eligible_riders_pob(
        year_sel_pob.value, class_sel_pob.value,
        round_sel_pob.value, moto_sel_pob.value
    )

    current_a = rider_sel_a.value
    rider_sel_a.options = eligible
    rider_sel_a.value   = current_a if current_a in eligible else (eligible[0] if eligible else None)

    current_b = rider_sel_b.value
    rider_sel_b.options = eligible
    default_b = eligible[1] if len(eligible) > 1 else (eligible[0] if eligible else None)
    rider_sel_b.value   = current_b if current_b in eligible else default_b

for w in [year_sel_pob, class_sel_pob, round_sel_pob, moto_sel_pob]:
    w.observe(refresh_riders_pob, names="value")

refresh_riders_pob()

def plot_pob(year_val, class_val, round_val, moto_val, rider_a, rider_b):
    if not rider_a or not rider_b:
        print("Please select two riders.")
        return

    fig = go.Figure()
    all_vals = []

    for i, rider in enumerate([rider_a, rider_b]):
        mask = (
            (pob_base["year"]        == year_val) &
            (pob_base["class_label"] == class_val) &
            (pob_base["name"]        == rider)
        )
        if round_val != "All":
            mask &= pob_base["round"] == round_val
        if moto_val != "All":
            mask &= pob_base["moto"] == moto_val

        subset = pob_base[mask]["pct_off_best"].dropna().values

        if len(subset) < 2:
            print(f"Not enough data for {rider}.")
            continue

        color  = RIDER_COLORS[i]
        all_vals.extend(subset)

        # KDE
        kde     = gaussian_kde(subset, bw_method=0.3)
        x_range = np.linspace(0, np.percentile(subset, 99) * 1.1, 500)
        density = kde(x_range)

        # Stats
        kurt   = kurtosis(subset, fisher=False)
        p95    = np.percentile(subset, 95)
        median = np.median(subset)

        fig.add_trace(go.Scatter(
            x=x_range,
            y=density,
            mode="lines",
            name=rider,
            line=dict(color=color, width=2),
            fill="tozeroy",
            fillcolor=hex_to_rgba(color, 0.08),
            customdata=np.full(len(x_range), fill_value=0),
            hovertemplate=(
                f"<b>{rider}</b><br>"
                "% off best: %{x:.2f}%<br>"
                "Density: %{y:.4f}<br>"
                f"Kurtosis: {kurt:.2f}<br>"
                f"P95: {p95:.1f}%<br>"
                f"Median: {median:.1f}%"
                "<extra></extra>"
            ),
        ))

        # Median dotted vertical line
        median_density = kde(np.array([median]))[0]
        fig.add_trace(go.Scatter(
            x=[median, median],
            y=[0, median_density],
            mode="lines",
            line=dict(color=color, width=1.5, dash="dot"),
            showlegend=False,
            hoverinfo="skip",
        ))

    if not all_vals:
        return

    # Use the larger of the two riders' 95th percentiles for the x-axis cap
    p95_per_rider = []
    for rider in [rider_a, rider_b]:
        mask = (
            (pob_base["year"]        == year_val) &
            (pob_base["class_label"] == class_val) &
            (pob_base["name"]        == rider)
        )
        if round_val != "All":
            mask &= pob_base["round"] == round_val
        if moto_val != "All":
            mask &= pob_base["moto"] == moto_val
        rider_vals = pob_base[mask]["pct_off_best"].dropna().values
        if len(rider_vals) >= 2:
            p95_per_rider.append(np.percentile(rider_vals, 95))
    x_max = max(p95_per_rider) * 1.1 if p95_per_rider else np.percentile(all_vals, 95) * 1.1

    round_str = f"Round {round_val}" if round_val != "All" else "All Rounds"
    moto_str  = f"Moto {moto_val}"   if moto_val  != "All" else "All Motos"

    fig.update_layout(
        title=f"% Off Best Lap — {rider_a} vs {rider_b} | {class_val} | {year_val} | {round_str} | {moto_str}",
        xaxis=dict(title="% Off Best Lap (lap-by-lap)", range=[0, x_max]),
        yaxis=dict(title="Density", showticklabels=False),
        height=500,
        width=950,
        margin=dict(l=60, r=60, t=60, b=60),
        legend=dict(title="Rider"),
        hovermode="x unified",
    )
    display(fig)

btn_pob = widgets.Button(description="Update", button_style="primary")
output_pob = widgets.Output()

def update_pob(_):
    output_pob.clear_output(wait=True)
    with output_pob:
        plot_pob(
            year_sel_pob.value, class_sel_pob.value,
            round_sel_pob.value, moto_sel_pob.value,
            rider_sel_a.value, rider_sel_b.value,
        )

btn_pob.on_click(update_pob)

display(widgets.VBox([
    widgets.HBox([year_sel_pob, class_sel_pob, round_sel_pob, moto_sel_pob]),
    widgets.HBox([rider_sel_a, rider_sel_b]),
    btn_pob,
    output_pob,
]))

update_pob(None)

print("\nNotes")
print("─────────────────")
print("• Each lap is compared to the FASTEST lap recorded by anyone at")
print("  that same lap number in the same moto. So lap 5's benchmark is")
print("  the fastest lap 5 in that moto, not the fastest overall lap.")
print("  This lap-by-lap benchmarking controls for track evolution: the")
print("  surface changes (ruts deepen, lines develop) over the course of")
print("  a moto, so comparing every rider's lap 5 against the fastest")
print("  lap 5 is more honest than comparing against an early-moto best.")
print("• The x-axis is capped at the larger of the two riders' 95th")
print("  percentiles. The underlying KDE is fit on all the data — only")
print("  the visible range is truncated. Hover stats reflect the full")
print("  distribution.")
print("• KDE bandwidth is 0.3 — same as other distribution charts in the")
print("  notebook. Tight bandwidth tracks the data closely; widening it")
print("  would produce smoother but less faithful curves.")
print("")
print("Stat definitions")
print("─────────────────────────────")
print("• Kurtosis: tail heaviness vs a normal distribution. Normal = 3.")
print("  Values above 3 mean heavier tails (more outliers); below 3 means")
print("  thinner tails (fewer outliers, more uniform).")
print("• P95: 95% of this rider's laps were within this % off the fastest")
print("  lap for that lap number.")
print("• Median: half this rider's laps were within this % off best;")
print("  half were further off.")


Notes
─────────────────
• Each lap is compared to the FASTEST lap recorded by anyone at
  that same lap number in the same moto. So lap 5's benchmark is
  the fastest lap 5 in that moto, not the fastest overall lap.
  This lap-by-lap benchmarking controls for track evolution: the
  surface changes (ruts deepen, lines develop) over the course of
  a moto, so comparing every rider's lap 5 against the fastest
  lap 5 is more honest than comparing against an early-moto best.
• The x-axis is capped at the larger of the two riders' 95th
  percentiles. The underlying KDE is fit on all the data — only
  the visible range is truncated. Hover stats reflect the full
  distribution.
• KDE bandwidth is 0.3 — same as other distribution charts in the
  notebook. Tight bandwidth tracks the data closely; widening it
  would produce smoother but less faithful curves.

Stat definitions
─────────────────────────────
• Kurtosis: tail heaviness vs a normal distribution. Normal = 3.
  Values above 3 mean he

In [25]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
import numpy as np
from scipy import stats

# ── Build regression data ─────────────────────────────────────────────────────
reg_base = (
    df.dropna(subset=["lap", "lap_time"])
    .copy()
)
reg_base["lap"] = reg_base["lap"].astype(float)
reg_base["year"] = reg_base["year"].astype(str)

# Lap-by-lap best time per moto
lap_best_reg = (
    reg_base.groupby(["race_id", "lap"], observed=True)["lap_time"]
    .min()
    .rename("best_lap_time")
    .reset_index()
)
reg_base = reg_base.merge(lap_best_reg, on=["race_id", "lap"], how="left")
reg_base["pct_off_best"] = (
        (reg_base["lap_time"] - reg_base["best_lap_time"]) /
        reg_base["best_lap_time"] * 100
)
reg_base = reg_base[reg_base["pct_off_best"] >= 0].copy()


# ── Regression per rider per moto ─────────────────────────────────────────────
def calc_regression(group):
    if len(group) < 3:
        return pd.Series({"slope": np.nan, "intercept": np.nan})
    slope, intercept, _, _, _ = stats.linregress(group["lap"], group["pct_off_best"])
    return pd.Series({"slope": slope, "intercept": intercept})


rider_moto_reg = (
    reg_base.groupby(["race_id", "name", "class_label", "year"], observed=True)
    .apply(calc_regression, include_groups=False)
    .reset_index()
)

# ── Aggregate to season level ─────────────────────────────────────────────────
season_reg = (
    rider_moto_reg.groupby(["name", "class_label", "year"], observed=True)
    .agg(
        mean_slope=("slope", "mean"),
        std_slope=("slope", "std"),
        mean_intercept=("intercept", "mean"),
    )
    .reset_index()
)

season_reg["mean_slope"] = season_reg["mean_slope"].round(3)
season_reg["std_slope"] = season_reg["std_slope"].round(3)
season_reg["mean_intercept"] = season_reg["mean_intercept"].round(2)
season_reg["year"] = season_reg["year"].astype(str)

# Join total motos
total_motos_reg = (
    df.drop_duplicates(subset=["race_id", "name"])
    .assign(year=lambda x: x["year"].astype(str))
    .groupby(["name", "class_label", "year"], observed=True)
    .size()
    .reset_index(name="total_motos")
)
season_reg = season_reg.merge(total_motos_reg, on=["name", "class_label", "year"], how="left")

# ── Filter options ────────────────────────────────────────────────────────────
all_years_reg = sorted(season_reg["year"].unique())
all_classes_reg = ["450", "250", "WMX"]

# ── Widgets ───────────────────────────────────────────────────────────────────
year_sel_reg = widgets.Dropdown(
    options=all_years_reg, value=all_years_reg[-1],
    description="Year:", layout=widgets.Layout(width="200px")
)
class_sel_reg = widgets.Dropdown(
    options=all_classes_reg, value="450",
    description="Class:", layout=widgets.Layout(width="200px")
)
top_n_sel_reg = widgets.IntSlider(
    value=20, min=5, max=50, step=5,
    description="Top N:", layout=widgets.Layout(width="300px")
)
sort_sel_reg = widgets.Dropdown(
    options=[
        ("Mean Slope (most improvement first)", "mean_slope_asc"),
        ("Mean Slope (least improvement first)", "mean_slope_desc"),
        ("Std Dev of Slopes (most consistent)", "std_slope_asc"),
        ("Std Dev of Slopes (least consistent)", "std_slope_desc"),
        ("Mean Intercept (lowest first)", "mean_intercept_asc"),
        ("Mean Intercept (highest first)", "mean_intercept_desc"),
        ("Total Motos", "total_motos"),
    ],
    value="mean_slope_asc",
    description="Sort by:", layout=widgets.Layout(width="350px")
)

btn_reg = widgets.Button(description="Update", button_style="primary")
output_reg = widgets.Output()

SORT_MAP = {
    "mean_slope_asc": ("mean_slope", True),
    "mean_slope_desc": ("mean_slope", False),
    "std_slope_asc": ("std_slope", True),
    "std_slope_desc": ("std_slope", False),
    "mean_intercept_asc": ("mean_intercept", True),
    "mean_intercept_desc": ("mean_intercept", False),
    "total_motos": ("total_motos", False),
}


# ── Update callback ───────────────────────────────────────────────────────────
def update_reg(_):
    output_reg.clear_output(wait=True)
    with output_reg:

        subset = season_reg[
            (season_reg["year"] == year_sel_reg.value) &
            (season_reg["class_label"] == class_sel_reg.value)
            ]

        if subset.empty:
            print("No data for selected filters.")
            return

        sort_col, ascending = SORT_MAP[sort_sel_reg.value]

        result = (
            subset
            .sort_values(sort_col, ascending=ascending, na_position="last")
            .head(top_n_sel_reg.value)
            .reset_index(drop=True)
        )
        result.index += 1
        result = result.rename(columns={
            "name": "Rider",
            "total_motos": "Total Motos",
            "mean_slope": "Mean Slope",
            "std_slope": "Std Dev Slope",
            "mean_intercept": "Mean Intercept (%)",
        })

        print(
            f"\n  % Off Best Regression — {class_sel_reg.value} | {year_sel_reg.value}  |  Sorted by: {sort_sel_reg.label}\n")
        print("  Mean Slope: Avg change in % off best per lap (positive = getting further off pace)")
        print("  Std Dev Slope: Consistency of that trend across motos")
        print("  Mean Intercept: Estimated % off best at lap 0 (opening pace)\n")
        display(result[[
            "Rider", "Total Motos",
            "Mean Slope", "Std Dev Slope", "Mean Intercept (%)"
        ]])


btn_reg.on_click(update_reg)

# ── Layout ────────────────────────────────────────────────────────────────────
display(widgets.VBox([
    widgets.HBox([year_sel_reg, class_sel_reg, top_n_sel_reg]),
    sort_sel_reg,
    btn_reg,
    output_reg,
]))

print("\nNotes")
print("─────────────────")
print("• A linear regression is fit per rider per moto with lap number")
print("  as the x-axis and '% off best' as the y-axis. The slope is")
print("  the average change in % off best per lap.")
print("• Pct off best uses the lap-by-lap benchmark (fastest lap N in")
print("  that moto), so a rising slope means the rider falls further")
print("  behind the fastest lap as the moto progresses — not necessarily")
print("  that they're slowing down in absolute terms, just that the")
print("  fastest rider is pulling away on later laps.")
print("• Motos with fewer than 3 laps of data return NaN slopes (can't")
print("  fit a line on 2 points). These are excluded from the season")
print("  mean.")
print("• All laps are included, including lap 1. Lap 1 tends to be")
print("  noisier due to the gate drop and first-corner chaos, but it's")
print("  part of the pace story so it's kept in.")
print("• Mean Intercept is the theoretical % off best at lap 0. This is")
print("  an extrapolation (lap 0 doesn't exist) and should be read as")
print("  'opening pace tendency' rather than literal lap 0 deficit.")
print("• Std Dev of Slopes measures consistency of the trend across the")
print("  rider's motos — a low value means they drift the same way every")
print("  moto, a high value means their fade pattern varies week to week.")

update_reg(None)


Notes
─────────────────
• A linear regression is fit per rider per moto with lap number
  as the x-axis and '% off best' as the y-axis. The slope is
  the average change in % off best per lap.
• Pct off best uses the lap-by-lap benchmark (fastest lap N in
  that moto), so a rising slope means the rider falls further
  behind the fastest lap as the moto progresses — not necessarily
  that they're slowing down in absolute terms, just that the
  fastest rider is pulling away on later laps.
• Motos with fewer than 3 laps of data return NaN slopes (can't
  fit a line on 2 points). These are excluded from the season
  mean.
• All laps are included, including lap 1. Lap 1 tends to be
  noisier due to the gate drop and first-corner chaos, but it's
  part of the pace story so it's kept in.
• Mean Intercept is the theoretical % off best at lap 0. This is
  an extrapolation (lap 0 doesn't exist) and should be read as
  'opening pace tendency' rather than literal lap 0 deficit.
• Std Dev of Slope

In [ ]:
import plotly.io as pio
import ipywidgets as widgets
from IPython.display import display, clear_output
import plotly.graph_objects as go
import numpy as np
from scipy.stats import gaussian_kde, kurtosis

pio.renderers.default = "notebook"


# ── Build z-score data ────────────────────────────────────────────────────────
pob_base = (
    df.dropna(subset=["lap", "lap_time"])
    .copy()
)
pob_base["lap"]   = pob_base["lap"].astype(float)
pob_base["year"]  = pob_base["year"].astype(str)
pob_base["round"] = pob_base["round"].astype(str)
pob_base["moto"]  = pob_base["moto"].astype(str)

# Mean and std per lap per moto (lap-by-lap benchmark)
lap_stats = (
    pob_base.groupby(["race_id", "lap"], observed=True)["lap_time"]
    .agg(mean_lap_time="mean", std_lap_time="std")
    .reset_index()
)
pob_base = pob_base.merge(lap_stats, on=["race_id", "lap"], how="left")

# Z-score: how many SDs from the mean for that lap in that moto
# Drop rows where std is 0 or NaN (only one rider on that lap)
pob_base = pob_base[pob_base["std_lap_time"].notna() & (pob_base["std_lap_time"] > 0)].copy()
pob_base["z_score"] = (
    (pob_base["lap_time"] - pob_base["mean_lap_time"]) /
    pob_base["std_lap_time"]
)

# ── Helper ────────────────────────────────────────────────────────────────────
def hex_to_rgba(hex_color, opacity=0.08):
    hex_color = hex_color.lstrip("#")
    r, g, b   = int(hex_color[0:2], 16), int(hex_color[2:4], 16), int(hex_color[4:6], 16)
    return f"rgba({r},{g},{b},{opacity})"

# ── Filter options ────────────────────────────────────────────────────────────
pob_years   = sorted(pob_base["year"].unique())
pob_classes = ["450", "250", "WMX"]
pob_rounds  = ["All"] + sorted(pob_base["round"].unique(), key=lambda x: int(x))
pob_motos   = ["All"] + sorted(pob_base["moto"].unique(), key=lambda x: int(x))
pob_riders  = sorted(pob_base["name"].dropna().unique())

RIDER_COLORS = ["#E8641A", "#1A7FE8"]

# ── Widgets ───────────────────────────────────────────────────────────────────
year_sel_pob = widgets.Dropdown(
    options=pob_years, value=pob_years[-1],
    description="Year:", layout=widgets.Layout(width="200px")
)
class_sel_pob = widgets.Dropdown(
    options=pob_classes, value="450",
    description="Class:", layout=widgets.Layout(width="200px")
)
round_sel_pob = widgets.Dropdown(
    options=pob_rounds, value="All",
    description="Round:", layout=widgets.Layout(width="200px")
)
moto_sel_pob = widgets.Dropdown(
    options=pob_motos, value="All",
    description="Moto:", layout=widgets.Layout(width="200px")
)
rider_sel_a = widgets.Dropdown(
    options=[], value=None,
    description="Rider A:", layout=widgets.Layout(width="250px")
)
rider_sel_b = widgets.Dropdown(
    options=[], value=None,
    description="Rider B:", layout=widgets.Layout(width="250px")
)

def get_eligible_riders_pob(year_val, class_val, round_val, moto_val):
    mask = (
        (pob_base["year"]        == year_val) &
        (pob_base["class_label"] == class_val)
    )
    if round_val != "All":
        mask &= pob_base["round"] == round_val
    if moto_val != "All":
        mask &= pob_base["moto"] == moto_val
    return sorted(pob_base[mask]["name"].dropna().unique())

def refresh_riders_pob(*args):
    eligible = get_eligible_riders_pob(
        year_sel_pob.value, class_sel_pob.value,
        round_sel_pob.value, moto_sel_pob.value
    )

    current_a = rider_sel_a.value
    rider_sel_a.options = eligible
    rider_sel_a.value   = current_a if current_a in eligible else (eligible[0] if eligible else None)

    current_b = rider_sel_b.value
    rider_sel_b.options = eligible
    default_b = eligible[1] if len(eligible) > 1 else (eligible[0] if eligible else None)
    rider_sel_b.value   = current_b if current_b in eligible else default_b

for w in [year_sel_pob, class_sel_pob, round_sel_pob, moto_sel_pob]:
    w.observe(refresh_riders_pob, names="value")

refresh_riders_pob()

def plot_pob(year_val, class_val, round_val, moto_val, rider_a, rider_b):
    if not rider_a or not rider_b:
        print("Please select two riders.")
        return

    fig = go.Figure()
    all_vals = []

    for i, rider in enumerate([rider_a, rider_b]):
        mask = (
            (pob_base["year"]        == year_val) &
            (pob_base["class_label"] == class_val) &
            (pob_base["name"]        == rider)
        )
        if round_val != "All":
            mask &= pob_base["round"] == round_val
        if moto_val != "All":
            mask &= pob_base["moto"] == moto_val

        subset = pob_base[mask]["z_score"].dropna().values

        if len(subset) < 2:
            print(f"Not enough data for {rider}.")
            continue

        color  = RIDER_COLORS[i]
        all_vals.extend(subset)

        # KDE — span from ~p1 to ~p99 to accommodate negative values
        x_min   = np.percentile(subset, 1)
        x_max   = np.percentile(subset, 99)
        kde     = gaussian_kde(subset, bw_method=0.3)
        x_range = np.linspace(x_min * 1.1 if x_min < 0 else x_min * 0.9,
                               x_max * 1.1, 500)
        density = kde(x_range)

        # Stats
        kurt   = kurtosis(subset, fisher=False)
        p95    = np.percentile(subset, 95)
        median = np.median(subset)

        fig.add_trace(go.Scatter(
            x=x_range,
            y=density,
            mode="lines",
            name=rider,
            line=dict(color=color, width=2),
            fill="tozeroy",
            fillcolor=hex_to_rgba(color, 0.08),
            customdata=np.full(len(x_range), fill_value=0),
            hovertemplate=(
                f"<b>{rider}</b><br>"
                "Z-Score: %{x:.2f}<br>"
                "Density: %{y:.4f}<br>"
                f"Kurtosis: {kurt:.2f}<br>"
                f"P95: {p95:.2f}<br>"
                f"Median: {median:.2f}"
                "<extra></extra>"
            ),
        ))

        # Median dotted vertical line
        median_density = kde(np.array([median]))[0]
        fig.add_trace(go.Scatter(
            x=[median, median],
            y=[0, median_density],
            mode="lines",
            line=dict(color=color, width=1.5, dash="dot"),
            showlegend=False,
            hoverinfo="skip",
        ))

    if not all_vals:
        return

    global_min = np.percentile(all_vals, 1)
    global_max = np.percentile(all_vals, 99)
    x_lo = global_min * 1.15 if global_min < 0 else global_min * 0.85
    x_hi = global_max * 1.15

    round_str = f"Round {round_val}" if round_val != "All" else "All Rounds"
    moto_str  = f"Moto {moto_val}"   if moto_val  != "All" else "All Motos"

    fig.update_layout(
        title=f"Lap Time Z-Score — {rider_a} vs {rider_b} | {class_val} | {year_val} | {round_str} | {moto_str}",
        xaxis=dict(title="Z-Score (lap time vs. moto average for that lap)", range=[x_lo, x_hi]),
        yaxis=dict(title="Density", showticklabels=False),
        height=500,
        width=950,
        margin=dict(l=60, r=60, t=60, b=60),
        legend=dict(title="Rider"),
        hovermode="x unified",
    )
    display(fig)

btn_pob = widgets.Button(description="Update", button_style="primary")
output_pob = widgets.Output()

def update_pob(_):
    output_pob.clear_output(wait=True)
    with output_pob:
        plot_pob(
            year_sel_pob.value, class_sel_pob.value,
            round_sel_pob.value, moto_sel_pob.value,
            rider_sel_a.value, rider_sel_b.value,
        )

btn_pob.on_click(update_pob)

display(widgets.VBox([
    widgets.HBox([year_sel_pob, class_sel_pob, round_sel_pob, moto_sel_pob]),
    widgets.HBox([rider_sel_a, rider_sel_b]),
    btn_pob,
    output_pob,
]))

update_pob(None)

print("\nNotes")
print("─────────────────")
print("• Z-score = (rider's lap time - mean of all riders' lap N in that")
print("  moto) / standard deviation of all riders' lap N in that moto.")
print("• Computed per lap per moto, so the benchmark adapts to who was")
print("  on track and how tight the field was at that exact lap.")
print("• Negative z-score = faster than the field average on that lap.")
print("  Positive = slower. Z=0 is exactly average.")
print("• Laps where only one rider was on track (std=0 or NaN) are")
print("  dropped, since z-score is undefined.")
print("• Unlike % off best, this metric isn't bounded at 0 — it gives a")
print("  continuous scale from 'much faster' to 'much slower' relative")
print("  to the field.")
print("• KDE bandwidth is 0.3, same as other distribution charts.")


Notes
─────────────────
• Z-score = (rider's lap time - mean of all riders' lap N in that
  moto) / standard deviation of all riders' lap N in that moto.
• Computed per lap per moto, so the benchmark adapts to who was
  on track and how tight the field was at that exact lap.
• Negative z-score = faster than the field average on that lap.
  Positive = slower. Z=0 is exactly average.
• Laps where only one rider was on track (std=0 or NaN) are
  dropped, since z-score is undefined.
• Unlike % off best, this metric isn't bounded at 0 — it gives a
  continuous scale from 'much faster' to 'much slower' relative
  to the field.
• KDE bandwidth is 0.3, same as other distribution charts.


In [27]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
import numpy as np

# ── Build z-score data ────────────────────────────────────────────────────────
zscore_base = (
    df.dropna(subset=["lap", "lap_time"])
    .copy()
)
zscore_base["lap"]  = zscore_base["lap"].astype(float)
zscore_base["year"] = zscore_base["year"].astype(str)

# Lap-by-lap mean and std per moto
lap_stats_z = (
    zscore_base.groupby(["race_id", "lap"], observed=True)["lap_time"]
    .agg(mean_lap_time="mean", std_lap_time="std")
    .reset_index()
)
zscore_base = zscore_base.merge(lap_stats_z, on=["race_id", "lap"], how="left")

# Drop laps where std is 0 or NaN
zscore_base = zscore_base[
    zscore_base["std_lap_time"].notna() & (zscore_base["std_lap_time"] > 0)
].copy()

zscore_base["z_score"] = (
    (zscore_base["lap_time"] - zscore_base["mean_lap_time"]) /
    zscore_base["std_lap_time"]
)

# ── Aggregate to season level ─────────────────────────────────────────────────
# Lap 1 z-score per rider per moto, then average across season
lap1_z = (
    zscore_base[zscore_base["lap"] == 1]
    .groupby(["name", "class_label", "year"], observed=True)["z_score"]
    .mean()
    .rename("mean_z_lap1")
    .reset_index()
)

# Overall mean z-score across all laps
overall_z = (
    zscore_base
    .groupby(["name", "class_label", "year"], observed=True)["z_score"]
    .mean()
    .rename("mean_z_overall")
    .reset_index()
)

# Total motos per rider
total_motos_z = (
    df.drop_duplicates(subset=["race_id", "name"])
    .assign(year=lambda x: x["year"].astype(str))
    .groupby(["name", "class_label", "year"], observed=True)
    .size()
    .reset_index(name="total_motos")
)

season_z = (
    total_motos_z
    .merge(lap1_z,    on=["name", "class_label", "year"], how="left")
    .merge(overall_z, on=["name", "class_label", "year"], how="left")
)

season_z["mean_z_lap1"]    = season_z["mean_z_lap1"].round(3)
season_z["mean_z_overall"] = season_z["mean_z_overall"].round(3)
season_z["year"]           = season_z["year"].astype(str)

# ── Filter options ────────────────────────────────────────────────────────────
all_years_z   = sorted(season_z["year"].unique())
all_classes_z = ["450", "250", "WMX"]

# ── Widgets ───────────────────────────────────────────────────────────────────
year_sel_z = widgets.Dropdown(
    options=all_years_z, value=all_years_z[-1],
    description="Year:", layout=widgets.Layout(width="200px")
)
class_sel_z = widgets.Dropdown(
    options=all_classes_z, value="450",
    description="Class:", layout=widgets.Layout(width="200px")
)
top_n_sel_z = widgets.IntSlider(
    value=20, min=5, max=50, step=5,
    description="Top N:", layout=widgets.Layout(width="300px")
)
sort_sel_z = widgets.Dropdown(
    options=[
        ("Mean Z — Lap 1 (best starters first)",   "mean_z_lap1_asc"),
        ("Mean Z — Lap 1 (worst starters first)",  "mean_z_lap1_desc"),
        ("Mean Z — Overall (best overall first)",  "mean_z_overall_asc"),
        ("Mean Z — Overall (worst overall first)", "mean_z_overall_desc"),
        ("Total Motos",                            "total_motos"),
    ],
    value="mean_z_overall_asc",
    description="Sort by:", layout=widgets.Layout(width="350px")
)

btn_z    = widgets.Button(description="Update", button_style="primary")
output_z = widgets.Output()

SORT_MAP_Z = {
    "mean_z_lap1_asc":      ("mean_z_lap1",    True),
    "mean_z_lap1_desc":     ("mean_z_lap1",    False),
    "mean_z_overall_asc":   ("mean_z_overall", True),
    "mean_z_overall_desc":  ("mean_z_overall", False),
    "total_motos":          ("total_motos",    False),
}

# ── Update callback ───────────────────────────────────────────────────────────
def update_z(_):
    output_z.clear_output(wait=True)
    with output_z:

        subset = season_z[
            (season_z["year"]        == year_sel_z.value) &
            (season_z["class_label"] == class_sel_z.value)
        ]

        if subset.empty:
            print("No data for selected filters.")
            return

        sort_col, ascending = SORT_MAP_Z[sort_sel_z.value]

        result = (
            subset
            .sort_values(sort_col, ascending=ascending, na_position="last")
            .head(top_n_sel_z.value)
            .reset_index(drop=True)
        )
        result.index += 1
        result = result.rename(columns={
            "name":            "Rider",
            "total_motos":     "Total Motos",
            "mean_z_lap1":     "Mean Z — Lap 1",
            "mean_z_overall":  "Mean Z — Overall",
        })

        print(f"\n  Lap Time Z-Scores — {class_sel_z.value} | {year_sel_z.value}  |  Sorted by: {sort_sel_z.label}\n")
        print("  Mean Z — Lap 1:   Avg z-score on lap 1 across all motos (negative = faster than field average on opening lap)")
        print("  Mean Z — Overall: Avg z-score across all laps (negative = faster than field average overall)\n")
        display(result[["Rider", "Total Motos", "Mean Z — Lap 1", "Mean Z — Overall"]])

btn_z.on_click(update_z)

# ── Layout ────────────────────────────────────────────────────────────────────
display(widgets.VBox([
    widgets.HBox([year_sel_z, class_sel_z, top_n_sel_z]),
    sort_sel_z,
    btn_z,
    output_z,
]))

update_z(None)

<a id="traffic-effects"></a>
## 12. Traffic Effects
- Pace and place journey by traffic situation - sandwiched, chasing, being chased, or clean air

In [28]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import plotly.graph_objects as go
import pandas as pd
import numpy as np
from scipy.stats import gaussian_kde

# ── Build traffic base from pre-computed df columns ──────────────────────────
traffic_base = df.dropna(subset=["lap", "lap_time", "place"]).copy()
traffic_base["lap"]   = traffic_base["lap"].astype(float)
traffic_base["place"] = traffic_base["place"].astype(float)
traffic_base["year"]  = traffic_base["year"].astype(str)
traffic_base["round"] = traffic_base["round"].astype(str)
traffic_base["moto"]  = traffic_base["moto"].astype(str)

# Boolean condition columns derived from pre-computed traffic_state
traffic_base["cond_sandwiched"]   = traffic_base["traffic_state"] == "Sandwiched"
traffic_base["cond_chasing"]      = traffic_base["traffic_state"] == "Chasing"
traffic_base["cond_being_chased"] = traffic_base["traffic_state"] == "Being Chased"
traffic_base["cond_clear_air"]    = traffic_base["traffic_state"] == "Clear Air"

# ── Filter options ─────────────────────────────────────────────────────────────────
traffic_years = sorted(traffic_base["year"].unique())
traffic_classes = ["450", "250", "WMX"]
traffic_rounds = ["All"] + sorted(traffic_base["round"].unique(), key=lambda x: int(x))
traffic_motos = ["All"] + sorted(traffic_base["moto"].unique(), key=lambda x: int(x))


# ── Helper ─────────────────────────────────────────────────────────────────────
def hex_to_rgba(hex_color, opacity=0.08):
    hex_color = hex_color.lstrip("#")
    r, g, b = int(hex_color[0:2], 16), int(hex_color[2:4], 16), int(hex_color[4:6], 16)
    return f"rgba({r},{g},{b},{opacity})"


CONDITIONS = [
    ("Sandwiched", "cond_sandwiched", "#9B27AE"),
    ("Chasing", "cond_chasing", "#E8641A"),
    ("Being Chased", "cond_being_chased", "#E8C11A"),
    ("Clear Air", "cond_clear_air", "#1A7FE8"),
]

# ── Widgets ────────────────────────────────────────────────────────────────────
year_sel_tr = widgets.Dropdown(
    options=traffic_years, value=traffic_years[-1],
    description="Year:", layout=widgets.Layout(width="200px")
)
class_sel_tr = widgets.Dropdown(
    options=traffic_classes, value="450",
    description="Class:", layout=widgets.Layout(width="200px")
)
round_sel_tr = widgets.Dropdown(
    options=traffic_rounds, value="All",
    description="Round:", layout=widgets.Layout(width="200px")
)
moto_sel_tr = widgets.Dropdown(
    options=traffic_motos, value="All",
    description="Moto:", layout=widgets.Layout(width="200px")
)
rider_sel_tr = widgets.Dropdown(
    options=[], value=None,
    description="Rider:", layout=widgets.Layout(width="280px")
)


def get_eligible_riders(year_val, class_val, round_val, moto_val):
    mask = (
            (traffic_base["year"] == year_val) &
            (traffic_base["class_label"] == class_val) &
            (traffic_base["place"] == 1)
    )
    if round_val != "All":
        mask &= traffic_base["round"] == round_val
    if moto_val != "All":
        mask &= traffic_base["moto"] == moto_val

    laps_in_first = (
        traffic_base[mask]
        .groupby("name", observed=True)["lap"]
        .count()
    )
    return sorted(laps_in_first[laps_in_first > 2].index.tolist())


def refresh_riders(*args):
    eligible = get_eligible_riders(
        year_sel_tr.value, class_sel_tr.value,
        round_sel_tr.value, moto_sel_tr.value
    )
    current = rider_sel_tr.value
    rider_sel_tr.options = eligible
    rider_sel_tr.value = current if current in eligible else (eligible[0] if eligible else None)


for w in [year_sel_tr, class_sel_tr, round_sel_tr, moto_sel_tr]:
    w.observe(refresh_riders, names="value")

refresh_riders()


# ── Plot function ──────────────────────────────────────────────────────────────
def plot_traffic(year_val, class_val, round_val, moto_val, rider):
    if not rider:
        print("No eligible riders for selected filters (need >2 laps in 1st place).")
        return

    mask = (
            (traffic_base["year"] == year_val) &
            (traffic_base["class_label"] == class_val) &
            (traffic_base["name"] == rider)
    )
    if round_val != "All":
        mask &= traffic_base["round"] == round_val
    if moto_val != "All":
        mask &= traffic_base["moto"] == moto_val

    subset = traffic_base[mask]
    fig = go.Figure()
    all_vals = []

    for label, cond_col, color in CONDITIONS:
        vals = subset[subset[cond_col]]["z_score"].dropna().values

        if len(vals) < 2:
            continue

        all_vals.extend(vals)
        kde = gaussian_kde(vals, bw_method=0.3)
        x_min = np.percentile(vals, 1)
        x_max = np.percentile(vals, 99)
        x_range = np.linspace(
            x_min * 1.1 if x_min < 0 else x_min * 0.9,
            x_max * 1.1, 500
        )
        density = kde(x_range)
        median = np.median(vals)
        median_density = kde(np.array([median]))[0]

        fig.add_trace(go.Scatter(
            x=x_range,
            y=density,
            mode="lines",
            name=f"{label} (n={len(vals)}, median={median:.2f})",
            line=dict(color=color, width=2),
            fill="tozeroy",
            fillcolor=hex_to_rgba(color, 0.08),
            hovertemplate=(
                f"<b>{label}</b><br>"
                "Z-Score: %{x:.2f}<br>"
                "Density: %{y:.4f}"
                "<extra></extra>"
            ),
        ))

        fig.add_trace(go.Scatter(
            x=[median, median],
            y=[0, median_density],
            mode="lines",
            line=dict(color=color, width=1.5, dash="dot"),
            showlegend=False,
            hoverinfo="skip",
        ))

    if not all_vals:
        print(f"No data for {rider} with selected filters.")
        return

    global_min = np.percentile(all_vals, 1)
    global_max = np.percentile(all_vals, 99)
    x_lo = global_min * 1.15 if global_min < 0 else global_min * 0.85
    x_hi = global_max * 1.15

    round_str = f"Round {round_val}" if round_val != "All" else "All Rounds"
    moto_str = f"Moto {moto_val}" if moto_val != "All" else "All Motos"

    fig.update_layout(
        title=f"Traffic Conditions — {rider} | {class_val} | {year_val} | {round_str} | {moto_str}",
        xaxis=dict(
            title="Z-Score (lap time vs. moto average for that lap)",
            range=[x_lo, x_hi]
        ),
        yaxis=dict(title="Density", showticklabels=False),
        height=500,
        width=950,
        margin=dict(l=60, r=60, t=60, b=60),
        legend=dict(title="Condition"),
        hovermode="x unified",
    )
    display(fig)


btn_tr = widgets.Button(description="Update", button_style="primary")
output_tr = widgets.Output()

def update_tr(_):
    output_tr.clear_output(wait=True)
    with output_tr:
        plot_traffic(
            year_sel_tr.value, class_sel_tr.value,
            round_sel_tr.value, moto_sel_tr.value,
            rider_sel_tr.value,
        )

btn_tr.on_click(update_tr)

display(widgets.VBox([
    widgets.HBox([year_sel_tr, class_sel_tr, round_sel_tr, moto_sel_tr]),
    rider_sel_tr,
    btn_tr,
    output_tr,
]))

update_tr(None)

print("\nNotes")
print("─────────────────")
print("• Sandwiched: Someone within 2.0 seconds ahead AND someone within")
print("  2.0 seconds behind. Simultaneous attack and defense pressure —")
print("  the highest-cognitive-load condition.")
print("• Chasing: Someone within 2.0 seconds ahead, AND no one within")
print("  2.0 seconds behind. The rider is trying to make a pass with no")
print("  defensive pressure.")
print("• Being Chased: No one within 2.0 seconds ahead (or they're in")
print("  1st place), AND someone within 2.0 seconds behind. Defensive")
print("  pressure with clear track ahead.")
print("• Clear Air: No one within 2.0 seconds in either direction. The")
print("  rider is in a gap, racing the clock.")
print("• These four conditions are mutually exclusive and exhaustive —")
print("  every lap falls into exactly one category.")
print("• Only riders who led a lap (≥3 laps in 1st place) in the selected")
print("  filter window are shown in the dropdown. This restricts the")
print("  analysis to front-runners who experience all four conditions in")
print("  meaningful quantities.")
print("• All laps are included, including lap 1. Lap 1 gaps are real")
print("  even though the gate drop adds noise.")
print("• Curves with fewer than 2 data points are silently skipped. If")
print("  a rider rarely sees one condition, that curve won't appear.")


Notes
─────────────────
• Sandwiched: Someone within 2.0 seconds ahead AND someone within
  2.0 seconds behind. Simultaneous attack and defense pressure —
  the highest-cognitive-load condition.
• Chasing: Someone within 2.0 seconds ahead, AND no one within
  2.0 seconds behind. The rider is trying to make a pass with no
  defensive pressure.
• Being Chased: No one within 2.0 seconds ahead (or they're in
  1st place), AND someone within 2.0 seconds behind. Defensive
  pressure with clear track ahead.
• Clear Air: No one within 2.0 seconds in either direction. The
  rider is in a gap, racing the clock.
• These four conditions are mutually exclusive and exhaustive —
  every lap falls into exactly one category.
• Only riders who led a lap (≥3 laps in 1st place) in the selected
  filter window are shown in the dropdown. This restricts the
  analysis to front-runners who experience all four conditions in
  meaningful quantities.
• All laps are included, including lap 1. Lap 1 gaps are real

In [30]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import plotly.graph_objects as go
import pandas as pd
import numpy as np

# ── Traffic base setup (self-contained) ──────────────────────────────────────
traffic_base = df.dropna(subset=["lap", "lap_time", "place"]).copy()
traffic_base["lap"]   = traffic_base["lap"].astype(float)
traffic_base["place"] = traffic_base["place"].astype(float)
traffic_base["year"]  = traffic_base["year"].astype(str)
traffic_base["round"] = traffic_base["round"].astype(str)
traffic_base["moto"]  = traffic_base["moto"].astype(str)
traffic_base["cond_sandwiched"]   = traffic_base["traffic_state"] == "Sandwiched"
traffic_base["cond_chasing"]      = traffic_base["traffic_state"] == "Chasing"
traffic_base["cond_being_chased"] = traffic_base["traffic_state"] == "Being Chased"
traffic_base["cond_clear_air"]    = traffic_base["traffic_state"] == "Clear Air"

# ── Filter options ─────────────────────────────────────────────────────────────
journey_years = sorted(traffic_base["year"].unique())
journey_classes = ["450", "250", "WMX"]
journey_rounds = sorted(traffic_base["round"].unique(), key=lambda x: int(x))
journey_motos = sorted(traffic_base["moto"].unique(), key=lambda x: int(x))

CONDITION_META = {
    "cond_sandwiched": ("Sandwiched", "#9B27AE"),
    "cond_chasing": ("Chasing", "#E8641A"),
    "cond_being_chased": ("Being Chased", "#E8C11A"),
    "cond_clear_air": ("Clear Air", "#1A7FE8"),
}


def get_condition(row):
    if row["cond_sandwiched"]:
        return "cond_sandwiched"
    if row["cond_chasing"]:
        return "cond_chasing"
    if row["cond_being_chased"]:
        return "cond_being_chased"
    if row["cond_clear_air"]:
        return "cond_clear_air"
    return None


# ── Widgets ────────────────────────────────────────────────────────────────────
year_sel_jn = widgets.Dropdown(
    options=journey_years, value=journey_years[-1],
    description="Year:", layout=widgets.Layout(width="200px")
)
class_sel_jn = widgets.Dropdown(
    options=journey_classes, value="450",
    description="Class:", layout=widgets.Layout(width="200px")
)
round_sel_jn = widgets.Dropdown(
    options=journey_rounds, value=journey_rounds[0],
    description="Round:", layout=widgets.Layout(width="200px")
)
moto_sel_jn = widgets.Dropdown(
    options=journey_motos, value=journey_motos[0],
    description="Moto:", layout=widgets.Layout(width="200px")
)
rider_sel_jn = widgets.Dropdown(
    options=[], value=None,
    description="Rider:", layout=widgets.Layout(width="280px")
)


def get_journey_riders(year_val, class_val, round_val, moto_val):
    mask = (
            (traffic_base["year"] == year_val) &
            (traffic_base["class_label"] == class_val) &
            (traffic_base["round"] == round_val) &
            (traffic_base["moto"] == moto_val)
    )
    return sorted(traffic_base[mask]["name"].dropna().unique().tolist())


def refresh_journey_riders(*args):
    riders = get_journey_riders(
        year_sel_jn.value, class_sel_jn.value,
        round_sel_jn.value, moto_sel_jn.value
    )
    current = rider_sel_jn.value
    rider_sel_jn.options = riders
    rider_sel_jn.value = current if current in riders else (riders[0] if riders else None)


for w in [year_sel_jn, class_sel_jn, round_sel_jn, moto_sel_jn]:
    w.observe(refresh_journey_riders, names="value")

refresh_journey_riders()


# ── Plot function ──────────────────────────────────────────────────────────────
def plot_journey(year_val, class_val, round_val, moto_val, rider):
    if not rider:
        print("No riders found for selected filters.")
        return

    mask = (
            (traffic_base["year"] == year_val) &
            (traffic_base["class_label"] == class_val) &
            (traffic_base["round"] == round_val) &
            (traffic_base["moto"] == moto_val) &
            (traffic_base["name"] == rider)
    )
    subset = (
        traffic_base[mask]
        .sort_values("lap")
        .copy()
    )

    if subset.empty:
        print(f"No data for {rider} with selected filters.")
        return

    subset["condition"] = subset.apply(get_condition, axis=1)

    laps = subset["lap"].tolist()
    places = subset["place"].tolist()
    conds = subset["condition"].tolist()
    gap_ahead = subset["gap_ahead"].tolist()
    gap_behind = subset["gap_behind"].tolist()
    max_place = int(subset["place"].max())

    fig = go.Figure()
    legend_shown = set()

    for i in range(len(laps) - 1):
        cond = conds[i]
        label, color = CONDITION_META.get(cond, ("Unknown", "#aaaaaa"))
        show_legend = label not in legend_shown

        ga = gap_ahead[i]
        gb = gap_behind[i]
        ahead_str = f"{ga:.2f}s" if pd.notna(ga) else "—"
        behind_str = f"{gb:.2f}s" if pd.notna(gb) else "—"

        fig.add_trace(go.Scatter(
            x=[laps[i], laps[i + 1]],
            y=[places[i], places[i + 1]],
            mode="lines",
            line=dict(color=color, width=3),
            name=label,
            legendgroup=label,
            showlegend=show_legend,
            customdata=[[ahead_str, behind_str], [ahead_str, behind_str]],
            hovertemplate=(
                f"<b>{rider}</b><br>"
                "Lap: %{x}<br>"
                "Place: %{y}<br>"
                f"Condition: {label}<br>"
                "Gap ahead: %{customdata[0]}<br>"
                "Gap behind: %{customdata[1]}"
                "<extra></extra>"
            ),
        ))
        legend_shown.add(label)

    fig.update_layout(
        title=f"Place Journey — {rider} | {class_val} | {year_val} | Round {round_val} | Moto {moto_val}",
        xaxis=dict(
            title="Lap",
            tickmode="linear",
            dtick=1,
        ),
        yaxis=dict(
            title="Place",
            autorange="reversed",
            tickmode="linear",
            dtick=1,
            range=[max_place + 0.5, 0.5],
        ),
        height=500,
        width=950,
        margin=dict(l=60, r=60, t=60, b=60),
        legend=dict(title="Condition"),
        hovermode="closest",
    )
    display(fig)


btn_jn = widgets.Button(description="Update", button_style="primary")
output_jn = widgets.Output()

def update_jn(_):
    output_jn.clear_output(wait=True)
    with output_jn:
        plot_journey(
            year_sel_jn.value, class_sel_jn.value,
            round_sel_jn.value, moto_sel_jn.value,
            rider_sel_jn.value,
        )

btn_jn.on_click(update_jn)

display(widgets.VBox([
    widgets.HBox([year_sel_jn, class_sel_jn, round_sel_jn, moto_sel_jn]),
    rider_sel_jn,
    btn_jn,
    output_jn,
]))

update_jn(None)

print("\nNotes")
print("─────────────────")
print("• Sandwiched: Someone within 2.0 seconds ahead AND someone within")
print("  2.0 seconds behind. Simultaneous attack and defense pressure —")
print("  the highest-cognitive-load condition.")
print("• Chasing: Someone within 2.0 seconds ahead, AND no one within")
print("  2.0 seconds behind. The rider is trying to make a pass with no")
print("  defensive pressure.")
print("• Being Chased: No one within 2.0 seconds ahead (or they're in")
print("  1st place), AND someone within 2.0 seconds behind. Defensive")
print("  pressure with clear track ahead.")
print("• Clear Air: No one within 2.0 seconds in either direction. The")
print("  rider is in a gap, racing the clock.")
print("• These four conditions are mutually exclusive and exhaustive —")
print("  every lap falls into exactly one category.")
print("• Gap ahead / gap behind in the hover show the raw time gap in")
print("  seconds to the rider immediately ahead and behind at that lap.")
print("  '—' indicates the rider is leading (no one ahead) or last (no")
print("  one behind).")


Notes
─────────────────
• Sandwiched: Someone within 2.0 seconds ahead AND someone within
  2.0 seconds behind. Simultaneous attack and defense pressure —
  the highest-cognitive-load condition.
• Chasing: Someone within 2.0 seconds ahead, AND no one within
  2.0 seconds behind. The rider is trying to make a pass with no
  defensive pressure.
• Being Chased: No one within 2.0 seconds ahead (or they're in
  1st place), AND someone within 2.0 seconds behind. Defensive
  pressure with clear track ahead.
• Clear Air: No one within 2.0 seconds in either direction. The
  rider is in a gap, racing the clock.
• These four conditions are mutually exclusive and exhaustive —
  every lap falls into exactly one category.
• Gap ahead / gap behind in the hover show the raw time gap in
  seconds to the rider immediately ahead and behind at that lap.
  '—' indicates the rider is leading (no one ahead) or last (no
  one behind).


<a id="gender-gap"></a>
## 13. Gender Gap
- Comparing the 250 and WMX class as they use the same bike
- Jamie Astudillo case study

In [31]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import plotly.graph_objects as go
import pandas as pd
import numpy as np
from scipy.stats import gaussian_kde

# ── Build gender gap base ─────────────────────────────────────────────────────
gg_base = (
    df.dropna(subset=["lap", "lap_time"])
    .copy()
)
gg_base["lap"]   = gg_base["lap"].astype(float)
gg_base["year"]  = gg_base["year"].astype(str)
gg_base["round"] = gg_base["round"].astype(str)
gg_base["moto"]  = gg_base["moto"].astype(str)

# Keep only 250 and WMX, exclude lap 1
gg_base = gg_base[
    gg_base["class_label"].isin(["250", "WMX"]) &
    (gg_base["lap"] > 1)
].copy()

# ── Moto remapping for round 2 three-moto weekends ───────────────────────────
gg_base = gg_base[
    ~(
        (gg_base["round"] == "2") &
        (gg_base["class_label"] == "WMX") &
        (gg_base["moto"] == "1")
    )
].copy()

gg_base.loc[
    (gg_base["round"] == "2") &
    (gg_base["class_label"] == "WMX") &
    (gg_base["moto"] == "2"),
    "moto"
] = "1"

gg_base.loc[
    (gg_base["round"] == "2") &
    (gg_base["class_label"] == "WMX") &
    (gg_base["moto"] == "3"),
    "moto"
] = "2"

# ── Filter to top 10 finishers per moto per class ────────────────────────────
top10_finishers = (
    gg_base.dropna(subset=["finish_position"])
    .groupby(["race_id", "name"], observed=True)["finish_position"]
    .first()
    .reset_index()
)
top10_finishers = top10_finishers[top10_finishers["finish_position"] <= 10][["race_id", "name"]]

gg_base = gg_base.merge(top10_finishers, on=["race_id", "name"], how="inner")

# ── Filter options ─────────────────────────────────────────────────────────────
gg_years  = sorted(gg_base["year"].unique())
gg_rounds = ["All"] + sorted(gg_base["round"].unique(), key=lambda x: int(x))
gg_motos  = ["All"] + sorted(gg_base["moto"].unique(),  key=lambda x: int(x))

# ── Helper ─────────────────────────────────────────────────────────────────────
def hex_to_rgba(hex_color, opacity=0.08):
    hex_color = hex_color.lstrip("#")
    r, g, b   = int(hex_color[0:2], 16), int(hex_color[2:4], 16), int(hex_color[4:6], 16)
    return f"rgba({r},{g},{b},{opacity})"

CLASS_COLORS = {
    "250": "#1A7FE8",
    "WMX": "#E8641A",
}

# ── Widgets ────────────────────────────────────────────────────────────────────
year_sel_gg = widgets.Dropdown(
    options=gg_years, value=gg_years[-1],
    description="Year:", layout=widgets.Layout(width="200px")
)
round_sel_gg = widgets.Dropdown(
    options=gg_rounds, value="All",
    description="Round:", layout=widgets.Layout(width="200px")
)
moto_sel_gg = widgets.Dropdown(
    options=gg_motos, value="All",
    description="Moto:", layout=widgets.Layout(width="200px")
)

# ── Plot function ──────────────────────────────────────────────────────────────
def plot_gender_gap(year_val, round_val, moto_val):
    mask = (gg_base["year"] == year_val)
    if round_val != "All":
        mask &= gg_base["round"] == round_val
    if moto_val != "All":
        mask &= gg_base["moto"] == moto_val

    subset   = gg_base[mask]
    fig      = go.Figure()
    all_vals = []

    for class_label, color in CLASS_COLORS.items():
        vals = subset[subset["class_label"] == class_label]["lap_time"].dropna().values

        if len(vals) < 2:
            continue

        all_vals.extend(vals)

        kde     = gaussian_kde(vals, bw_method=0.3)
        x_min   = np.percentile(vals, 1)
        x_max   = np.percentile(vals, 99)
        x_range = np.linspace(x_min * 0.995, x_max * 1.005, 500)
        density        = kde(x_range)
        median         = np.median(vals)
        median_density = kde(np.array([median]))[0]

        median_min = int(median // 60)
        median_sec = median % 60
        median_str = f"{median_min}:{median_sec:06.3f}"

        fig.add_trace(go.Scatter(
            x=x_range,
            y=density,
            mode="lines",
            name=f"{class_label} (n={len(vals)}, median={median_str})",
            line=dict(color=color, width=2),
            fill="tozeroy",
            fillcolor=hex_to_rgba(color, 0.08),
            hovertemplate=(
                f"<b>{class_label}</b><br>"
                "Lap Time: %{x:.3f}s<br>"
                "Density: %{y:.4f}"
                "<extra></extra>"
            ),
        ))

        fig.add_trace(go.Scatter(
            x=[median, median],
            y=[0, median_density],
            mode="lines",
            line=dict(color=color, width=1.5, dash="dot"),
            showlegend=False,
            hoverinfo="skip",
        ))

    if not all_vals:
        print("No data for selected filters.")
        return

    global_min = np.percentile(all_vals, 1)
    global_max = np.percentile(all_vals, 99)

    tick_vals = np.linspace(global_min, global_max, 8)
    tick_text = [
        f"{int(t // 60)}:{t % 60:05.2f}"
        for t in tick_vals
    ]

    round_str = f"Round {round_val}" if round_val != "All" else "All Rounds"
    moto_str  = f"Moto {moto_val}"   if moto_val  != "All" else "All Motos"

    fig.update_layout(
        title=f"Gender Gap — 250 vs WMX — Top 10 | {year_val} | {round_str} | {moto_str}",
        xaxis=dict(
            title="Lap Time",
            range=[global_min * 0.995, global_max * 1.005],
            tickvals=tick_vals,
            ticktext=tick_text,
        ),
        yaxis=dict(title="Density", showticklabels=False),
        height=500,
        width=950,
        margin=dict(l=60, r=60, t=60, b=60),
        legend=dict(title="Class"),
        hovermode="x unified",
    )
    display(fig)

btn_gg = widgets.Button(description="Update", button_style="primary")
output_gg = widgets.Output()

def update_gg(_):
    output_gg.clear_output(wait=True)
    with output_gg:
        plot_gender_gap(year_sel_gg.value, round_sel_gg.value, moto_sel_gg.value)

btn_gg.on_click(update_gg)

display(widgets.VBox([
    widgets.HBox([year_sel_gg, round_sel_gg, moto_sel_gg]),
    btn_gg,
    output_gg,
]))

update_gg(None)

print("\nNotes")
print("─────────────────")
print("• Direct lap-time comparison between the 250 and WMX classes.")
print("  Both classes ride 250-displacement bikes on the same tracks in")
print("  the same race weekends, so this is the cleanest available gender")
print("  comparison the data permits.")
print("• Restricted to top 10 finishers in each moto. This removes back-")
print("  markers from both sides so the comparison reflects competitive")
print("  pace, not field-depth differences.")
print("• WMX runs three motos at Round 2 weekends instead of the usual")
print("  two. The first moto happens on the men's day off, then motos 2")
print("  and 3 run on the main race day alongside the 250 class.")
print("  To keep the comparison consistent, Round 2 WMX moto 1 is dropped")
print("  entirely, and motos 2/3 are remapped to '1' and '2' so they")
print("  align with the same-day moto structure of the 250 class.")
print("• Lap 1 is excluded from both classes.")
print("• KDE bandwidth is 0.3, same as the other distribution charts.")
print("• X-axis is capped at the 1st and 99th percentiles across both")
print("  classes for visual clarity, with mm:ss formatting.")
print("• The horizontal gap between the two median lines is the typical")
print("  pace difference between top-10 250 riders and top-10 WMX riders.")
print("  Read this as 'on a typical lap, the median 250 top-10 rider is X")
print("  seconds faster than the median WMX top-10 rider'.")


Notes
─────────────────
• Direct lap-time comparison between the 250 and WMX classes.
  Both classes ride 250-displacement bikes on the same tracks in
  the same race weekends, so this is the cleanest available gender
  comparison the data permits.
• Restricted to top 10 finishers in each moto. This removes back-
  markers from both sides so the comparison reflects competitive
  pace, not field-depth differences.
• WMX runs three motos at Round 2 weekends instead of the usual
  two. The first moto happens on the men's day off, then motos 2
  and 3 run on the main race day alongside the 250 class.
  To keep the comparison consistent, Round 2 WMX moto 1 is dropped
  entirely, and motos 2/3 are remapped to '1' and '2' so they
  align with the same-day moto structure of the 250 class.
• Lap 1 is excluded from both classes.
• KDE bandwidth is 0.3, same as the other distribution charts.
• X-axis is capped at the 1st and 99th percentiles across both
  classes for visual clarity, with mm:ss f

In [32]:
import pandas as pd
import numpy as np
from IPython.display import display, HTML

# ── Jamie's data ──────────────────────────────────────────────────────────────
JAMIE = "Jamie Astudillo"
JAMIE_YEAR = "2025"

jamie_base = (
    df[df["year"].astype(str) == JAMIE_YEAR]
    .copy()
)
jamie_base["lap"] = jamie_base["lap"].astype(float)
jamie_base["round"] = jamie_base["round"].astype(str)
jamie_base["moto"] = jamie_base["moto"].astype(str)
jamie_base["year"] = jamie_base["year"].astype(str)


# ── Helper formatters ─────────────────────────────────────────────────────────
def fmt_time(seconds):
    if pd.isna(seconds):
        return "—"
    m = int(seconds // 60)
    s = seconds % 60
    return f"{m}:{s:06.3f}"


def fmt_place(val):
    if pd.isna(val):
        return "—"
    return str(int(val))


# ── Within-class z-score base ─────────────────────────────────────────────────
for cls in ["WMX", "250"]:
    cls_data = jamie_base[jamie_base["class_label"] == cls].copy()
    lap_stats = (
        cls_data.groupby(["race_id", "lap"], observed=True)["lap_time"]
        .agg(mean_lt="mean", std_lt="std")
        .reset_index()
    )
    jamie_base = jamie_base.merge(
        lap_stats.rename(columns={"mean_lt": f"mean_lt_{cls}", "std_lt": f"std_lt_{cls}"}),
        on=["race_id", "lap"],
        how="left"
    )

jamie_base["z_wmx"] = np.where(
    jamie_base["class_label"] == "WMX",
    (jamie_base["lap_time"] - jamie_base["mean_lt_WMX"]) / jamie_base["std_lt_WMX"],
    np.nan
)
jamie_base["z_250"] = np.where(
    jamie_base["class_label"] == "250",
    (jamie_base["lap_time"] - jamie_base["mean_lt_250"]) / jamie_base["std_lt_250"],
    np.nan
)

# ── Moto remapping for round 2 ────────────────────────────────────────────────
jamie_base["mapped_moto"] = jamie_base["moto"]
jamie_base.loc[
    (jamie_base["round"] == "2") &
    (jamie_base["class_label"] == "WMX") &
    (jamie_base["moto"] == "2"),
    "mapped_moto"
] = "1"
jamie_base.loc[
    (jamie_base["round"] == "2") &
    (jamie_base["class_label"] == "WMX") &
    (jamie_base["moto"] == "3"),
    "mapped_moto"
] = "2"

# ── Get Jamie's WMX motos ─────────────────────────────────────────────────────
jamie_wmx = jamie_base[
    (jamie_base["name"] == JAMIE) &
    (jamie_base["class_label"] == "WMX")
    ].copy()

wmx_motos = (
    jamie_wmx.drop_duplicates(subset=["race_id"])
    [["race_id", "round", "moto", "mapped_moto", "track", "finish_position"]]
    .sort_values(["round", "moto"])
)

# ── Build rows ────────────────────────────────────────────────────────────────
rows = []

for _, moto_row in wmx_motos.iterrows():
    race_id = moto_row["race_id"]
    round_num = moto_row["round"]
    moto_num = moto_row["moto"]
    mapped_moto = moto_row["mapped_moto"]
    track = moto_row["track"]

    # ── WMX moto stats ────────────────────────────────────────────────────────
    jamie_wmx_laps = jamie_base[
        (jamie_base["name"] == JAMIE) &
        (jamie_base["class_label"] == "WMX") &
        (jamie_base["race_id"] == race_id)
        ].dropna(subset=["lap_time"])

    wmx_place = moto_row["finish_position"]
    wmx_avg_time = jamie_wmx_laps["lap_time"].mean() if not jamie_wmx_laps.empty else np.nan
    wmx_avg_z = jamie_wmx_laps["z_wmx"].dropna().mean() if not jamie_wmx_laps.empty else np.nan
    jamie_max_lap = jamie_wmx_laps["lap"].max() if not jamie_wmx_laps.empty else np.nan

    # ── 250 equivalent moto ───────────────────────────────────────────────────
    is_r2_moto1 = (round_num == "2") and (moto_num == "1")

    if not is_r2_moto1:
        r250_races = jamie_base[
            (jamie_base["class_label"] == "250") &
            (jamie_base["round"] == round_num) &
            (jamie_base["moto"] == mapped_moto)
            ]
        r250_race_id = r250_races["race_id"].iloc[0] if not r250_races.empty else None

        jamie_250_laps = jamie_base[
            (jamie_base["name"] == JAMIE) &
            (jamie_base["class_label"] == "250") &
            (jamie_base["round"] == round_num) &
            (jamie_base["moto"] == mapped_moto)
            ].dropna(subset=["lap_time"])

        has_250 = not jamie_250_laps.empty

        if has_250:
            p250_place = jamie_250_laps["finish_position"].dropna().iloc[0] if jamie_250_laps[
                "finish_position"].notna().any() else np.nan
            p250_avg_time = jamie_250_laps["lap_time"].mean()
            p250_avg_z = jamie_250_laps["z_250"].dropna().mean()
        else:
            p250_place = np.nan
            p250_avg_time = np.nan
            p250_avg_z = np.nan

        if r250_race_id is not None and not pd.isna(jamie_max_lap):
            r250_all = jamie_base[
                (jamie_base["class_label"] == "250") &
                (jamie_base["race_id"] == r250_race_id) &
                (jamie_base["lap"] <= jamie_max_lap)
                ].dropna(subset=["lap_time"])

            r250_total_laps = jamie_base[
                (jamie_base["class_label"] == "250") &
                (jamie_base["race_id"] == r250_race_id)
                ]["lap"].max()

            r250_cumul = (
                r250_all.groupby("name", observed=True)["lap_time"]
                .sum()
                .reset_index(name="cumul_time")
            )

            jamie_cumul = jamie_wmx_laps[
                jamie_wmx_laps["lap"] <= jamie_max_lap
                ]["lap_time"].sum()

            riders_ahead = (r250_cumul["cumul_time"] < jamie_cumul).sum()
            hyp_place = int(riders_ahead) + 1
            hyp_note = f"Lap {int(jamie_max_lap)} of {int(r250_total_laps)}"
        else:
            hyp_place = np.nan
            hyp_note = "—"
    else:
        r250_race_id = None
        has_250 = False
        p250_place = np.nan
        p250_avg_time = np.nan
        p250_avg_z = np.nan
        hyp_place = np.nan
        hyp_note = "—"

    rows.append({
        "Round": round_num,
        "Moto": moto_num,
        "Track": track,
        "WMX Place": fmt_place(wmx_place),
        "WMX Avg Lap": fmt_time(wmx_avg_time),
        "WMX Avg Z": f"{wmx_avg_z:.3f}" if not pd.isna(wmx_avg_z) else "—",
        "250 Place": fmt_place(p250_place),
        "250 Avg Lap": fmt_time(p250_avg_time),
        "250 Avg Z": f"{p250_avg_z:.3f}" if not pd.isna(p250_avg_z) else "—",
        "Hyp. 250 Place": fmt_place(hyp_place),
        "Hyp. Laps": hyp_note,
    })

# ── Display ───────────────────────────────────────────────────────────────────
result = pd.DataFrame(rows)

print(f"\n  Jamie Astudillo — 2025 Season Analysis\n")
print(f"  WMX Avg Z:      Z-score vs WMX field (all laps in moto)")
print(f"  250 Avg Z:      Z-score vs 250 field (all laps in moto, only motos she raced)")
print(f"  Hyp. 250 Place: Projected 250 finish using WMX cumulative lap times\n")

html = """
<style>
  .jamie-table {
    border-collapse: collapse;
    font-family: Arial, sans-serif;
    font-size: 13px;
    table-layout: auto;
  }
  .jamie-table th, .jamie-table td {
    padding: 6px 14px;
    text-align: center;
    border: 1px solid #ccc;
    white-space: nowrap;
  }
  .jamie-table td.left {
    text-align: left;
    white-space: nowrap;
  }
  .jamie-table th.group-wmx { background: #E8F0FB; color: #14304D; border-bottom: 3px solid #1A7FE8; }
  .jamie-table th.group-250 { background: #E8FBF0; color: #14402B; border-bottom: 3px solid #1AE87F; }
  .jamie-table th.group-hyp { background: #FCEEE3; color: #5C3115; border-bottom: 3px solid #E8641A; }
  .jamie-table th.group-idx { background: #F2F2F2; color: #222;   border-bottom: 3px solid #999; }
  .jamie-table th.sub       { background: #FAFAFA; color: #444; font-weight: normal; font-size: 12px; }
  .jamie-table tr:nth-child(even) td { background: #FFFFFF; }
  .jamie-table tr:nth-child(odd)  td { background: #F7F7F7; }
  .jamie-table td { color: #1a1a1a; }
</style>
<table class="jamie-table">
  <thead>
    <tr>
      <th class="group-idx" colspan="3"></th>
      <th class="group-wmx" colspan="3">WMX</th>
      <th class="group-250" colspan="3">250</th>
      <th class="group-hyp" colspan="2">Hypothetical</th>
    </tr>
    <tr>
      <th class="sub">Round</th>
      <th class="sub">Moto</th>
      <th class="sub">Track</th>
      <th class="sub">Place</th>
      <th class="sub">Avg Lap</th>
      <th class="sub">Avg Z</th>
      <th class="sub">Place</th>
      <th class="sub">Avg Lap</th>
      <th class="sub">Avg Z</th>
      <th class="sub">250 Place</th>
      <th class="sub">Laps</th>
    </tr>
  </thead>
  <tbody>
"""

for _, r in result.iterrows():
    html += f"""
    <tr>
      <td>{r['Round']}</td>
      <td>{r['Moto']}</td>
      <td class="left">{r['Track']}</td>
      <td>{r['WMX Place']}</td>
      <td>{r['WMX Avg Lap']}</td>
      <td>{r['WMX Avg Z']}</td>
      <td>{r['250 Place']}</td>
      <td>{r['250 Avg Lap']}</td>
      <td>{r['250 Avg Z']}</td>
      <td>{r['Hyp. 250 Place']}</td>
      <td>{r['Hyp. Laps']}</td>
    </tr>"""

html += "\n  </tbody>\n</table>"
display(HTML(html))

print("\nNotes")
print("──────────────────")
print("• Jamie Astudillo raced both WMX and 250 in 2025, making her the")
print("  only rider in the dataset with direct cross-class data. This")
print("  table compares her actual results in each class moto-by-moto,")
print("  plus a hypothetical 250 finish based on her WMX pace.")
print("• The hypothetical column is the most interesting: it asks 'if")
print("  Jamie's WMX cumulative lap times had been posted in the 250")
print("  race that same day, where would they have placed her?'")
print("• The 'Hyp. Laps' column shows how many laps the comparison was")
print("  based on. WMX motos are shorter than 250 motos, so a 'Lap 8 of")
print("  12' note means the projection only uses the portion of the 250")
print("  race Jamie's WMX laps cover.")
print("• Uses cumulative time rather than average lap time — this")
print("  captures her actual pace including any traffic, mistakes, or")
print("  fade across her laps.")
print("• Z-scores within each class are independent: a z-score of -1 in")
print("  WMX (faster than WMX field average) and a z-score of 0 in 250")
print("  (average for 250 field) tell different stories about the same")
print("  rider on the same day. The difference in z-scores between class")
print("  tells you how she stacks up within each class relative to its own")
print("  competitive distribution.")


  Jamie Astudillo — 2025 Season Analysis

  WMX Avg Z:      Z-score vs WMX field (all laps in moto)
  250 Avg Z:      Z-score vs 250 field (all laps in moto, only motos she raced)
  Hyp. 250 Place: Projected 250 finish using WMX cumulative lap times




Notes
──────────────────
• Jamie Astudillo raced both WMX and 250 in 2025, making her the
  only rider in the dataset with direct cross-class data. This
  table compares her actual results in each class moto-by-moto,
  plus a hypothetical 250 finish based on her WMX pace.
• The hypothetical column is the most interesting: it asks 'if
  Jamie's WMX cumulative lap times had been posted in the 250
  race that same day, where would they have placed her?'
• The 'Hyp. Laps' column shows how many laps the comparison was
  based on. WMX motos are shorter than 250 motos, so a 'Lap 8 of
  12' note means the projection only uses the portion of the 250
  race Jamie's WMX laps cover.
• Uses cumulative time rather than average lap time — this
  captures her actual pace including any traffic, mistakes, or
  fade across her laps.
• Z-scores within each class are independent: a z-score of -1 in
  WMX (faster than WMX field average) and a z-score of 0 in 250
  (average for 250 field) tell different sto

<a id="prize-money"></a>
## 14. Prize Money
- Does prize money alone make the Triple Crown Motocross series financially viable for riders?

![2026 MXTOUR Payout](2026%20MXTOUR%20Payout.png)

In [34]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
import numpy as np

# ── Payout tables ─────────────────────────────────────────────────────────────
PAYOUT_450 = {
    1: 1200, 2: 1000, 3: 950, 4: 850, 5: 700,
    6: 600,  7: 500,  8: 400, 9: 300, 10: 250,
    11: 200, 12: 200, 13: 150, 14: 150, 15: 150,
}
PAYOUT_250 = {
    1: 900, 2: 700, 3: 600, 4: 500, 5: 400,
    6: 300, 7: 250, 8: 200, 9: 200, 10: 175,
    11: 150, 12: 150, 13: 150, 14: 125, 15: 100,
}
PAYOUT_WMX = {
    1: 900, 2: 700, 3: 600, 4: 500, 5: 400,
    6: 300, 7: 250, 8: 200, 9: 175, 10: 150,
}
PAYOUT_MAP = {"450": PAYOUT_450, "250": PAYOUT_250, "WMX": PAYOUT_WMX}

# ── TCMX points table ─────────────────────────────────────────────────────────
TCMX_POINTS = {
    1: 25, 2: 22, 3: 20, 4: 18, 5: 16,
    6: 15, 7: 14, 8: 13, 9: 12, 10: 11,
    11: 10, 12: 9, 13: 8, 14: 7, 15: 6,
    16: 5, 17: 4, 18: 3, 19: 2, 20: 1,
}

# ── Build prize money base ────────────────────────────────────────────────────
prize_base = (
    df.drop_duplicates(subset=["race_id", "name"])
    .dropna(subset=["finish_position"])
    .copy()
)
prize_base["year"]            = prize_base["year"].astype(str)
prize_base["round"]           = prize_base["round"].astype(int)
prize_base["moto"]            = prize_base["moto"].astype(int)
prize_base["finish_position"] = prize_base["finish_position"].astype(int)

# Map finish position to TCMX points
prize_base["moto_points"] = prize_base["finish_position"].map(TCMX_POINTS).fillna(0)

# ── Combined moto points per rider per round ──────────────────────────────────
round_points = (
    prize_base.groupby(["name", "class_label", "year", "round"], observed=True)
    .agg(
        combined_points=("moto_points", "sum"),
        last_moto_pos=("finish_position", lambda x: x.loc[x.index[
            prize_base.loc[x.index, "moto"] == prize_base.loc[x.index, "moto"].max()
        ]].iloc[0] if len(x) > 0 else 999)
    )
    .reset_index()
)

# ── Overall place per round with tiebreak on last moto result ─────────────────
round_points["overall_place"] = (
    round_points.groupby(["class_label", "year", "round"], observed=True)
    .apply(lambda g: g.sort_values(
        ["combined_points", "last_moto_pos"],
        ascending=[False, True]
    ).assign(overall_place=range(1, len(g) + 1))["overall_place"],
    include_groups=False)
    .reset_index(level=[0, 1, 2], drop=True)
)

# ── Map payout ────────────────────────────────────────────────────────────────
def get_payout(row):
    payout_table = PAYOUT_MAP.get(row["class_label"], {})
    return payout_table.get(int(row["overall_place"]), 0)

round_points["prize_money"] = round_points.apply(get_payout, axis=1)

# ── Season totals ─────────────────────────────────────────────────────────────
season_prize = (
    round_points.groupby(["name", "class_label", "year"], observed=True)
    .agg(
        rounds_entered=("round", "count"),
        total_prize=("prize_money", "sum"),
    )
    .reset_index()
)
season_prize["year"] = season_prize["year"].astype(str)

# ── Filter options ─────────────────────────────────────────────────────────────
prize_years   = sorted(season_prize["year"].unique())
prize_classes = ["450", "250", "WMX"]

# ── Widgets ────────────────────────────────────────────────────────────────────
year_sel_pr = widgets.Dropdown(
    options=prize_years, value=prize_years[-1],
    description="Year:", layout=widgets.Layout(width="200px")
)
class_sel_pr = widgets.Dropdown(
    options=prize_classes, value="450",
    description="Class:", layout=widgets.Layout(width="200px")
)

btn_pr    = widgets.Button(description="Update", button_style="primary")
output_pr = widgets.Output()

# ── Update callback ────────────────────────────────────────────────────────────
def update_pr(_):
    output_pr.clear_output(wait=True)
    with output_pr:

        subset = season_prize[
            (season_prize["year"]        == year_sel_pr.value) &
            (season_prize["class_label"] == class_sel_pr.value) &
            (season_prize["total_prize"]  > 0)
        ].sort_values("total_prize", ascending=False).reset_index(drop=True)

        if subset.empty:
            print("No data for selected filters.")
            return

        subset.index += 1
        subset = subset.rename(columns={
            "name":           "Rider",
            "rounds_entered": "Rounds",
            "total_prize":    "Total Prize ($)",
        })
        subset["Total Prize ($)"] = subset["Total Prize ($)"].apply(
            lambda x: f"${x:,.0f}"
        )

        total = round_points[
            (round_points["year"]        == year_sel_pr.value) &
            (round_points["class_label"] == class_sel_pr.value)
        ]["prize_money"].sum()

        print(f"\n  Prize Money — {class_sel_pr.value} | {year_sel_pr.value}\n")
        print(f"  Overall place per round = Combined moto points")
        print(f"  Tiebreak: Better last moto result")
        display(subset[["Rider", "Rounds", "Total Prize ($)"]])
        print(f"\n  Total class payout: ${total:,.0f}")

btn_pr.on_click(update_pr)

# ── Layout ─────────────────────────────────────────────────────────────────────
display(widgets.VBox([
    widgets.HBox([year_sel_pr, class_sel_pr, btn_pr]),
    output_pr,
]))

print("\nNotes")
print("──────────────────")
print("• Simulates how much prize money each rider would have earned per")
print("  season using the 2026 MXTOUR payout structure.")
print("• Payout is determined by overall round place, not moto-by-moto")
print("  place — riders are paid based on their combined two-moto")
print("  finishing position at each round, the way TCMX actually awards")
print("  the cheques.")
print("• Tiebreak: Better last-moto finish wins. So if two riders both")
print("  total 40 points across a round, the one with the better moto 2")
print("  result gets the higher overall position. This matches how TCMX")
print("  resolves ties in the official standings.")
print("• Manufacturer team contracts, performance bonuses, and sponsorships")
print("  are not included.")
print("• When considering entry fees and all of the associated costs like bike")
print("  maintenance, transportation, and accomodation, prize money alone is not ")
print("  enough for even the best of the best to break even on.")
print("• Series prize money seems designed more as a points incentive to field")
print("  competitive entries.")

update_pr(None)


Notes
──────────────────
• Simulates how much prize money each rider would have earned per
  season using the 2026 MXTOUR payout structure.
• Payout is determined by overall round place, not moto-by-moto
  place — riders are paid based on their combined two-moto
  finishing position at each round, the way CMRC actually awards
  the cheques.
• Tiebreak: Better last-moto finish wins. So if two riders both
  total 40 points across a round, the one with the better moto 2
  result gets the higher overall position. This matches how CMRC
  resolves ties in the official standings.
• Manufacturer team contracts, performance bonuses, and sponsorships
  are not included.
• When considering entry fees and all of the associated costs like bike
  maintenance, transportation, and accomodation, prize money alone is not 
  enough for even the best of the best to break even on.
• Series prize money seems designed more as a points incentive to field
  competitive entries.


<a id="weather-effects"></a>
## 15. Weather Effects
- Influence of weather on year-over-year lap times?
- Are top riders just as strong in wet conditions?

In [35]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import plotly.graph_objects as go
import pandas as pd
import numpy as np
from scipy.stats import gaussian_kde

# ── Load weather data ─────────────────────────────────────────────────────────
WEATHER_PATH = "weather_data.csv"
weather_df = pd.read_csv(WEATHER_PATH, parse_dates=["date"])
weather_df["date"] = weather_df["date"].dt.strftime("%Y-%m-%d")
weather_df["year"] = weather_df["date"].str[:4]

# ── Join weather onto lap data ────────────────────────────────────────────────
wx_base = df.copy()
wx_base["date"] = pd.to_datetime(wx_base["date"]).dt.strftime("%Y-%m-%d")
wx_base["lap"] = wx_base["lap"].astype(float)
wx_base["year"] = wx_base["year"].astype(str)

wx_base = wx_base.merge(
    weather_df[["track", "date", "max_temp_c", "precipitation_mm",
                "climate_station_id", "lat", "lon"]],
    on=["date", "track"],
    how="left"
)
wx_base = wx_base[wx_base["lap"].astype(float) > 1].copy()

# ── Top 5 finishers by combined moto points per track per year per class ─────
TCMX_POINTS = {
    1: 25, 2: 22, 3: 20, 4: 18, 5: 16, 6: 15, 7: 14, 8: 13, 9: 12, 10: 11,
    11: 10, 12: 9, 13: 8, 14: 7, 15: 6, 16: 5, 17: 4, 18: 3, 19: 2, 20: 1,
}

moto_pts = (
    wx_base.drop_duplicates(subset=["race_id", "name"])
    .dropna(subset=["finish_position"])
    .copy()
)
moto_pts["finish_position"] = moto_pts["finish_position"].astype(int)
moto_pts["moto_points"] = moto_pts["finish_position"].map(TCMX_POINTS).fillna(0)

round_pts = (
    moto_pts.groupby(["name", "class_label", "year", "track"], observed=True)
    ["moto_points"].sum()
    .reset_index(name="total_points")
)

top5_by_track = (
    round_pts.sort_values(["class_label", "year", "track", "total_points"], ascending=[True, True, True, False])
    .groupby(["class_label", "year", "track"], observed=True)
    .head(5)
    [["name", "class_label", "year", "track"]]
)

wx_base = wx_base.merge(
    top5_by_track,
    on=["name", "class_label", "year", "track"],
    how="inner"
)

# ── % off best per lap per moto ───────────────────────────────────────────────
lap_best_wx = (
    wx_base.groupby(["race_id", "lap"], observed=True)["lap_time"]
    .min()
    .rename("best_lap_time")
    .reset_index()
)
wx_base = wx_base.merge(lap_best_wx, on=["race_id", "lap"], how="left")
wx_base["pct_off_best"] = (
        (wx_base["lap_time"] - wx_base["best_lap_time"]) /
        wx_base["best_lap_time"] * 100
)
wx_base = wx_base[wx_base["pct_off_best"] >= 0].copy()

# ── Within-class z-score per lap per moto ────────────────────────────────────
lap_stats_wx = (
    wx_base.groupby(["race_id", "lap"], observed=True)["lap_time"]
    .agg(mean_lt="mean", std_lt="std")
    .reset_index()
)
wx_base = wx_base.merge(lap_stats_wx, on=["race_id", "lap"], how="left")
wx_base = wx_base[
    wx_base["std_lt"].notna() & (wx_base["std_lt"] > 0)
    ].copy()
wx_base["z_score"] = (
        (wx_base["lap_time"] - wx_base["mean_lt"]) / wx_base["std_lt"]
)

# ── Wet/dry flag ──────────────────────────────────────────────────────────────
RAIN_THRESHOLD = 1.0
wx_base["is_wet"] = wx_base["precipitation_mm"] >= RAIN_THRESHOLD

# ── Repeated venues ───────────────────────────────────────────────────────────
venue_years = (
    wx_base.drop_duplicates(subset=["track", "year"])
    .groupby("track", observed=True)["year"]
    .nunique()
)
repeated_venues = venue_years[venue_years > 1].index.tolist()


# ── Helpers ───────────────────────────────────────────────────────────────────
def hex_to_rgba(hex_color, opacity=0.08):
    hex_color = hex_color.lstrip("#")
    r, g, b = int(hex_color[0:2], 16), int(hex_color[2:4], 16), int(hex_color[4:6], 16)
    return f"rgba({r},{g},{b},{opacity})"


def fmt_time(seconds):
    if pd.isna(seconds):
        return "—"
    m = int(seconds // 60)
    s = seconds % 60
    return f"{m}:{s:05.2f}"


YEAR_COLORS = {"2024": "#1A7FE8", "2025": "#E8641A"}

# ── Widgets ───────────────────────────────────────────────────────────────────
class_sel_yoy = widgets.Dropdown(
    options=["450", "250", "WMX"], value="450",
    description="Class:", layout=widgets.Layout(width="200px")
)
track_sel_yoy = widgets.Dropdown(
    options=sorted(repeated_venues),
    value=sorted(repeated_venues)[0] if repeated_venues else None,
    description="Track:", layout=widgets.Layout(width="250px")
)

btn_yoy = widgets.Button(description="Update", button_style="primary")
output_yoy = widgets.Output()


def plot_yoy(class_val, track_val):
    if not track_val:
        print("No repeat venues found.")
        return

    subset = wx_base[
        (wx_base["class_label"] == class_val) &
        (wx_base["track"] == track_val)
        ].dropna(subset=["lap_time"])

    if subset.empty:
        print("No data for selected filters.")
        return

    fig = go.Figure()

    for year, color in YEAR_COLORS.items():
        vals = subset[subset["year"] == year]["lap_time"].dropna().values

        if len(vals) < 5:
            continue

        kde = gaussian_kde(vals, bw_method=0.3)
        x_range = np.linspace(
            np.percentile(vals, 1),
            np.percentile(vals, 99),
            400
        )
        density = kde(x_range)
        median = np.median(vals)
        temp_val = subset[subset["year"] == year]["max_temp_c"].iloc[0]
        precip_val = subset[subset["year"] == year]["precipitation_mm"].iloc[0]

        fig.add_trace(go.Scatter(
            x=x_range, y=density,
            mode="lines",
            name=f"{track_val} {year} (max {temp_val:.1f}°C, {precip_val:.1f}mm rain)",
            line=dict(
                color=color, width=2,
                dash="solid" if year == "2024" else "dash"
            ),
            fill="tozeroy",
            fillcolor=hex_to_rgba(color, 0.05),
            hovertemplate=(
                f"<b>{track_val} {year}</b><br>"
                "Lap: %{x:.2f}s<br>"
                f"Max temp: {temp_val:.1f}°C<br>"
                f"Precip: {precip_val:.1f}mm"
                "<extra></extra>"
            ),
        ))
        fig.add_trace(go.Scatter(
            x=[median, median],
            y=[0, kde(np.array([median]))[0]],
            mode="lines",
            line=dict(color=color, width=1, dash="dot"),
            showlegend=False, hoverinfo="skip",
        ))

    all_vals = subset["lap_time"].dropna().values
    p1, p99 = np.percentile(all_vals, 1), np.percentile(all_vals, 99)
    tick_vals = np.linspace(p1, p99, 7)
    tick_text = [fmt_time(t) for t in tick_vals]

    fig.update_layout(
        title=f"Year-over-Year Lap Times — {class_val} | {track_val} | Solid=2024, Dashed=2025",
        xaxis=dict(
            title="Lap Time",
            range=[p1 * 0.998, p99 * 1.002],
            tickvals=tick_vals, ticktext=tick_text,
        ),
        yaxis=dict(title="Density", showticklabels=False),
        height=550, width=1000,
        margin=dict(l=60, r=60, t=60, b=60),
        legend=dict(title="Year"),
        hovermode="x unified",
    )
    display(fig)


def update_yoy(_):
    output_yoy.clear_output(wait=True)
    with output_yoy:
        print("  Solid=2024, Dashed=2025. Legend shows max temp and precipitation on race day.")
        print("  Restricted to top 5 finishers by points at each track in each year.")
        print("  Lap 1 is excluded since the shorter lap times would skew the KDE curve.")
        plot_yoy(class_sel_yoy.value, track_sel_yoy.value)


btn_yoy.on_click(update_yoy)
display(widgets.VBox([
    widgets.HBox([class_sel_yoy, track_sel_yoy, btn_yoy]),
    output_yoy,
]))
update_yoy(None)

In [36]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import plotly.graph_objects as go
import numpy as np
from scipy.stats import gaussian_kde

# ── Weather setup ────────────
WEATHER_PATH = "weather_data.csv"
weather_df = pd.read_csv(WEATHER_PATH, parse_dates=["date"])
weather_df["date"] = weather_df["date"].dt.strftime("%Y-%m-%d")
weather_df["year"] = weather_df["date"].str[:4]

TCMX_POINTS = {
    1: 25, 2: 22, 3: 20, 4: 18, 5: 16, 6: 15, 7: 14, 8: 13, 9: 12, 10: 11,
    11: 10, 12: 9, 13: 8, 14: 7, 15: 6, 16: 5, 17: 4, 18: 3, 19: 2, 20: 1,
}
RAIN_THRESHOLD = 1.0

wx_base = df.copy()
wx_base["date"] = pd.to_datetime(wx_base["date"]).dt.strftime("%Y-%m-%d")
wx_base["lap"]  = wx_base["lap"].astype(float)
wx_base["year"] = wx_base["year"].astype(str)

wx_base = wx_base.merge(
    weather_df[["track", "date", "max_temp_c", "precipitation_mm",
                "climate_station_id", "lat", "lon"]],
    on=["date", "track"], how="left"
)
wx_base = wx_base[wx_base["lap"] > 1].copy()

moto_pts = (
    wx_base.drop_duplicates(subset=["race_id", "name"])
    .dropna(subset=["finish_position"]).copy()
)
moto_pts["finish_position"] = moto_pts["finish_position"].astype(int)
moto_pts["moto_points"] = moto_pts["finish_position"].map(TCMX_POINTS).fillna(0)

round_pts = (
    moto_pts.groupby(["name", "class_label", "year", "track"], observed=True)
    ["moto_points"].sum().reset_index(name="total_points")
)
top5_by_track = (
    round_pts
    .sort_values(["class_label", "year", "track", "total_points"], ascending=[True, True, True, False])
    .groupby(["class_label", "year", "track"], observed=True)
    .head(5)[["name", "class_label", "year", "track"]]
)
wx_base = wx_base.merge(top5_by_track, on=["name", "class_label", "year", "track"], how="inner")

lap_best_wx = (
    wx_base.groupby(["race_id", "lap"], observed=True)["lap_time"]
    .min().rename("best_lap_time").reset_index()
)
wx_base = wx_base.merge(lap_best_wx, on=["race_id", "lap"], how="left")
wx_base["pct_off_best"] = (
    (wx_base["lap_time"] - wx_base["best_lap_time"]) / wx_base["best_lap_time"] * 100
)
wx_base = wx_base[wx_base["pct_off_best"] >= 0].copy()

lap_stats_wx = (
    wx_base.groupby(["race_id", "lap"], observed=True)["lap_time"]
    .agg(mean_lt="mean", std_lt="std").reset_index()
)
wx_base = wx_base.merge(lap_stats_wx, on=["race_id", "lap"], how="left")
wx_base = wx_base[wx_base["std_lt"].notna() & (wx_base["std_lt"] > 0)].copy()
wx_base["z_score"] = (wx_base["lap_time"] - wx_base["mean_lt"]) / wx_base["std_lt"]
wx_base["is_wet"]  = wx_base["precipitation_mm"] >= RAIN_THRESHOLD

venue_years = (
    wx_base.drop_duplicates(subset=["track", "year"])
    .groupby("track", observed=True)["year"].nunique()
)
repeated_venues = venue_years[venue_years > 1].index.tolist()


# ── Requires wx_base from Chunk 1 ─────────────────────────────────────────────

def hex_to_rgba(hex_color, opacity=0.08):
    hex_color = hex_color.lstrip("#")
    r, g, b   = int(hex_color[0:2], 16), int(hex_color[2:4], 16), int(hex_color[4:6], 16)
    return f"rgba({r},{g},{b},{opacity})"

# ── Widgets ───────────────────────────────────────────────────────────────────
class_sel_wd = widgets.Dropdown(
    options=["450", "250", "WMX"], value="450",
    description="Class:", layout=widgets.Layout(width="200px")
)
year_sel_wd = widgets.Dropdown(
    options=["2024", "2025"], value="2025",
    description="Year:", layout=widgets.Layout(width="200px")
)

btn_wd    = widgets.Button(description="Update", button_style="primary")
output_wd = widgets.Output()

def plot_wet_dry(class_val, year_val):
    subset = wx_base[
        (wx_base["class_label"] == class_val) &
        (wx_base["year"]        == year_val)
    ].dropna(subset=["pct_off_best", "is_wet"])

    dry_vals = subset[~subset["is_wet"]]["pct_off_best"].values
    wet_vals = subset[ subset["is_wet"]]["pct_off_best"].values

    wet_rounds = (
        wx_base[
            (wx_base["is_wet"]) &
            (wx_base["year"] == year_val)
        ][["track", "precipitation_mm"]]
        .drop_duplicates()
        .sort_values("precipitation_mm", ascending=False)
    )
    wet_summary = ", ".join(
        f"{r['track']} ({r['precipitation_mm']}mm)"
        for _, r in wet_rounds.iterrows()
    )

    fig = go.Figure()

    for label, vals, color in [
        (f"Dry (<{RAIN_THRESHOLD}mm)",  dry_vals, "#1A7FE8"),
        (f"Wet (≥{RAIN_THRESHOLD}mm)", wet_vals, "#E8641A"),
    ]:
        if len(vals) < 5:
            print(f"Not enough data for {label}.")
            continue

        p99            = np.percentile(vals, 99)
        kde            = gaussian_kde(vals, bw_method=0.3)
        x_range        = np.linspace(0, p99, 400)
        density        = kde(x_range)
        median         = np.median(vals)
        median_density = kde(np.array([median]))[0]

        fig.add_trace(go.Scatter(
            x=x_range, y=density,
            mode="lines",
            name=f"{label} (n={len(vals)}, median={median:.1f}%)",
            line=dict(color=color, width=2),
            fill="tozeroy",
            fillcolor=hex_to_rgba(color, 0.08),
            hovertemplate=(
                f"<b>{label}</b><br>"
                "% off best: %{x:.2f}%<br>"
                "Density: %{y:.4f}"
                "<extra></extra>"
            ),
        ))
        fig.add_trace(go.Scatter(
            x=[median, median], y=[0, median_density],
            mode="lines",
            line=dict(color=color, width=1.5, dash="dot"),
            showlegend=False, hoverinfo="skip",
        ))

    all_vals = subset["pct_off_best"].values
    if len(all_vals) == 0:
        return

    fig.update_layout(
        title=f"Wet vs Dry % Off Best — {class_val} | {year_val}",
        xaxis=dict(
            title="% Off Best (lap-by-lap)",
            range=[0, np.percentile(all_vals, 99) * 1.1],
        ),
        yaxis=dict(title="Density", showticklabels=False),
        height=500, width=1000,
        margin=dict(l=60, r=60, t=60, b=60),
        legend=dict(title="Condition"),
        hovermode="x unified",
    )
    display(fig)
    print(f"  Wet rounds: {wet_summary}")

def update_wd(_):
    output_wd.clear_output(wait=True)
    with output_wd:
        plot_wet_dry(class_sel_wd.value, year_sel_wd.value)

btn_wd.on_click(update_wd)
display(widgets.VBox([
    widgets.HBox([class_sel_wd, year_sel_wd, btn_wd]),
    output_wd,
]))

update_wd(None)

In [37]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import plotly.graph_objects as go
import pandas as pd
import numpy as np

# ── Weather setup (self-contained — reproduces Section 15 Cell 1) ────────────
WEATHER_PATH = "weather_data.csv"
weather_df = pd.read_csv(WEATHER_PATH, parse_dates=["date"])
weather_df["date"] = weather_df["date"].dt.strftime("%Y-%m-%d")
weather_df["year"] = weather_df["date"].str[:4]

TCMX_POINTS = {
    1: 25, 2: 22, 3: 20, 4: 18, 5: 16, 6: 15, 7: 14, 8: 13, 9: 12, 10: 11,
    11: 10, 12: 9, 13: 8, 14: 7, 15: 6, 16: 5, 17: 4, 18: 3, 19: 2, 20: 1,
}
RAIN_THRESHOLD = 1.0

wx_base = df.copy()
wx_base["date"] = pd.to_datetime(wx_base["date"]).dt.strftime("%Y-%m-%d")
wx_base["lap"]  = wx_base["lap"].astype(float)
wx_base["year"] = wx_base["year"].astype(str)

wx_base = wx_base.merge(
    weather_df[["track", "date", "max_temp_c", "precipitation_mm",
                "climate_station_id", "lat", "lon"]],
    on=["date", "track"], how="left"
)
wx_base = wx_base[wx_base["lap"] > 1].copy()

moto_pts = (
    wx_base.drop_duplicates(subset=["race_id", "name"])
    .dropna(subset=["finish_position"]).copy()
)
moto_pts["finish_position"] = moto_pts["finish_position"].astype(int)
moto_pts["moto_points"] = moto_pts["finish_position"].map(TCMX_POINTS).fillna(0)

round_pts = (
    moto_pts.groupby(["name", "class_label", "year", "track"], observed=True)
    ["moto_points"].sum().reset_index(name="total_points")
)
top5_by_track = (
    round_pts
    .sort_values(["class_label", "year", "track", "total_points"], ascending=[True, True, True, False])
    .groupby(["class_label", "year", "track"], observed=True)
    .head(5)[["name", "class_label", "year", "track"]]
)
wx_base = wx_base.merge(top5_by_track, on=["name", "class_label", "year", "track"], how="inner")

lap_best_wx = (
    wx_base.groupby(["race_id", "lap"], observed=True)["lap_time"]
    .min().rename("best_lap_time").reset_index()
)
wx_base = wx_base.merge(lap_best_wx, on=["race_id", "lap"], how="left")
wx_base["pct_off_best"] = (
    (wx_base["lap_time"] - wx_base["best_lap_time"]) / wx_base["best_lap_time"] * 100
)
wx_base = wx_base[wx_base["pct_off_best"] >= 0].copy()

lap_stats_wx = (
    wx_base.groupby(["race_id", "lap"], observed=True)["lap_time"]
    .agg(mean_lt="mean", std_lt="std").reset_index()
)
wx_base = wx_base.merge(lap_stats_wx, on=["race_id", "lap"], how="left")
wx_base = wx_base[wx_base["std_lt"].notna() & (wx_base["std_lt"] > 0)].copy()
wx_base["z_score"] = (wx_base["lap_time"] - wx_base["mean_lt"]) / wx_base["std_lt"]
wx_base["is_wet"]  = wx_base["precipitation_mm"] >= RAIN_THRESHOLD

venue_years = (
    wx_base.drop_duplicates(subset=["track", "year"])
    .groupby("track", observed=True)["year"].nunique()
)
repeated_venues = venue_years[venue_years > 1].index.tolist()

# ── round_points (self-contained — reproduces Section 14 Prize Money) ─────────
_prize_base = (
    df.drop_duplicates(subset=["race_id", "name"])
    .dropna(subset=["finish_position"]).copy()
)
_prize_base["year"]            = _prize_base["year"].astype(str)
_prize_base["round"]           = _prize_base["round"].astype(int)
_prize_base["moto"]            = _prize_base["moto"].astype(int)
_prize_base["finish_position"] = _prize_base["finish_position"].astype(int)
_prize_base["moto_points"]     = _prize_base["finish_position"].map(TCMX_POINTS).fillna(0)

round_points = (
    _prize_base.groupby(["name", "class_label", "year", "round"], observed=True)
    .agg(
        combined_points=("moto_points", "sum"),
        last_moto_pos=("finish_position", lambda x: x.loc[x.index[
            _prize_base.loc[x.index, "moto"] == _prize_base.loc[x.index, "moto"].max()
        ]].iloc[0] if len(x) > 0 else 999)
    )
    .reset_index()
)


# ── Build a class-year-wide z-score (unfiltered by top 5) ─────────────────────
wz_base = df.copy()
wz_base["date"] = pd.to_datetime(wz_base["date"]).dt.strftime("%Y-%m-%d")
wz_base["lap"] = wz_base["lap"].astype(float)
wz_base["year"] = wz_base["year"].astype(str)

wz_base = wz_base.merge(
    weather_df[["track", "date", "precipitation_mm"]],
    on=["date", "track"],
    how="left"
)
wz_base = wz_base[wz_base["lap"] > 1].copy()
wz_base["is_wet"] = wz_base["precipitation_mm"] >= RAIN_THRESHOLD

# Z-score per lap per moto using ALL riders in the class
lap_stats_wz = (
    wz_base.groupby(["race_id", "lap"], observed=True)["lap_time"]
    .agg(mean_lt="mean", std_lt="std")
    .reset_index()
)
wz_base = wz_base.merge(lap_stats_wz, on=["race_id", "lap"], how="left")
wz_base = wz_base[
    wz_base["std_lt"].notna() & (wz_base["std_lt"] > 0)
    ].copy()
wz_base["z_score"] = (
        (wz_base["lap_time"] - wz_base["mean_lt"]) / wz_base["std_lt"]
)

# ── Season points ─────────────────────────────────────────────────────────────
season_points_wx = (
    round_points.groupby(["name", "class_label", "year"], observed=True)
    ["combined_points"]
    .sum()
    .reset_index(name="season_points")
)
season_points_wx["year"] = season_points_wx["year"].astype(str)

# ── Widgets ───────────────────────────────────────────────────────────────────
class_sel_wz = widgets.Dropdown(
    options=["450", "250", "WMX"], value="450",
    description="Class:", layout=widgets.Layout(width="200px")
)
year_sel_wz = widgets.Dropdown(
    options=["2024", "2025"], value="2025",
    description="Year:", layout=widgets.Layout(width="200px")
)

btn_wz = widgets.Button(description="Update", button_style="primary")
output_wz = widgets.Output()


def plot_top10_wet_dry(class_val, year_val):
    top10 = (
        season_points_wx[
            (season_points_wx["class_label"] == class_val) &
            (season_points_wx["year"] == year_val)
            ]
        .sort_values("season_points", ascending=False)
        .head(10)["name"]
        .tolist()
    )

    if not top10:
        print("No points data for selected filters.")
        return

    subset = wz_base[
        (wz_base["class_label"] == class_val) &
        (wz_base["year"] == year_val) &
        (wz_base["name"].isin(top10))
        ].dropna(subset=["z_score", "is_wet"])

    rows = []
    for rider in top10:
        r = subset[subset["name"] == rider]
        dry_z = r[~r["is_wet"]]["z_score"].dropna()
        wet_z = r[r["is_wet"]]["z_score"].dropna()
        rows.append({
            "rider": rider,
            "dry_median_z": dry_z.median() if len(dry_z) >= 2 else np.nan,
            "wet_median_z": wet_z.median() if len(wet_z) >= 2 else np.nan,
            "dry_n": len(dry_z),
            "wet_n": len(wet_z),
        })

    result = pd.DataFrame(rows).dropna(subset=["dry_median_z"])
    result = result.sort_values("dry_median_z")

    wet_rounds = (
        wz_base[
            (wz_base["is_wet"]) &
            (wz_base["year"] == year_val)
            ][["track", "precipitation_mm"]]
        .drop_duplicates()
        .sort_values("precipitation_mm", ascending=False)
    )
    wet_summary = ", ".join(
        f"{r['track']} ({r['precipitation_mm']}mm)"
        for _, r in wet_rounds.iterrows()
    )

    fig = go.Figure()

    fig.add_trace(go.Bar(
        y=result["rider"],
        x=result["dry_median_z"],
        name="Dry",
        orientation="h",
        marker_color="#1A7FE8",
        customdata=result["dry_n"],
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Dry median z: %{x:.3f}<br>"
            "n laps: %{customdata}"
            "<extra></extra>"
        ),
    ))
    fig.add_trace(go.Bar(
        y=result["rider"],
        x=result["wet_median_z"],
        name="Wet",
        orientation="h",
        marker_color="#E8641A",
        customdata=result["wet_n"],
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Wet median z: %{x:.3f}<br>"
            "n laps: %{customdata}"
            "<extra></extra>"
        ),
    ))

    fig.add_vline(x=0, line_width=1, line_dash="dash", line_color="grey")

    fig.update_layout(
        title=f"Wet vs Dry Median Z-Score — {class_val} | {year_val}",
        xaxis=dict(title="Median Z-Score (negative = faster than field avg)"),
        yaxis=dict(title=""),
        barmode="group",
        height=520, width=1000,
        margin=dict(l=180, r=60, t=60, b=60),
        legend=dict(title="Condition"),
        hovermode="closest",
    )
    display(fig)
    print(f"  Wet rounds: {wet_summary}")
    print(f"  Wet threshold: ≥{RAIN_THRESHOLD}mm. Top 10 by season points. Sorted by dry median z-score.")

    no_wet = [r["rider"] for _, r in pd.DataFrame(rows).iterrows() if r["wet_n"] < 2]
    if no_wet:
        print(f"  Note: Insufficient wet laps for {', '.join(no_wet)} — wet bar not shown.")


def update_wz(_):
    output_wz.clear_output(wait=True)
    with output_wz:
        plot_top10_wet_dry(class_sel_wz.value, year_sel_wz.value)


btn_wz.on_click(update_wz)
display(widgets.VBox([
    widgets.HBox([class_sel_wz, year_sel_wz, btn_wz]),
    output_wz,
]))
update_wz(None)

print("\nNotes")
print("──────────────────")
print("• For each of the top 10 riders by season points in the selected")
print("  class and year, compares their median lap-time z-score on wet")
print("  race days vs dry race days.")
print("• Z-scores are computed against the FULL field on each lap (not")
print("  just the top 10 riders shown), so a value of 0 means the rider")
print("  was at the field average for that lap. Negative = faster than")
print("  field average; positive = slower.")
print("• Top 10 riders by season points (not race-day attendance). A")
print("  rider must have scored well enough across the year to rank in")
print("  the top 10 to appear.")
print("• Riders with fewer than 2 wet laps are excluded from the wet")
print("  bar (insufficient sample); their dry bar still shows.")


Notes
──────────────────
• For each of the top 10 riders by season points in the selected
  class and year, compares their median lap-time z-score on wet
  race days vs dry race days.
• Z-scores are computed against the FULL field on each lap (not
  just the top 10 riders shown), so a value of 0 means the rider
  was at the field average for that lap. Negative = faster than
  field average; positive = slower.
• Top 10 riders by season points (not race-day attendance). A
  rider must have scored well enough across the year to rank in
  the top 10 to appear.
• Riders with fewer than 2 wet laps are excluded from the wet
  bar (insufficient sample); their dry bar still shows.


<a id="fall-detection"></a>
## 16. Fall Detection
- Recoverable anomaly: Lap time > N SD above rider's own median and place drop >= 2
- Unrecoverable anomaly: DNF or |finish_position - place_on_last_lap| >= 2

In [38]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
import numpy as np

# ── Build anomaly base ────────────────────────────────────────────────────────
anom_base = (
    df.dropna(subset=["lap", "lap_time", "place", "finish_position"])
    .copy()
)
anom_base["lap"] = anom_base["lap"].astype(float)
anom_base["place"] = anom_base["place"].astype(float)
anom_base["finish_position"] = anom_base["finish_position"].astype(float)
anom_base["year"] = anom_base["year"].astype(str)
anom_base["round"] = anom_base["round"].astype(str)
anom_base["moto"] = anom_base["moto"].astype(str)

# ── Within-rider median and std per moto ──────────────────────────────────────
rider_stats = (
    anom_base.groupby(["race_id", "name"], observed=True)["lap_time"]
    .agg(rider_median="median", rider_std="std", rider_laps="count")
    .reset_index()
)
anom_base = anom_base.merge(rider_stats, on=["race_id", "name"], how="left")

# Drop riders with < 3 laps (can't compute meaningful stats)
anom_base = anom_base[anom_base["rider_laps"] >= 3].copy()

# Within-rider z-score
anom_base = anom_base[
    anom_base["rider_std"].notna() & (anom_base["rider_std"] > 0)
    ].copy()
anom_base["rider_z"] = (
        (anom_base["lap_time"] - anom_base["rider_median"]) /
        anom_base["rider_std"]
)

# ── Place on previous lap ─────────────────────────────────────────────────────
anom_base = anom_base.sort_values(["race_id", "name", "lap"])
anom_base["prev_place"] = anom_base.groupby(
    ["race_id", "name"], observed=True
)["place"].shift(1)

anom_base["place_drop"] = anom_base["place"] - anom_base["prev_place"]

# ── Last lap place per rider per moto ─────────────────────────────────────────
last_lap = (
    anom_base.groupby(["race_id", "name"], observed=True)
    .agg(
        last_lap_num=("lap", "max"),
        total_laps_in_moto=("lap", lambda x: anom_base.loc[
            anom_base["race_id"] == x.name[0], "lap"
        ].max()),
    )
    .reset_index()
)

# Simpler: get place on last recorded lap per rider
last_lap_place = (
    anom_base.sort_values(["race_id", "name", "lap"])
    .groupby(["race_id", "name"], observed=True)
    .agg(
        last_lap_num=("lap", "max"),
        last_lap_place=("place", "last"),
    )
    .reset_index()
)

# Max lap in each moto (field-wide)
moto_max_lap = (
    anom_base.groupby("race_id", observed=True)["lap"]
    .max()
    .rename("moto_max_lap")
    .reset_index()
)

anom_base = anom_base.merge(last_lap_place, on=["race_id", "name"], how="left")
anom_base = anom_base.merge(moto_max_lap, on="race_id", how="left")

# ── Finish position per rider per moto (one value) ───────────────────────────
finish_pos = (
    anom_base.drop_duplicates(subset=["race_id", "name"])
    [["race_id", "name", "finish_position"]]
)


# ── Detect anomalies ──────────────────────────────────────────────────────────
def detect_anomalies(threshold):
    # ── Recoverable: spike + place drop ≥ 2 on same lap + finishes cleanly ───
    recoverable = anom_base[
        (anom_base["rider_z"] >= threshold) &
        (anom_base["place_drop"] >= 2) &
        (anom_base["lap"] > 1) &
        # Finisher: last lap == moto max lap
        (anom_base["last_lap_num"] == anom_base["moto_max_lap"]) &
        # Clean finish: finish_position == last_lap_place
        (abs(anom_base["finish_position"] - anom_base["last_lap_place"]) < 2)
        ].copy()
    recoverable["anomaly_type"] = "Recoverable"

    # ── Unrecoverable: |finish - last lap place| >= 2 ─────────────────────────
    unrecoverable = (
        anom_base.drop_duplicates(subset=["race_id", "name"])
        .copy()
    )
    unrecoverable = unrecoverable[
        abs(unrecoverable["finish_position"] - unrecoverable["last_lap_place"]) >= 2
        ].copy()
    unrecoverable["anomaly_type"] = "Unrecoverable"
    unrecoverable["lap"] = unrecoverable["last_lap_num"]
    unrecoverable["rider_z"] = (
            (unrecoverable["lap_time"] - unrecoverable["rider_median"]) /
            unrecoverable["rider_std"]
    )
    unrecoverable["place_drop"] = (
            unrecoverable["finish_position"] - unrecoverable["last_lap_place"]
    )

    return recoverable, unrecoverable


# ── Format helpers ────────────────────────────────────────────────────────────
def fmt_time(seconds):
    if pd.isna(seconds):
        return "—"
    m = int(seconds // 60)
    s = seconds % 60
    return f"{m}:{s:06.3f}"


def fmt_lap_time_col(series):
    return series.apply(fmt_time)


# ── Filter options ─────────────────────────────────────────────────────────────
anom_years = sorted(anom_base["year"].unique())
anom_classes = ["450", "250", "WMX"]
anom_rounds = ["All"] + sorted(anom_base["round"].unique(), key=lambda x: int(x))
anom_motos = ["All"] + sorted(anom_base["moto"].unique(), key=lambda x: int(x))

# ── Widgets ───────────────────────────────────────────────────────────────────
year_sel_an = widgets.Dropdown(
    options=anom_years, value=anom_years[-1],
    description="Year:", layout=widgets.Layout(width="200px")
)
class_sel_an = widgets.Dropdown(
    options=anom_classes, value="450",
    description="Class:", layout=widgets.Layout(width="200px")
)
round_sel_an = widgets.Dropdown(
    options=anom_rounds, value="All",
    description="Round:", layout=widgets.Layout(width="200px")
)
moto_sel_an = widgets.Dropdown(
    options=anom_motos, value="All",
    description="Moto:", layout=widgets.Layout(width="200px")
)
threshold_sel = widgets.FloatSlider(
    value=2.5, min=1.5, max=4.0, step=0.25,
    description="SD Threshold:",
    layout=widgets.Layout(width="350px"),
    style={"description_width": "120px"},
)
type_sel_an = widgets.Dropdown(
    options=["Both", "Recoverable", "Unrecoverable"],
    value="Both",
    description="Type:", layout=widgets.Layout(width="220px")
)

btn_an = widgets.Button(description="Update", button_style="primary")
output_an = widgets.Output()


# ── Update callback ───────────────────────────────────────────────────────────
def update_an(_):
    output_an.clear_output(wait=True)
    with output_an:

        recoverable, unrecoverable = detect_anomalies(threshold_sel.value)

        # Apply filters to recoverable
        rec = recoverable.copy()
        unrec = unrecoverable.copy()

        for subset, label in [(rec, "rec"), (unrec, "unrec")]:
            pass

        def apply_filters(subset):
            mask = subset["year"] == year_sel_an.value
            mask &= subset["class_label"] == class_sel_an.value
            if round_sel_an.value != "All":
                mask &= subset["round"] == round_sel_an.value
            if moto_sel_an.value != "All":
                mask &= subset["moto"] == moto_sel_an.value
            return subset[mask]

        rec = apply_filters(recoverable)
        unrec = apply_filters(unrecoverable)

        show_type = type_sel_an.value

        if show_type in ["Both", "Recoverable"] and not rec.empty:
            print(f"\n  ── Recoverable Anomalies (n={len(rec)}) ──────────────────────────────\n")
            rec_display = rec[[
                "name", "round", "moto", "track", "lap",
                "lap_time", "rider_median", "rider_z",
                "prev_place", "place", "place_drop",
            ]].copy()
            rec_display["lap_time"] = fmt_lap_time_col(rec_display["lap_time"])
            rec_display["rider_median"] = fmt_lap_time_col(rec_display["rider_median"])
            rec_display["rider_z"] = rec_display["rider_z"].round(2)
            rec_display["place_drop"] = rec_display["place_drop"].astype(int)
            rec_display["lap"] = rec_display["lap"].astype(int)
            rec_display["prev_place"] = rec_display["prev_place"].astype(int)
            rec_display["place"] = rec_display["place"].astype(int)
            rec_display = rec_display.rename(columns={
                "name": "Rider",
                "round": "Rd",
                "moto": "Moto",
                "track": "Track",
                "lap": "Lap",
                "lap_time": "Anomaly Lap Time",
                "rider_median": "Median Lap Time",
                "rider_z": "Rider Z Score",
                "prev_place": "Place Before",
                "place": "Place After",
                "place_drop": "Places Lost",
            }).reset_index(drop=True)
            rec_display.index += 1
            display(rec_display)
        elif show_type in ["Both", "Recoverable"]:
            print("  No recoverable anomalies for selected filters.")

        if show_type in ["Both", "Unrecoverable"] and not unrec.empty:
            print(f"\n  ── Unrecoverable Anomalies / DNFs (n={len(unrec)}) ───────────────────\n")
            unrec_display = unrec[[
                "name", "round", "moto", "track",
                "last_lap_num", "moto_max_lap",
                "last_lap_place", "finish_position",
                "place_drop",
            ]].copy()
            unrec_display["place_drop"] = unrec_display["place_drop"].astype(int)
            unrec_display["last_lap_num"] = unrec_display["last_lap_num"].astype(int)
            unrec_display["moto_max_lap"] = unrec_display["moto_max_lap"].astype(int)
            unrec_display["last_lap_place"] = unrec_display["last_lap_place"].astype(int)
            unrec_display["finish_position"] = unrec_display["finish_position"].astype(int)
            unrec_display = unrec_display.rename(columns={
                "name": "Rider",
                "round": "Rd",
                "moto": "Moto",
                "track": "Track",
                "last_lap_num": "Last Lap",
                "moto_max_lap": "Moto Laps",
                "last_lap_place": "Place at Last Lap",
                "finish_position": "Finish Position",
                "place_drop": "Places Lost",
            }).reset_index(drop=True)
            unrec_display.index += 1
            display(unrec_display)
        elif show_type in ["Both", "Unrecoverable"]:
            print("  No unrecoverable anomalies for selected filters.")


btn_an.on_click(update_an)

display(widgets.VBox([
    widgets.HBox([year_sel_an, class_sel_an, round_sel_an, moto_sel_an]),
    widgets.HBox([threshold_sel, type_sel_an]),
    btn_an,
    output_an,
]))

update_an(None)

In [40]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import plotly.graph_objects as go
import pandas as pd
import numpy as np
from scipy.stats import gaussian_kde

# ── Anomaly setup (self-contained — reproduces Section 16 Cell 1) ────────────
anom_base = (
    df.dropna(subset=["lap", "lap_time", "place", "finish_position"])
    .copy()
)
anom_base["lap"]             = anom_base["lap"].astype(float)
anom_base["place"]           = anom_base["place"].astype(float)
anom_base["finish_position"] = anom_base["finish_position"].astype(float)
anom_base["year"]            = anom_base["year"].astype(str)
anom_base["round"]           = anom_base["round"].astype(str)
anom_base["moto"]            = anom_base["moto"].astype(str)

rider_stats = (
    anom_base.groupby(["race_id", "name"], observed=True)["lap_time"]
    .agg(rider_median="median", rider_std="std", rider_laps="count")
    .reset_index()
)
anom_base = anom_base.merge(rider_stats, on=["race_id", "name"], how="left")
anom_base = anom_base[anom_base["rider_laps"] >= 3].copy()
anom_base = anom_base[
    anom_base["rider_std"].notna() & (anom_base["rider_std"] > 0)
].copy()
anom_base["rider_z"] = (
    (anom_base["lap_time"] - anom_base["rider_median"]) / anom_base["rider_std"]
)

anom_base = anom_base.sort_values(["race_id", "name", "lap"])
anom_base["prev_place"] = anom_base.groupby(
    ["race_id", "name"], observed=True
)["place"].shift(1)
anom_base["place_drop"] = anom_base["place"] - anom_base["prev_place"]

last_lap_place = (
    anom_base.sort_values(["race_id", "name", "lap"])
    .groupby(["race_id", "name"], observed=True)
    .agg(last_lap_num=("lap", "max"), last_lap_place=("place", "last"))
    .reset_index()
)
moto_max_lap = (
    anom_base.groupby("race_id", observed=True)["lap"]
    .max().rename("moto_max_lap").reset_index()
)
anom_base = anom_base.merge(last_lap_place, on=["race_id", "name"], how="left")
anom_base = anom_base.merge(moto_max_lap, on="race_id", how="left")

def detect_anomalies(threshold):
    recoverable = anom_base[
        (anom_base["rider_z"] >= threshold) &
        (anom_base["place_drop"] >= 2) &
        (anom_base["lap"] > 1) &
        (anom_base["last_lap_num"] == anom_base["moto_max_lap"]) &
        (abs(anom_base["finish_position"] - anom_base["last_lap_place"]) < 2)
    ].copy()
    recoverable["anomaly_type"] = "Recoverable"

    unrecoverable = anom_base.drop_duplicates(subset=["race_id", "name"]).copy()
    unrecoverable = unrecoverable[
        abs(unrecoverable["finish_position"] - unrecoverable["last_lap_place"]) >= 2
    ].copy()
    unrecoverable["anomaly_type"] = "Unrecoverable"
    unrecoverable["lap"]        = unrecoverable["last_lap_num"]
    unrecoverable["rider_z"]    = (
        (unrecoverable["lap_time"] - unrecoverable["rider_median"]) /
        unrecoverable["rider_std"]
    )
    unrecoverable["place_drop"] = (
        unrecoverable["finish_position"] - unrecoverable["last_lap_place"]
    )
    return recoverable, unrecoverable

def fmt_time(seconds):
    if pd.isna(seconds): return "—"
    m = int(seconds // 60); s = seconds % 60
    return f"{m}:{s:06.3f}"

def fmt_lap_time_col(series):
    return series.apply(fmt_time)

anom_years   = sorted(anom_base["year"].unique())
anom_classes = ["450", "250", "WMX"]
anom_rounds  = ["All"] + sorted(anom_base["round"].unique(), key=lambda x: int(x))
anom_motos   = ["All"] + sorted(anom_base["moto"].unique(), key=lambda x: int(x))


# ── Helper ────────────────────────────────────────────────────────────────────
def hex_to_rgba(hex_color, opacity=0.08):
    hex_color = hex_color.lstrip("#")
    r, g, b   = int(hex_color[0:2], 16), int(hex_color[2:4], 16), int(hex_color[4:6], 16)
    return f"rgba({r},{g},{b},{opacity})"

ANOM_COLORS = {
    "Recoverable":   "#1A7FE8",
    "Unrecoverable": "#E8641A",
}

VIEWS = [
    "1. Rider Frequency",
    "2. Lap in Moto (When do anomalies occur?)",
]

# ── Widgets ───────────────────────────────────────────────────────────────────
year_sel_a2 = widgets.Dropdown(
    options=anom_years, value=anom_years[-1],
    description="Year:", layout=widgets.Layout(width="200px")
)
class_sel_a2 = widgets.Dropdown(
    options=anom_classes, value="450",
    description="Class:", layout=widgets.Layout(width="200px")
)
threshold_sel_a2 = widgets.FloatSlider(
    value=2.5, min=1.5, max=4.0, step=0.25,
    description="SD Threshold:",
    layout=widgets.Layout(width="350px"),
    style={"description_width": "120px"},
)
view_sel_a2 = widgets.Dropdown(
    options=VIEWS, value=VIEWS[0],
    description="View:",
    layout=widgets.Layout(width="350px"),
    style={"description_width": "50px"},
)

btn_a2    = widgets.Button(description="Update", button_style="primary")
output_a2 = widgets.Output()

# ── Plot functions ─────────────────────────────────────────────────────────────
def plot_rider_frequency(rec, unrec):
    fig = go.Figure()

    # Build full rider list across both anomaly types
    all_riders = pd.concat([
        rec[["name"]], unrec[["name"]]
    ]).drop_duplicates()["name"].tolist()

    # Count motos entered per rider in the same class/year window
    motos_entered = (
        anom_base[
            (anom_base["year"]        == year_sel_a2.value) &
            (anom_base["class_label"] == class_sel_a2.value) &
            (anom_base["name"].isin(all_riders))
        ]
        .drop_duplicates(subset=["race_id", "name"])
        .groupby("name", observed=True)
        .size()
        .reset_index(name="motos")
    )

    for label, subset, color in [
        ("Recoverable",   rec,   ANOM_COLORS["Recoverable"]),
        ("Unrecoverable", unrec, ANOM_COLORS["Unrecoverable"]),
    ]:
        if subset.empty:
            continue

        counts = (
            subset.groupby("name", observed=True)
            .size()
            .reset_index(name="count")
            .merge(motos_entered, on="name", how="left")
            .sort_values("count", ascending=True)
        )
        counts["motos"] = counts["motos"].fillna(0).astype(int)

        fig.add_trace(go.Bar(
            y=counts["name"],
            x=counts["count"],
            name=label,
            orientation="h",
            marker_color=color,
            customdata=counts["motos"],
            hovertemplate=(
                "<b>%{y}</b><br>"
                f"{label}: %{{x}}<br>"
                "Motos entered: %{customdata}"
                "<extra></extra>"
            ),
        ))

    fig.update_layout(
        title="Anomaly Frequency by Rider",
        xaxis=dict(title="Number of Anomalies", dtick=1),
        yaxis=dict(title=""),
        barmode="group",
        height=max(400, len(motos_entered) * 20),
        width=950,
        margin=dict(l=180, r=60, t=60, b=60),
        legend=dict(title="Type"),
        hovermode="closest",
    )
    display(fig)


def plot_lap_in_moto(rec, unrec):
    fig      = go.Figure()
    all_vals = []

    for label, subset, color in [
        ("Recoverable",   rec,   ANOM_COLORS["Recoverable"]),
        ("Unrecoverable", unrec, ANOM_COLORS["Unrecoverable"]),
    ]:
        if subset.empty:
            continue

        lap_col = "lap" if label == "Recoverable" else "last_lap_num"
        pct     = (subset[lap_col] / subset["moto_max_lap"] * 100).dropna().values

        if len(pct) < 2:
            continue

        all_vals.extend(pct)

        kde            = gaussian_kde(pct, bw_method=0.4)
        x_range        = np.linspace(0, 100, 400)
        density        = kde(x_range)
        median         = np.median(pct)
        median_density = kde(np.array([median]))[0]

        fig.add_trace(go.Scatter(
            x=x_range, y=density,
            mode="lines",
            name=f"{label} (n={len(pct)}, median={median:.0f}% through)",
            line=dict(color=color, width=2),
            fill="tozeroy",
            fillcolor=hex_to_rgba(color, 0.08),
            hovertemplate=(
                f"<b>{label}</b><br>"
                "% through moto: %{x:.1f}%<br>"
                "Density: %{y:.4f}"
                "<extra></extra>"
            ),
        ))
        fig.add_trace(go.Scatter(
            x=[median, median],
            y=[0, median_density],
            mode="lines",
            line=dict(color=color, width=1.5, dash="dot"),
            showlegend=False, hoverinfo="skip",
        ))

    if not all_vals:
        print("Not enough data for KDE.")
        return

    fig.update_layout(
        title="When in the Moto Do Anomalies Occur?",
        xaxis=dict(
            title="% Through Moto",
            range=[0, 100],
            tickvals=[0, 25, 50, 75, 100],
            ticktext=["Start", "25%", "50%", "75%", "End"],
        ),
        yaxis=dict(title="Density", showticklabels=False),
        height=500, width=950,
        margin=dict(l=60, r=60, t=60, b=60),
        legend=dict(title="Type"),
        hovermode="x unified",
    )
    display(fig)


# ── Update callback ───────────────────────────────────────────────────────────
def update_a2(_):
    output_a2.clear_output(wait=True)
    with output_a2:

        rec, unrec = detect_anomalies(threshold_sel_a2.value)

        def apply_filters(subset):
            mask = (
                (subset["year"]        == year_sel_a2.value) &
                (subset["class_label"] == class_sel_a2.value)
            )
            return subset[mask]

        rec   = apply_filters(rec)
        unrec = apply_filters(unrec)

        total = len(rec) + len(unrec)
        if total == 0:
            print("No anomalies detected for selected filters.")
            return

        print(f"\n  Anomaly Characteristics — {class_sel_a2.value} | {year_sel_a2.value} | "
              f"SD threshold: {threshold_sel_a2.value}")
        print(f"  Recoverable: {len(rec)}  |  Unrecoverable: {len(unrec)}\n")

        if view_sel_a2.value == VIEWS[0]:
            plot_rider_frequency(rec, unrec)
        elif view_sel_a2.value == VIEWS[1]:
            plot_lap_in_moto(rec, unrec)

btn_a2.on_click(update_a2)

display(widgets.VBox([
    widgets.HBox([year_sel_a2, class_sel_a2, threshold_sel_a2]),
    widgets.HBox([view_sel_a2, btn_a2]),
    output_a2,
]))

update_a2(None)

In [41]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
import numpy as np

# ── Anomaly setup (self-contained — reproduces Section 16 Cell 1) ────────────
anom_base = (
    df.dropna(subset=["lap", "lap_time", "place", "finish_position"])
    .copy()
)
anom_base["lap"]             = anom_base["lap"].astype(float)
anom_base["place"]           = anom_base["place"].astype(float)
anom_base["finish_position"] = anom_base["finish_position"].astype(float)
anom_base["year"]            = anom_base["year"].astype(str)
anom_base["round"]           = anom_base["round"].astype(str)
anom_base["moto"]            = anom_base["moto"].astype(str)

rider_stats = (
    anom_base.groupby(["race_id", "name"], observed=True)["lap_time"]
    .agg(rider_median="median", rider_std="std", rider_laps="count")
    .reset_index()
)
anom_base = anom_base.merge(rider_stats, on=["race_id", "name"], how="left")
anom_base = anom_base[anom_base["rider_laps"] >= 3].copy()
anom_base = anom_base[
    anom_base["rider_std"].notna() & (anom_base["rider_std"] > 0)
].copy()
anom_base["rider_z"] = (
    (anom_base["lap_time"] - anom_base["rider_median"]) / anom_base["rider_std"]
)

anom_base = anom_base.sort_values(["race_id", "name", "lap"])
anom_base["prev_place"] = anom_base.groupby(
    ["race_id", "name"], observed=True
)["place"].shift(1)
anom_base["place_drop"] = anom_base["place"] - anom_base["prev_place"]

last_lap_place = (
    anom_base.sort_values(["race_id", "name", "lap"])
    .groupby(["race_id", "name"], observed=True)
    .agg(last_lap_num=("lap", "max"), last_lap_place=("place", "last"))
    .reset_index()
)
moto_max_lap = (
    anom_base.groupby("race_id", observed=True)["lap"]
    .max().rename("moto_max_lap").reset_index()
)
anom_base = anom_base.merge(last_lap_place, on=["race_id", "name"], how="left")
anom_base = anom_base.merge(moto_max_lap, on="race_id", how="left")

def detect_anomalies(threshold):
    recoverable = anom_base[
        (anom_base["rider_z"] >= threshold) &
        (anom_base["place_drop"] >= 2) &
        (anom_base["lap"] > 1) &
        (anom_base["last_lap_num"] == anom_base["moto_max_lap"]) &
        (abs(anom_base["finish_position"] - anom_base["last_lap_place"]) < 2)
    ].copy()
    recoverable["anomaly_type"] = "Recoverable"

    unrecoverable = anom_base.drop_duplicates(subset=["race_id", "name"]).copy()
    unrecoverable = unrecoverable[
        abs(unrecoverable["finish_position"] - unrecoverable["last_lap_place"]) >= 2
    ].copy()
    unrecoverable["anomaly_type"] = "Unrecoverable"
    unrecoverable["lap"]        = unrecoverable["last_lap_num"]
    unrecoverable["rider_z"]    = (
        (unrecoverable["lap_time"] - unrecoverable["rider_median"]) /
        unrecoverable["rider_std"]
    )
    unrecoverable["place_drop"] = (
        unrecoverable["finish_position"] - unrecoverable["last_lap_place"]
    )
    return recoverable, unrecoverable

def fmt_time(seconds):
    if pd.isna(seconds): return "—"
    m = int(seconds // 60); s = seconds % 60
    return f"{m}:{s:06.3f}"

def fmt_lap_time_col(series):
    return series.apply(fmt_time)

anom_years   = sorted(anom_base["year"].unique())
anom_classes = ["450", "250", "WMX"]
anom_rounds  = ["All"] + sorted(anom_base["round"].unique(), key=lambda x: int(x))
anom_motos   = ["All"] + sorted(anom_base["moto"].unique(), key=lambda x: int(x))

# ── prize_base (self-contained — reproduces Section 14 Prize Money) ──────────
TCMX_POINTS = {
    1: 25, 2: 22, 3: 20, 4: 18, 5: 16,
    6: 15, 7: 14, 8: 13, 9: 12, 10: 11,
    11: 10, 12: 9, 13: 8, 14: 7, 15: 6,
    16: 5, 17: 4, 18: 3, 19: 2, 20: 1,
}
prize_base = (
    df.drop_duplicates(subset=["race_id", "name"])
    .dropna(subset=["finish_position"]).copy()
)
prize_base["year"]            = prize_base["year"].astype(str)
prize_base["round"]           = prize_base["round"].astype(int)
prize_base["moto"]            = prize_base["moto"].astype(int)
prize_base["finish_position"] = prize_base["finish_position"].astype(int)
prize_base["moto_points"]     = prize_base["finish_position"].map(TCMX_POINTS).fillna(0)

# ── Simulation logic ──────────────────────────────────────────────────────────
def simulate_no_dnf(year_val, class_val):
    """
    For each unrecoverable DNF in the selected year/class:
    - Restore the rider to their last_lap_place for finish position
    - Cascade: all riders who were behind that place move down one
    - Recompute moto points, round combined points, overall place, season standings
    """

    # SD threshold doesn't affect unrecoverable detection — pass a fixed value
    _, unrec = detect_anomalies(2.5)
    unrec = unrec[
        (unrec["year"] == year_val) &
        (unrec["class_label"] == class_val)
        ].copy()

    if unrec.empty:
        return None, None, "No unrecoverable DNFs detected for selected filters."

    # ── Actual finish positions for all riders ────────────────────────────────
    actual = (
        prize_base[
            (prize_base["year"] == year_val) &
            (prize_base["class_label"] == class_val)
            ][["race_id", "name", "round", "moto", "finish_position", "moto_points"]]
        .copy()
    )

    # ── Build simulated finish positions ─────────────────────────────────────
    sim = actual.copy()
    sim["sim_finish"] = sim["finish_position"]
    sim["sim_moto_points"] = sim["moto_points"]

    # Process each DNF rider — one race_id at a time
    for race_id, group in unrec.groupby("race_id", observed=True):
        race_sim = sim[sim["race_id"] == race_id].copy()

        for _, dnf_row in group.iterrows():
            rider = dnf_row["name"]
            restored_pos = int(dnf_row["last_lap_place"])

            # Current sim finish for this rider
            current_pos = race_sim.loc[
                race_sim["name"] == rider, "sim_finish"
            ].values
            if len(current_pos) == 0:
                continue
            current_pos = int(current_pos[0])

            # Riders whose sim_finish is between restored_pos and current_pos-1
            # need to move down one (cascade)
            cascade_mask = (
                    (race_sim["name"] != rider) &
                    (race_sim["sim_finish"] >= restored_pos) &
                    (race_sim["sim_finish"] < current_pos)
            )
            race_sim.loc[cascade_mask, "sim_finish"] += 1

            # Restore DNF rider
            race_sim.loc[race_sim["name"] == rider, "sim_finish"] = restored_pos

        # Remap sim_finish to TCMX points
        race_sim["sim_moto_points"] = race_sim["sim_finish"].map(TCMX).fillna(0)

        # Write back
        sim.loc[sim["race_id"] == race_id, "sim_finish"] = race_sim["sim_finish"].values
        sim.loc[sim["race_id"] == race_id, "sim_moto_points"] = race_sim["sim_moto_points"].values

    # ── Round combined points (actual) ────────────────────────────────────────
    actual_round = (
        actual.groupby(["name", "round"], observed=True)
        .agg(combined_points=("moto_points", "sum"),
             last_moto_pos=("finish_position", "last"))
        .reset_index()
    )
    actual_round["overall_place"] = (
        actual_round.groupby("round", observed=True)
        .apply(lambda g: g.sort_values(
            ["combined_points", "last_moto_pos"],
            ascending=[False, True]
        ).assign(overall_place=range(1, len(g) + 1))["overall_place"],
               include_groups=False)
        .reset_index(level=0, drop=True)
    )
    actual_season = (
        actual_round.groupby("name", observed=True)["combined_points"]
        .sum()
        .reset_index(name="actual_season_points")
        .sort_values("actual_season_points", ascending=False)
        .reset_index(drop=True)
    )
    actual_season.index += 1
    actual_season["actual_rank"] = actual_season.index

    # ── Round combined points (simulated) ─────────────────────────────────────
    sim_round = (
        sim.groupby(["name", "round"], observed=True)
        .agg(combined_points=("sim_moto_points", "sum"),
             last_moto_pos=("sim_finish", "last"))
        .reset_index()
    )
    sim_round["overall_place"] = (
        sim_round.groupby("round", observed=True)
        .apply(lambda g: g.sort_values(
            ["combined_points", "last_moto_pos"],
            ascending=[False, True]
        ).assign(overall_place=range(1, len(g) + 1))["overall_place"],
               include_groups=False)
        .reset_index(level=0, drop=True)
    )
    sim_season = (
        sim_round.groupby("name", observed=True)["combined_points"]
        .sum()
        .reset_index(name="sim_season_points")
        .sort_values("sim_season_points", ascending=False)
        .reset_index(drop=True)
    )
    sim_season.index += 1
    sim_season["sim_rank"] = sim_season.index

    # ── Merge actual vs simulated ─────────────────────────────────────────────
    comparison = actual_season.merge(sim_season, on="name", how="outer")
    comparison["points_delta"] = (
            comparison["sim_season_points"] - comparison["actual_season_points"]
    )
    comparison["rank_delta"] = (
            comparison["actual_rank"] - comparison["sim_rank"]
    )
    comparison = comparison.sort_values("sim_rank").reset_index(drop=True)

    # ── DNF summary ───────────────────────────────────────────────────────────
    dnf_summary = unrec[[
        "name", "round", "moto", "track",
        "last_lap_num", "moto_max_lap",
        "last_lap_place", "finish_position"
    ]].copy()
    dnf_summary["last_lap_num"] = dnf_summary["last_lap_num"].astype(int)
    dnf_summary["moto_max_lap"] = dnf_summary["moto_max_lap"].astype(int)
    dnf_summary["last_lap_place"] = dnf_summary["last_lap_place"].astype(int)
    dnf_summary["finish_position"] = dnf_summary["finish_position"].astype(int)
    dnf_summary = dnf_summary.rename(columns={
        "name": "Rider",
        "round": "Rd",
        "moto": "Moto",
        "track": "Track",
        "last_lap_num": "Last Lap",
        "moto_max_lap": "Moto Laps",
        "last_lap_place": "Place at DNF",
        "finish_position": "Actual Finish",
    })

    return comparison, dnf_summary, None


# ── Widgets ───────────────────────────────────────────────────────────────────
year_sel_a3 = widgets.Dropdown(
    options=anom_years, value=anom_years[-1],
    description="Year:", layout=widgets.Layout(width="200px")
)
class_sel_a3 = widgets.Dropdown(
    options=anom_classes, value="450",
    description="Class:", layout=widgets.Layout(width="200px")
)

btn_a3 = widgets.Button(description="Update", button_style="primary")
output_a3 = widgets.Output()


# ── Update callback ───────────────────────────────────────────────────────────
def update_a3(_):
    output_a3.clear_output(wait=True)
    with output_a3:

        comparison, dnf_summary, err = simulate_no_dnf(
            year_sel_a3.value,
            class_sel_a3.value,
        )

        if err:
            print(f"  {err}")
            return

        # ── DNF summary table ─────────────────────────────────────────────────
        print(f"\n  What if DNFs were removed? — {class_sel_a3.value} | {year_sel_a3.value}\n")
        display(dnf_summary.reset_index(drop=True))

        # ── Standings comparison table ────────────────────────────────────────
        # Only show riders affected (points delta != 0) + top 5 for context
        top5 = comparison.head(5)["name"].tolist()
        affected = comparison[comparison["points_delta"] != 0]["name"].tolist()
        show_riders = comparison[
            comparison["name"].isin(set(top5 + affected))
        ].copy()

        print(f"\n  Season Standings Comparison\n")

        display_df = show_riders[[
            "name", "actual_rank", "actual_season_points",
            "sim_rank", "sim_season_points", "points_delta", "rank_delta"
        ]].copy()

        display_df["actual_season_points"] = display_df["actual_season_points"].astype(int)
        display_df["sim_season_points"] = display_df["sim_season_points"].astype(int)

        display_df["rank_delta_str"] = display_df["rank_delta"].apply(
            lambda x: f"+{int(x)}" if x > 0 else (f"{int(x)}" if x < 0 else "—")
        )
        display_df["points_delta_str"] = display_df["points_delta"].apply(
            lambda x: f"+{int(x)}" if x > 0 else (f"{int(x)}" if x < 0 else "—")
        )
        display_df = display_df.rename(columns={
            "name": "Rider",
            "actual_rank": "Actual Rank",
            "actual_season_points": "Actual Pts",
            "sim_rank": "Sim Rank",
            "sim_season_points": "Sim Pts",
            "points_delta_str": "Pts Δ",
            "rank_delta_str": "Rank Δ",
        }).reset_index(drop=True)
        display_df.index += 1

        display(display_df[[
            "Rider", "Actual Rank", "Actual Pts",
            "Sim Rank", "Sim Pts", "Pts Δ", "Rank Δ"
        ]])

        print(f"  Cascade effect applied — riders behind restored position move down one place.")


btn_a3.on_click(update_a3)

display(widgets.VBox([
    widgets.HBox([year_sel_a3, class_sel_a3]),
    btn_a3,
    output_a3,
]))

update_a3(None)

<a id="win-probability-model"></a>
## 17. Win Probability Model
- Model 1: Logistic Regression
- Model 2: LR w/o balanced class weighting
- Model 3: LR w/o behind_time
- Model 4: LR w/ Platt Scaling
- Model 5: XGBoost
- Model 6: Bradley-Terry

In [42]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import plotly.graph_objects as go
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.calibration import calibration_curve
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")

# ── Build model base ──────────────────────────────────────────────────────────
model_base = (
    df.dropna(subset=["lap", "lap_time", "place",
                       "finish_position", "race_id"])
    .copy()
)
model_base["lap"]             = model_base["lap"].astype(float)
model_base["place"]           = model_base["place"].astype(float)
model_base["behind_time"]     = model_base["behind_time"].fillna(0).astype(float)
model_base["finish_position"] = model_base["finish_position"].astype(float)
model_base["year"]            = model_base["year"].astype(str)

# ── Target ────────────────────────────────────────────────────────────────────
model_base["won_moto"] = (model_base["finish_position"] == 1).astype(int)

# ── Fraction of race remaining ────────────────────────────────────────────────
moto_max_laps = (
    model_base.groupby("race_id", observed=True)["lap"]
    .max()
    .rename("max_lap")
    .reset_index()
)
model_base = model_base.merge(moto_max_laps, on="race_id", how="left")
model_base["frac_remaining"] = (
    (model_base["max_lap"] - model_base["lap"]) / model_base["max_lap"]
)

# ── Interaction terms ─────────────────────────────────────────────────────────
model_base["place_x_frac"]  = model_base["place"]       * model_base["frac_remaining"]
model_base["behind_x_frac"] = model_base["behind_time"] * model_base["frac_remaining"]

# ── Feature set ───────────────────────────────────────────────────────────────
FEATURES = [
    "place",
    "behind_time",
    "frac_remaining",
    "place_x_frac",
    "behind_x_frac",
]

# ── Train / test split ────────────────────────────────────────────────────────
train = model_base[
    model_base["year"] == "2024"
].dropna(subset=FEATURES + ["won_moto"]).copy()

test = model_base[
    model_base["year"] == "2025"
].dropna(subset=FEATURES + ["won_moto"]).copy()

X_train = train[FEATURES].values
y_train = train["won_moto"].values
X_test  = test[FEATURES].values
y_test  = test["won_moto"].values

# ── Scale features ────────────────────────────────────────────────────────────
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

# ── Fit model ─────────────────────────────────────────────────────────────────
model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42,
)
model.fit(X_train, y_train)

# ── Predictions ───────────────────────────────────────────────────────────────
train["win_prob"] = model.predict_proba(X_train)[:, 1]
test["win_prob"]  = model.predict_proba(X_test)[:, 1]

# Write win_prob back to model_base
model_base = model_base.merge(
    pd.concat([
        train[["race_id", "name", "lap", "win_prob"]],
        test[["race_id", "name", "lap", "win_prob"]],
    ]),
    on=["race_id", "name", "lap"],
    how="left"
)

# ── Metrics ───────────────────────────────────────────────────────────────────
train_auc   = roc_auc_score(y_train, train["win_prob"])
test_auc    = roc_auc_score(y_test,  test["win_prob"])
train_brier = brier_score_loss(y_train, train["win_prob"])
test_brier  = brier_score_loss(y_test,  test["win_prob"])

# ── Coefficient table ─────────────────────────────────────────────────────────
coef_df = pd.DataFrame({
    "Feature":     FEATURES,
    "Coefficient": model.coef_[0].round(4),
    "Odds Ratio":  np.exp(model.coef_[0]).round(4),
}).sort_values("Coefficient", ascending=False).reset_index(drop=True)
coef_df.index += 1

# ── Log results for comparison ────────────────────────────────────────────────
if "model_results" not in globals():
    model_results = {}
model_results["Model 1 — Logistic Regression"] = {
    "Train AUC":   round(train_auc, 4),
    "Test AUC":    round(test_auc, 4),
    "Train Brier": round(train_brier, 4),
    "Test Brier":  round(test_brier, 4),
}

# ── Calibration curve (test set) ──────────────────────────────────────────────
frac_pos, mean_pred = calibration_curve(
    y_test, test["win_prob"], n_bins=10, strategy="quantile"
)

# ── Print model summary ───────────────────────────────────────────────────────
print("=" * 60)
print("  Win Probability Model — Logistic Regression (Model 1)")
print("=" * 60)
print(f"\n  Training set (2024): {len(train):,} observations")
print(f"  Test set     (2025): {len(test):,} observations")
print(f"  Positive rate train: {y_train.mean():.3f}")
print(f"  Positive rate test:  {y_test.mean():.3f}")
print(f"\n  {'Metric':<25} {'Train':>10} {'Test':>10}")
print(f"  {'-'*45}")
print(f"  {'ROC-AUC':<25} {train_auc:>10.4f} {test_auc:>10.4f}")
print(f"  {'Brier Score':<25} {train_brier:>10.4f} {test_brier:>10.4f}")
print(f"  Features: place, behind_time, frac_remaining + two interaction terms\n")
print("\n  Feature Coefficients (scaled):\n")
display(coef_df)

# ── Calibration plot ──────────────────────────────────────────────────────────
fig_cal = go.Figure()
fig_cal.add_trace(go.Scatter(
    x=mean_pred, y=frac_pos,
    mode="lines+markers",
    name="Model 1",
    line=dict(color="#1A7FE8", width=2),
    marker=dict(size=8),
    hovertemplate=(
        "Predicted: %{x:.2f}<br>"
        "Actual: %{y:.2f}<extra></extra>"
    ),
))
fig_cal.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1],
    mode="lines",
    name="Perfect calibration",
    line=dict(color="grey", width=1, dash="dash"),
))
fig_cal.update_layout(
    title="Calibration Plot — Test Set (2025) — Model 1",
    xaxis=dict(title="Mean Predicted Probability", range=[0, 1]),
    yaxis=dict(title="Fraction of Positives",      range=[0, 1]),
    height=450, width=700,
    margin=dict(l=60, r=60, t=60, b=60),
    legend=dict(title=""),
)
display(fig_cal)

print("Model Comparison:")
comparison_df = pd.DataFrame(model_results).T
comparison_df.index.name = "Model"
comparison_df = comparison_df.reset_index()
display(comparison_df)

# ── Notes: Interpreting the metrics above ─────────────────────────────────────
print("\n" + "=" * 60)
print("  Notes: Interpreting the metrics above")
print("=" * 60)
print("""
  Positive rate
    Share of lap-rows belonging to the eventual moto winner.
    Roughly tracks 1 / (avg field size), so train and test rates
    being close is a good sanity check on the target.

  Odds ratio (exp(coefficient))
    Features are standardized, so this is the multiplicative
    change in win odds per 1-SD increase in that feature.
    OR > 1 -> higher win odds, OR < 1 -> lower, OR = 1 -> no effect.

  ROC-AUC
    P(a random winner-lap ranks above a random non-winner-lap).
    0.5 = random, 1.0 = perfect ranking. Only cares about relative
    order, so it's largely insensitive to class_weight="balanced".

  Brier score
    Mean squared error between predicted prob and actual 0/1
    outcome. Lower is better, 0 = perfect. Baseline ~
    positive_rate * (1 - positive_rate). Unlike AUC, this IS
    sensitive to class_weight="balanced" -- expect inflated
    win_prob values relative to true rates.

  Calibration plot
    Test predictions binned into 10 quantile groups; mean predicted
    prob vs. actual win fraction per bin. Above the diagonal =
    underconfident, below = overconfident. Likely skewed here since
    class_weight="balanced" treats a ~5-8% positive rate as ~50%.
""")

# ── Interactive moto viewer ───────────────────────────────────────────────────
viewer_base = model_base.copy()
viewer_base["round"] = viewer_base["round"].astype(str)
viewer_base["moto"]  = viewer_base["moto"].astype(str)
viewer_base["year"]  = viewer_base["year"].astype(str)

viewer_years  = sorted(viewer_base["year"].unique())
viewer_rounds = sorted(viewer_base["round"].unique(), key=lambda x: int(x))
viewer_motos  = sorted(viewer_base["moto"].unique(),  key=lambda x: int(x))

year_sel_wp = widgets.Dropdown(
    options=viewer_years, value=viewer_years[-1],
    description="Year:", layout=widgets.Layout(width="200px")
)
class_sel_wp = widgets.Dropdown(
    options=["450", "250", "WMX"], value="450",
    description="Class:", layout=widgets.Layout(width="200px")
)
round_sel_wp = widgets.Dropdown(
    options=viewer_rounds, value=viewer_rounds[0],
    description="Round:", layout=widgets.Layout(width="200px")
)
moto_sel_wp = widgets.Dropdown(
    options=viewer_motos, value=viewer_motos[0],
    description="Moto:", layout=widgets.Layout(width="200px")
)
rider_sel_wp = widgets.Dropdown(
    options=[], value=None,
    description="Rider:", layout=widgets.Layout(width="280px")
)

def refresh_wp_riders(*args):
    mask = (
        (viewer_base["year"]        == year_sel_wp.value) &
        (viewer_base["class_label"] == class_sel_wp.value) &
        (viewer_base["round"]       == round_sel_wp.value) &
        (viewer_base["moto"]        == moto_sel_wp.value)
    )
    riders  = sorted(viewer_base[mask]["name"].dropna().unique().tolist())
    current = rider_sel_wp.value
    rider_sel_wp.options = riders
    rider_sel_wp.value   = current if current in riders else (riders[0] if riders else None)

for w in [year_sel_wp, class_sel_wp, round_sel_wp, moto_sel_wp]:
    w.observe(refresh_wp_riders, names="value")

refresh_wp_riders()

btn_wp    = widgets.Button(description="Update", button_style="primary")
output_wp = widgets.Output()

def plot_win_prob(year_val, class_val, round_val, moto_val, rider):
    if not rider:
        print("No riders found.")
        return

    mask = (
        (viewer_base["year"]        == year_val) &
        (viewer_base["class_label"] == class_val) &
        (viewer_base["round"]       == round_val) &
        (viewer_base["moto"]        == moto_val)
    )
    moto_data  = viewer_base[mask].copy()
    rider_data = moto_data[moto_data["name"] == rider].sort_values("lap")

    if rider_data.empty:
        print(f"No data for {rider}.")
        return

    fig = go.Figure()

    # Grey traces for all other riders
    for other in moto_data["name"].unique():
        if other == rider:
            continue
        od = moto_data[moto_data["name"] == other].sort_values("lap")
        if od["win_prob"].isna().all():
            continue
        fig.add_trace(go.Scatter(
            x=od["lap"], y=od["win_prob"],
            mode="lines",
            line=dict(color="lightgrey", width=1),
            showlegend=False,
            hoverinfo="skip",
        ))

    # Highlighted rider
    fig.add_trace(go.Scatter(
        x=rider_data["lap"],
        y=rider_data["win_prob"],
        mode="lines+markers",
        name=rider,
        line=dict(color="#1A7FE8", width=3),
        marker=dict(size=7),
        customdata=rider_data[[
            "place", "behind_time", "frac_remaining"
        ]].values,
        hovertemplate=(
            f"<b>{rider}</b><br>"
            "Lap: %{x}<br>"
            "Win prob: %{y:.1%}<br>"
            "Place: %{customdata[0]:.0f}<br>"
            "Gap to leader: %{customdata[1]:.2f}s<br>"
            "Race remaining: %{customdata[2]:.1%}"
            "<extra></extra>"
        ),
    ))

    # Actual finish annotation
    actual_finish = int(rider_data["finish_position"].iloc[0])
    suffix = (
        "st" if actual_finish == 1 else
        "nd" if actual_finish == 2 else
        "rd" if actual_finish == 3 else "th"
    )
    fig.add_annotation(
        x=rider_data["lap"].max(),
        y=rider_data["win_prob"].dropna().iloc[-1] if rider_data["win_prob"].notna().any() else 0,
        text=f"  Finished {actual_finish}{suffix}",
        showarrow=False,
        font=dict(color="#1A7FE8", size=11),
        xanchor="left",
    )

    track = rider_data["track"].iloc[0]
    fig.update_layout(
        title=f"Win Probability — {rider} | {class_val} | {year_val} | Rd {round_val} | Moto {moto_val} | {track}",
        xaxis=dict(title="Lap", dtick=1),
        yaxis=dict(
            title="Win Probability",
            range=[0, 1],
            tickformat=".0%",
        ),
        height=500, width=950,
        margin=dict(l=60, r=60, t=60, b=60),
        legend=dict(title=""),
        hovermode="x unified",
    )
    display(fig)

def update_wp(_):
    output_wp.clear_output(wait=True)
    with output_wp:
        plot_win_prob(
            year_sel_wp.value, class_sel_wp.value,
            round_sel_wp.value, moto_sel_wp.value,
            rider_sel_wp.value,
        )

btn_wp.on_click(update_wp)

display(widgets.VBox([
    widgets.HBox([year_sel_wp, class_sel_wp, round_sel_wp, moto_sel_wp]),
    rider_sel_wp,
    btn_wp,
    output_wp,
]))

update_wp(None)

  Win Probability Model — Logistic Regression (Model 1)

  Training set (2024): 16,217 observations
  Test set     (2025): 15,720 observations
  Positive rate train: 0.036
  Positive rate test:  0.038

  Metric                         Train       Test
  ---------------------------------------------
  ROC-AUC                       0.9909     0.9918
  Brier Score                   0.0478     0.0507
  Features: place, behind_time, frac_remaining + two interaction terms


  Feature Coefficients (scaled):



,Feature,Coefficient,Odds Ratio
1,place_x_frac,2.2886,9.8614
2,frac_remaining,-0.4223,0.6555
3,behind_x_frac,-2.7883,0.0615
4,place,-4.3705,0.0126
5,behind_time,-12.4952,0.0000


Model Comparison:


,Model,Train AUC,Test AUC,Train Brier,Test Brier
0,Model 1 — Logistic Regression,0.9909,0.9918,0.0478,0.0507



  Notes: Interpreting the metrics above

  Positive rate
    Share of lap-rows belonging to the eventual moto winner.
    Roughly tracks 1 / (avg field size), so train and test rates
    being close is a good sanity check on the target.

  Odds ratio (exp(coefficient))
    Features are standardized, so this is the multiplicative
    change in win odds per 1-SD increase in that feature.
    OR > 1 -> higher win odds, OR < 1 -> lower, OR = 1 -> no effect.

  ROC-AUC
    P(a random winner-lap ranks above a random non-winner-lap).
    0.5 = random, 1.0 = perfect ranking. Only cares about relative
    order, so it's largely insensitive to class_weight="balanced".

  Brier score
    Mean squared error between predicted prob and actual 0/1
    outcome. Lower is better, 0 = perfect. Baseline ~
    positive_rate * (1 - positive_rate). Unlike AUC, this IS
    sensitive to class_weight="balanced" -- expect inflated
    win_prob values relative to true rates.

  Calibration plot
    Test predicti

In [43]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import plotly.graph_objects as go
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.calibration import calibration_curve
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")

# ── Build model base ──────────────────────────────────────────────────────────
model_base = (
    df.dropna(subset=["lap", "lap_time", "place",
                       "finish_position", "race_id"])
    .copy()
)
model_base["lap"]             = model_base["lap"].astype(float)
model_base["place"]           = model_base["place"].astype(float)
model_base["behind_time"]     = model_base["behind_time"].fillna(0).astype(float)
model_base["finish_position"] = model_base["finish_position"].astype(float)
model_base["year"]            = model_base["year"].astype(str)

# ── Target ────────────────────────────────────────────────────────────────────
model_base["won_moto"] = (model_base["finish_position"] == 1).astype(int)

# ── Fraction of race remaining ────────────────────────────────────────────────
moto_max_laps = (
    model_base.groupby("race_id", observed=True)["lap"]
    .max()
    .rename("max_lap")
    .reset_index()
)
model_base = model_base.merge(moto_max_laps, on="race_id", how="left")
model_base["frac_remaining"] = (
    (model_base["max_lap"] - model_base["lap"]) / model_base["max_lap"]
)

# ── Interaction terms ─────────────────────────────────────────────────────────
model_base["place_x_frac"]  = model_base["place"]       * model_base["frac_remaining"]
model_base["behind_x_frac"] = model_base["behind_time"] * model_base["frac_remaining"]

# ── Feature set ───────────────────────────────────────────────────────────────
FEATURES = [
    "place",
    "behind_time",
    "frac_remaining",
    "place_x_frac",
    "behind_x_frac",
]

# ── Train / test split ────────────────────────────────────────────────────────
train = model_base[
    model_base["year"] == "2024"
].dropna(subset=FEATURES + ["won_moto"]).copy()

test = model_base[
    model_base["year"] == "2025"
].dropna(subset=FEATURES + ["won_moto"]).copy()

X_train = train[FEATURES].values
y_train = train["won_moto"].values
X_test  = test[FEATURES].values
y_test  = test["won_moto"].values

# ── Scale features ────────────────────────────────────────────────────────────
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

# ── Fit model (no class weighting) ───────────────────────────────────────────
model = LogisticRegression(
    max_iter=1000,
    random_state=42,
)
model.fit(X_train, y_train)

# ── Predictions ───────────────────────────────────────────────────────────────
train["win_prob"] = model.predict_proba(X_train)[:, 1]
test["win_prob"]  = model.predict_proba(X_test)[:, 1]

# Write win_prob back to model_base
model_base = model_base.merge(
    pd.concat([
        train[["race_id", "name", "lap", "win_prob"]],
        test[["race_id", "name", "lap", "win_prob"]],
    ]),
    on=["race_id", "name", "lap"],
    how="left"
)

# ── Metrics ───────────────────────────────────────────────────────────────────
train_auc   = roc_auc_score(y_train, train["win_prob"])
test_auc    = roc_auc_score(y_test,  test["win_prob"])
train_brier = brier_score_loss(y_train, train["win_prob"])
test_brier  = brier_score_loss(y_test,  test["win_prob"])

# ── Coefficient table ─────────────────────────────────────────────────────────
coef_df = pd.DataFrame({
    "Feature":     FEATURES,
    "Coefficient": model.coef_[0].round(4),
    "Odds Ratio":  np.exp(model.coef_[0]).round(4),
}).sort_values("Coefficient", ascending=False).reset_index(drop=True)
coef_df.index += 1

# ── Log results for comparison ────────────────────────────────────────────────
if "model_results" not in globals():
    model_results = {}
model_results["Model 2 — LR w/ No class weighting"] = {
    "Train AUC":   round(train_auc, 4),
    "Test AUC":    round(test_auc, 4),
    "Train Brier": round(train_brier, 4),
    "Test Brier":  round(test_brier, 4),
}

# ── Calibration curve (test set) ──────────────────────────────────────────────
frac_pos, mean_pred = calibration_curve(
    y_test, test["win_prob"], n_bins=10, strategy="quantile"
)

# ── Print model summary ───────────────────────────────────────────────────────
print("=" * 60)
print("  Win Probability Model — Logistic Regression (Model 2)")
print("  No balanced class weighting (no 50/50 winner/loser assumption)")
print("=" * 60)
print(f"\n  Training set (2024): {len(train):,} observations")
print(f"  Test set     (2025): {len(test):,} observations")
print(f"  Positive rate train: {y_train.mean():.3f}")
print(f"  Positive rate test:  {y_test.mean():.3f}")
print(f"\n  {'Metric':<25} {'Train':>10} {'Test':>10}")
print(f"  {'-'*45}")
print(f"  {'ROC-AUC':<25} {train_auc:>10.4f} {test_auc:>10.4f}")
print(f"  {'Brier Score':<25} {train_brier:>10.4f} {test_brier:>10.4f}")
print(f"  Features: place, behind_time, frac_remaining + two interaction terms\n")
print("\n  Feature Coefficients (scaled):\n")
display(coef_df)

# ── Calibration plot ──────────────────────────────────────────────────────────
fig_cal = go.Figure()
fig_cal.add_trace(go.Scatter(
    x=mean_pred, y=frac_pos,
    mode="lines+markers",
    name="Model 2",
    line=dict(color="#1A7FE8", width=2),
    marker=dict(size=8),
    hovertemplate=(
        "Predicted: %{x:.2f}<br>"
        "Actual: %{y:.2f}<extra></extra>"
    ),
))
fig_cal.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1],
    mode="lines",
    name="Perfect calibration",
    line=dict(color="grey", width=1, dash="dash"),
))
fig_cal.update_layout(
    title="Calibration Plot — Test Set (2025) — Model 2",
    xaxis=dict(title="Mean Predicted Probability", range=[0, 1]),
    yaxis=dict(title="Fraction of Positives",      range=[0, 1]),
    height=450, width=700,
    margin=dict(l=60, r=60, t=60, b=60),
    legend=dict(title=""),
)
display(fig_cal)

print("Model Comparison:")
comparison_df = pd.DataFrame(model_results).T
comparison_df.index.name = "Model"
comparison_df = comparison_df.reset_index()
display(comparison_df)

# ── Notes: Interpreting the metrics above ─────────────────────────────────────
print("\n" + "=" * 60)
print("  Notes: Interpreting the metrics above")
print("=" * 60)
print("""
  Positive rate
    Share of lap-rows belonging to the eventual moto winner.
    Roughly tracks 1 / (avg field size), so train and test rates
    being close is a good sanity check on the target.

  Odds ratio (exp(coefficient))
    Features are standardized, so this is the multiplicative
    change in win odds per 1-SD increase in that feature.
    OR > 1 -> higher win odds, OR < 1 -> lower, OR = 1 -> no effect.

  ROC-AUC
    P(a random winner-lap ranks above a random non-winner-lap).
    0.5 = random, 1.0 = perfect ranking. Only cares about relative
    order, so it's largely insensitive to class weighting -- expect
    a similar value to Model 1.

  Brier score
    Mean squared error between predicted prob and actual 0/1
    outcome. Lower is better, 0 = perfect. Baseline ~
    positive_rate * (1 - positive_rate). With no class weighting,
    predictions are fit to the true ~5-8% win rate directly, so
    this should improve noticeably vs. Model 1.

  Calibration plot
    Test predictions binned into 10 quantile groups; mean predicted
    prob vs. actual win fraction per bin. Above the diagonal =
    underconfident, below = overconfident. With no class weighting,
    this should sit much closer to the diagonal than Model 1, where
    class_weight="balanced" inflated predictions toward 50/50.

  Model 1 vs Model 2
    Same features, same train/test split -- only difference is
    class_weight="balanced" (Model 1) vs none (Model 2). AUC should
    be similar across both; Brier and calibration are where the
    difference shows up, and Model 2 should look better on both.
""")

# ── Interactive moto viewer ───────────────────────────────────────────────────
viewer_base = model_base.copy()
viewer_base["round"] = viewer_base["round"].astype(str)
viewer_base["moto"]  = viewer_base["moto"].astype(str)
viewer_base["year"]  = viewer_base["year"].astype(str)

viewer_years  = sorted(viewer_base["year"].unique())
viewer_rounds = sorted(viewer_base["round"].unique(), key=lambda x: int(x))
viewer_motos  = sorted(viewer_base["moto"].unique(),  key=lambda x: int(x))

year_sel_wp = widgets.Dropdown(
    options=viewer_years, value=viewer_years[-1],
    description="Year:", layout=widgets.Layout(width="200px")
)
class_sel_wp = widgets.Dropdown(
    options=["450", "250", "WMX"], value="450",
    description="Class:", layout=widgets.Layout(width="200px")
)
round_sel_wp = widgets.Dropdown(
    options=viewer_rounds, value=viewer_rounds[0],
    description="Round:", layout=widgets.Layout(width="200px")
)
moto_sel_wp = widgets.Dropdown(
    options=viewer_motos, value=viewer_motos[0],
    description="Moto:", layout=widgets.Layout(width="200px")
)
rider_sel_wp = widgets.Dropdown(
    options=[], value=None,
    description="Rider:", layout=widgets.Layout(width="280px")
)

def refresh_wp_riders(*args):
    mask = (
        (viewer_base["year"]        == year_sel_wp.value) &
        (viewer_base["class_label"] == class_sel_wp.value) &
        (viewer_base["round"]       == round_sel_wp.value) &
        (viewer_base["moto"]        == moto_sel_wp.value)
    )
    riders  = sorted(viewer_base[mask]["name"].dropna().unique().tolist())
    current = rider_sel_wp.value
    rider_sel_wp.options = riders
    rider_sel_wp.value   = current if current in riders else (riders[0] if riders else None)

for w in [year_sel_wp, class_sel_wp, round_sel_wp, moto_sel_wp]:
    w.observe(refresh_wp_riders, names="value")

refresh_wp_riders()

btn_wp    = widgets.Button(description="Update", button_style="primary")
output_wp = widgets.Output()

def plot_win_prob(year_val, class_val, round_val, moto_val, rider):
    if not rider:
        print("No riders found.")
        return

    mask = (
        (viewer_base["year"]        == year_val) &
        (viewer_base["class_label"] == class_val) &
        (viewer_base["round"]       == round_val) &
        (viewer_base["moto"]        == moto_val)
    )
    moto_data  = viewer_base[mask].copy()
    rider_data = moto_data[moto_data["name"] == rider].sort_values("lap")

    if rider_data.empty:
        print(f"No data for {rider}.")
        return

    fig = go.Figure()

    # Grey traces for all other riders
    for other in moto_data["name"].unique():
        if other == rider:
            continue
        od = moto_data[moto_data["name"] == other].sort_values("lap")
        if od["win_prob"].isna().all():
            continue
        fig.add_trace(go.Scatter(
            x=od["lap"], y=od["win_prob"],
            mode="lines",
            line=dict(color="lightgrey", width=1),
            showlegend=False,
            hoverinfo="skip",
        ))

    # Highlighted rider
    fig.add_trace(go.Scatter(
        x=rider_data["lap"],
        y=rider_data["win_prob"],
        mode="lines+markers",
        name=rider,
        line=dict(color="#1A7FE8", width=3),
        marker=dict(size=7),
        customdata=rider_data[[
            "place", "behind_time", "frac_remaining"
        ]].values,
        hovertemplate=(
            f"<b>{rider}</b><br>"
            "Lap: %{x}<br>"
            "Win prob: %{y:.1%}<br>"
            "Place: %{customdata[0]:.0f}<br>"
            "Gap to leader: %{customdata[1]:.2f}s<br>"
            "Race remaining: %{customdata[2]:.1%}"
            "<extra></extra>"
        ),
    ))

    # Actual finish annotation
    actual_finish = int(rider_data["finish_position"].iloc[0])
    suffix = (
        "st" if actual_finish == 1 else
        "nd" if actual_finish == 2 else
        "rd" if actual_finish == 3 else "th"
    )
    fig.add_annotation(
        x=rider_data["lap"].max(),
        y=rider_data["win_prob"].dropna().iloc[-1] if rider_data["win_prob"].notna().any() else 0,
        text=f"  Finished {actual_finish}{suffix}",
        showarrow=False,
        font=dict(color="#1A7FE8", size=11),
        xanchor="left",
    )

    track = rider_data["track"].iloc[0]
    fig.update_layout(
        title=f"Win Probability — {rider} | {class_val} | {year_val} | Rd {round_val} | Moto {moto_val} | {track}",
        xaxis=dict(title="Lap", dtick=1),
        yaxis=dict(
            title="Win Probability",
            range=[0, 1],
            tickformat=".0%",
        ),
        height=500, width=950,
        margin=dict(l=60, r=60, t=60, b=60),
        legend=dict(title=""),
        hovermode="x unified",
    )
    display(fig)

def update_wp(_):
    output_wp.clear_output(wait=True)
    with output_wp:
        plot_win_prob(
            year_sel_wp.value, class_sel_wp.value,
            round_sel_wp.value, moto_sel_wp.value,
            rider_sel_wp.value,
        )

btn_wp.on_click(update_wp)

display(widgets.VBox([
    widgets.HBox([year_sel_wp, class_sel_wp, round_sel_wp, moto_sel_wp]),
    rider_sel_wp,
    btn_wp,
    output_wp,
]))

update_wp(None)

  Win Probability Model — Logistic Regression (Model 2)
  No balanced class weighting (no 50/50 winner/loser assumption)

  Training set (2024): 16,217 observations
  Test set     (2025): 15,720 observations
  Positive rate train: 0.036
  Positive rate test:  0.038

  Metric                         Train       Test
  ---------------------------------------------
  ROC-AUC                       0.9904     0.9925
  Brier Score                   0.0135     0.0144
  Features: place, behind_time, frac_remaining + two interaction terms


  Feature Coefficients (scaled):



,Feature,Coefficient,Odds Ratio
1,place_x_frac,0.4499,1.5682
2,frac_remaining,-0.2304,0.7942
3,behind_x_frac,-4.1137,0.0163
4,place,-5.5318,0.0040
5,behind_time,-9.1646,0.0001


Model Comparison:


,Model,Train AUC,Test AUC,Train Brier,Test Brier
0,Model 1 — Logistic Regression,0.9909,0.9918,0.0478,0.0507
1,Model 2 — LR w/ No class weighting,0.9904,0.9925,0.0135,0.0144



  Notes: Interpreting the metrics above

  Positive rate
    Share of lap-rows belonging to the eventual moto winner.
    Roughly tracks 1 / (avg field size), so train and test rates
    being close is a good sanity check on the target.

  Odds ratio (exp(coefficient))
    Features are standardized, so this is the multiplicative
    change in win odds per 1-SD increase in that feature.
    OR > 1 -> higher win odds, OR < 1 -> lower, OR = 1 -> no effect.

  ROC-AUC
    P(a random winner-lap ranks above a random non-winner-lap).
    0.5 = random, 1.0 = perfect ranking. Only cares about relative
    order, so it's largely insensitive to class weighting -- expect
    a similar value to Model 1.

  Brier score
    Mean squared error between predicted prob and actual 0/1
    outcome. Lower is better, 0 = perfect. Baseline ~
    positive_rate * (1 - positive_rate). With no class weighting,
    predictions are fit to the true ~5-8% win rate directly, so
    this should improve noticeably vs. 

In [44]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import plotly.graph_objects as go
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.calibration import calibration_curve
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings("ignore")

# ── Build model base ──────────────────────────────────────────────────────────
model_base = (
    df.dropna(subset=["lap", "lap_time", "place",
                      "finish_position", "race_id"])
    .copy()
)
model_base["lap"] = model_base["lap"].astype(float)
model_base["place"] = model_base["place"].astype(float)
model_base["behind_time"] = model_base["behind_time"].fillna(0).astype(float)
model_base["finish_position"] = model_base["finish_position"].astype(float)
model_base["year"] = model_base["year"].astype(str)

# ── Target ────────────────────────────────────────────────────────────────────
model_base["won_moto"] = (model_base["finish_position"] == 1).astype(int)

# ── Fraction of race remaining ────────────────────────────────────────────────
moto_max_laps = (
    model_base.groupby("race_id", observed=True)["lap"]
    .max()
    .rename("max_lap")
    .reset_index()
)
model_base = model_base.merge(moto_max_laps, on="race_id", how="left")
model_base["frac_remaining"] = (
        (model_base["max_lap"] - model_base["lap"]) / model_base["max_lap"]
)

# ── Interaction terms ─────────────────────────────────────────────────────────
model_base["place_x_frac"] = model_base["place"] * model_base["frac_remaining"]
model_base["behind_x_frac"] = model_base["behind_time"] * model_base["frac_remaining"]

# ── Feature set (Model 3: behind_time / behind_x_frac dropped) ────────────────
FEATURES = [
    "place",
    "frac_remaining",
    "place_x_frac",
]

# ── Train / test split ────────────────────────────────────────────────────────
train = model_base[
    model_base["year"] == "2024"
    ].dropna(subset=FEATURES + ["behind_time", "behind_x_frac", "won_moto"]).copy()

test = model_base[
    model_base["year"] == "2025"
    ].dropna(subset=FEATURES + ["behind_time", "behind_x_frac", "won_moto"]).copy()

# ── Collinearity check (dropped features vs. retained features) ───────────────
corr_df = train[["place", "place_x_frac", "behind_time", "behind_x_frac"]].corr()

print("=" * 60)
print("  Collinearity check (train, 2024)")
print("=" * 60)
print("\n  Correlation: behind_time vs place")
print(f"    {corr_df.loc['behind_time', 'place']:.3f}")
print("\n  Correlation: behind_x_frac vs place_x_frac")
print(f"    {corr_df.loc['behind_x_frac', 'place_x_frac']:.3f}")
print("\n  Note: high correlation (|r| > ~0.7) suggests the dropped")
print("  features may be largely redundant with the retained ones.\n")

X_train = train[FEATURES].values
y_train = train["won_moto"].values
X_test = test[FEATURES].values
y_test = test["won_moto"].values

# ── Scale features ────────────────────────────────────────────────────────────
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ── Fit model (no class weighting) ───────────────────────────────────────────
model = LogisticRegression(
    max_iter=1000,
    random_state=42,
)
model.fit(X_train, y_train)

# ── Predictions ───────────────────────────────────────────────────────────────
train["win_prob"] = model.predict_proba(X_train)[:, 1]
test["win_prob"] = model.predict_proba(X_test)[:, 1]

# Write win_prob back to model_base
model_base = model_base.merge(
    pd.concat([
        train[["race_id", "name", "lap", "win_prob"]],
        test[["race_id", "name", "lap", "win_prob"]],
    ]),
    on=["race_id", "name", "lap"],
    how="left"
)

# ── Metrics ───────────────────────────────────────────────────────────────────
train_auc = roc_auc_score(y_train, train["win_prob"])
test_auc = roc_auc_score(y_test, test["win_prob"])
train_brier = brier_score_loss(y_train, train["win_prob"])
test_brier = brier_score_loss(y_test, test["win_prob"])

# ── Coefficient table ─────────────────────────────────────────────────────────
coef_df = pd.DataFrame({
    "Feature": FEATURES,
    "Coefficient": model.coef_[0].round(4),
    "Odds Ratio": np.exp(model.coef_[0]).round(4),
}).sort_values("Coefficient", ascending=False).reset_index(drop=True)
coef_df.index += 1

# ── Log results for comparison ────────────────────────────────────────────────
if "model_results" not in globals():
    model_results = {}
model_results["Model 3 — LR w/o behind time"] = {
    "Train AUC":   round(train_auc, 4),
    "Test AUC":    round(test_auc, 4),
    "Train Brier": round(train_brier, 4),
    "Test Brier":  round(test_brier, 4),
}

# ── Calibration curve (test set) ──────────────────────────────────────────────
frac_pos, mean_pred = calibration_curve(
    y_test, test["win_prob"], n_bins=10, strategy="quantile"
)

# ── Print model summary ───────────────────────────────────────────────────────
print("=" * 60)
print("  Win Probability Model — Logistic Regression (Model 3)")
print("  No class weighting | behind_time / behind_x_frac dropped")
print("=" * 60)
print(f"\n  Training set (2024): {len(train):,} observations")
print(f"  Test set     (2025): {len(test):,} observations")
print(f"  Positive rate train: {y_train.mean():.3f}")
print(f"  Positive rate test:  {y_test.mean():.3f}")
print(f"\n  {'Metric':<25} {'Train':>10} {'Test':>10}")
print(f"  {'-' * 45}")
print(f"  {'ROC-AUC':<25} {train_auc:>10.4f} {test_auc:>10.4f}")
print(f"  {'Brier Score':<25} {train_brier:>10.4f} {test_brier:>10.4f}")
print(f"  Features: place, frac_remaining, place_x_frac\n")
print("\n  Feature Coefficients (scaled):\n")
display(coef_df)

# ── Calibration plot ──────────────────────────────────────────────────────────
fig_cal = go.Figure()
fig_cal.add_trace(go.Scatter(
    x=mean_pred, y=frac_pos,
    mode="lines+markers",
    name="Model 3",
    line=dict(color="#1A7FE8", width=2),
    marker=dict(size=8),
    hovertemplate=(
        "Predicted: %{x:.2f}<br>"
        "Actual: %{y:.2f}<extra></extra>"
    ),
))
fig_cal.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1],
    mode="lines",
    name="Perfect calibration",
    line=dict(color="grey", width=1, dash="dash"),
))
fig_cal.update_layout(
    title="Calibration Plot — Test Set (2025) — Model 3",
    xaxis=dict(title="Mean Predicted Probability", range=[0, 1]),
    yaxis=dict(title="Fraction of Positives", range=[0, 1]),
    height=450, width=700,
    margin=dict(l=60, r=60, t=60, b=60),
    legend=dict(title=""),
)
display(fig_cal)

print("Model Comparison:")
comparison_df = pd.DataFrame(model_results).T
comparison_df.index.name = "Model"
comparison_df = comparison_df.reset_index()
display(comparison_df.style.hide(axis="index"))

# ── Notes: Interpreting the results above ─────────────────────────────────────
print("\n" + "=" * 60)
print("  Notes: Interpreting the ablation")
print("=" * 60)
print("""
  What changed vs Model 2
    behind_time and behind_x_frac removed. Same train/test split,
    same scaling, no class weighting. Only the feature set differs.

    Compare ROC-AUC and Brier (test set) directly against Model 2's.
    If both are roughly unchanged, the dropped features
    were largely redundant -- Model 3 is the better choice on
    parsimony grounds alone. If Brier or AUC noticeably worsens,
    gap-to-leader was carrying real signal beyond 'place', and
    Model 2's feature set is worth keeping despite the extra
    complexity.

  Collinearity check
    High correlation between behind_time/behind_x_frac and their
    place-based counterparts is consistent with -- but not proof
    of -- redundancy. The ablation comparison is the direct test;
    the correlation numbers just help explain *why* the ablation
    came out the way it did.
""")

# ── Interactive moto viewer ───────────────────────────────────────────────────
viewer_base = model_base.copy()
viewer_base["round"] = viewer_base["round"].astype(str)
viewer_base["moto"] = viewer_base["moto"].astype(str)
viewer_base["year"] = viewer_base["year"].astype(str)

viewer_years = sorted(viewer_base["year"].unique())
viewer_rounds = sorted(viewer_base["round"].unique(), key=lambda x: int(x))
viewer_motos = sorted(viewer_base["moto"].unique(), key=lambda x: int(x))

year_sel_wp = widgets.Dropdown(
    options=viewer_years, value=viewer_years[-1],
    description="Year:", layout=widgets.Layout(width="200px")
)
class_sel_wp = widgets.Dropdown(
    options=["450", "250", "WMX"], value="450",
    description="Class:", layout=widgets.Layout(width="200px")
)
round_sel_wp = widgets.Dropdown(
    options=viewer_rounds, value=viewer_rounds[0],
    description="Round:", layout=widgets.Layout(width="200px")
)
moto_sel_wp = widgets.Dropdown(
    options=viewer_motos, value=viewer_motos[0],
    description="Moto:", layout=widgets.Layout(width="200px")
)
rider_sel_wp = widgets.Dropdown(
    options=[], value=None,
    description="Rider:", layout=widgets.Layout(width="280px")
)


def refresh_wp_riders(*args):
    mask = (
            (viewer_base["year"] == year_sel_wp.value) &
            (viewer_base["class_label"] == class_sel_wp.value) &
            (viewer_base["round"] == round_sel_wp.value) &
            (viewer_base["moto"] == moto_sel_wp.value)
    )
    riders = sorted(viewer_base[mask]["name"].dropna().unique().tolist())
    current = rider_sel_wp.value
    rider_sel_wp.options = riders
    rider_sel_wp.value = current if current in riders else (riders[0] if riders else None)


for w in [year_sel_wp, class_sel_wp, round_sel_wp, moto_sel_wp]:
    w.observe(refresh_wp_riders, names="value")

refresh_wp_riders()

btn_wp = widgets.Button(description="Update", button_style="primary")
output_wp = widgets.Output()


def plot_win_prob(year_val, class_val, round_val, moto_val, rider):
    if not rider:
        print("No riders found.")
        return

    mask = (
            (viewer_base["year"] == year_val) &
            (viewer_base["class_label"] == class_val) &
            (viewer_base["round"] == round_val) &
            (viewer_base["moto"] == moto_val)
    )
    moto_data = viewer_base[mask].copy()
    rider_data = moto_data[moto_data["name"] == rider].sort_values("lap")

    if rider_data.empty:
        print(f"No data for {rider}.")
        return

    fig = go.Figure()

    # Grey traces for all other riders
    for other in moto_data["name"].unique():
        if other == rider:
            continue
        od = moto_data[moto_data["name"] == other].sort_values("lap")
        if od["win_prob"].isna().all():
            continue
        fig.add_trace(go.Scatter(
            x=od["lap"], y=od["win_prob"],
            mode="lines",
            line=dict(color="lightgrey", width=1),
            showlegend=False,
            hoverinfo="skip",
        ))

    # Highlighted rider
    fig.add_trace(go.Scatter(
        x=rider_data["lap"],
        y=rider_data["win_prob"],
        mode="lines+markers",
        name=rider,
        line=dict(color="#1A7FE8", width=3),
        marker=dict(size=7),
        customdata=rider_data[[
            "place", "behind_time", "frac_remaining"
        ]].values,
        hovertemplate=(
            f"<b>{rider}</b><br>"
            "Lap: %{x}<br>"
            "Win prob: %{y:.1%}<br>"
            "Place: %{customdata[0]:.0f}<br>"
            "Gap to leader: %{customdata[1]:.2f}s<br>"
            "Race remaining: %{customdata[2]:.1%}"
            "<extra></extra>"
        ),
    ))

    # Actual finish annotation
    actual_finish = int(rider_data["finish_position"].iloc[0])
    suffix = (
        "st" if actual_finish == 1 else
        "nd" if actual_finish == 2 else
        "rd" if actual_finish == 3 else "th"
    )
    fig.add_annotation(
        x=rider_data["lap"].max(),
        y=rider_data["win_prob"].dropna().iloc[-1] if rider_data["win_prob"].notna().any() else 0,
        text=f"  Finished {actual_finish}{suffix}",
        showarrow=False,
        font=dict(color="#1A7FE8", size=11),
        xanchor="left",
    )

    track = rider_data["track"].iloc[0]
    fig.update_layout(
        title=f"Win Probability — {rider} | {class_val} | {year_val} | Rd {round_val} | Moto {moto_val} | {track}",
        xaxis=dict(title="Lap", dtick=1),
        yaxis=dict(
            title="Win Probability",
            range=[0, 1],
            tickformat=".0%",
        ),
        height=500, width=950,
        margin=dict(l=60, r=60, t=60, b=60),
        legend=dict(title=""),
        hovermode="x unified",
    )
    display(fig)


def update_wp(_):
    output_wp.clear_output(wait=True)
    with output_wp:
        plot_win_prob(
            year_sel_wp.value, class_sel_wp.value,
            round_sel_wp.value, moto_sel_wp.value,
            rider_sel_wp.value,
        )


btn_wp.on_click(update_wp)

display(widgets.VBox([
    widgets.HBox([year_sel_wp, class_sel_wp, round_sel_wp, moto_sel_wp]),
    rider_sel_wp,
    btn_wp,
    output_wp,
]))

update_wp(None)

  Collinearity check (train, 2024)

  Correlation: behind_time vs place
    0.548

  Correlation: behind_x_frac vs place_x_frac
    0.440

  Note: high correlation (|r| > ~0.7) suggests the dropped
  features may be largely redundant with the retained ones.

  Win Probability Model — Logistic Regression (Model 3)
  No class weighting | behind_time / behind_x_frac dropped

  Training set (2024): 16,217 observations
  Test set     (2025): 15,720 observations
  Positive rate train: 0.036
  Positive rate test:  0.038

  Metric                         Train       Test
  ---------------------------------------------
  ROC-AUC                       0.9863     0.9931
  Brier Score                   0.0149     0.0154
  Features: place, frac_remaining, place_x_frac


  Feature Coefficients (scaled):



,Feature,Coefficient,Odds Ratio
1,place_x_frac,0.5375,1.7117
2,frac_remaining,-0.0371,0.9636
3,place,-12.2284,0.0000


Model Comparison:


Model,Train AUC,Test AUC,Train Brier,Test Brier
Model 1 — Logistic Regression,0.990900,0.991800,0.047800,0.050700
Model 2 — LR w/ No class weighting,0.990400,0.992500,0.013500,0.014400
Model 3 — LR w/o behind time,0.986300,0.993100,0.014900,0.015400



  Notes: Interpreting the ablation

  What changed vs Model 2
    behind_time and behind_x_frac removed. Same train/test split,
    same scaling, no class weighting. Only the feature set differs.

    Compare ROC-AUC and Brier (test set) directly against Model 2's.
    If both are roughly unchanged, the dropped features
    were largely redundant -- Model 3 is the better choice on
    parsimony grounds alone. If Brier or AUC noticeably worsens,
    gap-to-leader was carrying real signal beyond 'place', and
    Model 2's feature set is worth keeping despite the extra
    complexity.

  Collinearity check
    High correlation between behind_time/behind_x_frac and their
    place-based counterparts is consistent with -- but not proof
    of -- redundancy. The ablation comparison is the direct test;
    the correlation numbers just help explain *why* the ablation
    came out the way it did.



In [46]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import plotly.graph_objects as go
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")

# ── Build model base ──────────────────────────────────────────────────────────
model_base = (
    df.dropna(subset=["lap", "lap_time", "place",
                       "finish_position", "race_id"])
    .copy()
)
model_base["lap"]             = model_base["lap"].astype(float)
model_base["place"]           = model_base["place"].astype(float)
model_base["behind_time"]     = model_base["behind_time"].fillna(0).astype(float)
model_base["finish_position"] = model_base["finish_position"].astype(float)
model_base["year"]            = model_base["year"].astype(str)

# ── Target ────────────────────────────────────────────────────────────────────
model_base["won_moto"] = (model_base["finish_position"] == 1).astype(int)

# ── Fraction of race remaining ────────────────────────────────────────────────
moto_max_laps = (
    model_base.groupby("race_id", observed=True)["lap"]
    .max()
    .rename("max_lap")
    .reset_index()
)
model_base = model_base.merge(moto_max_laps, on="race_id", how="left")
model_base["frac_remaining"] = (
    (model_base["max_lap"] - model_base["lap"]) / model_base["max_lap"]
)

# ── Interaction terms ─────────────────────────────────────────────────────────
model_base["place_x_frac"]  = model_base["place"]       * model_base["frac_remaining"]
model_base["behind_x_frac"] = model_base["behind_time"] * model_base["frac_remaining"]

# ── Feature set ───────────────────────────────────────────────────────────────
FEATURES = [
    "place",
    "behind_time",
    "frac_remaining",
    "place_x_frac",
    "behind_x_frac",
]

# ── Train / test split ────────────────────────────────────────────────────────
train = model_base[
    model_base["year"] == "2024"
].dropna(subset=FEATURES + ["won_moto"]).copy()

test = model_base[
    model_base["year"] == "2025"
].dropna(subset=FEATURES + ["won_moto"]).copy()

X_train = train[FEATURES].values
y_train = train["won_moto"].values
X_test  = test[FEATURES].values
y_test  = test["won_moto"].values

# ── Scale features ────────────────────────────────────────────────────────────
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

# ── Base logistic regression ──────────────────────────────────────────────────
base_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
)

# ── Platt scaling via CalibratedClassifierCV ──────────────────────────────────
# cv=5: uses 5-fold cross-validation on training data to fit the calibration
# method="sigmoid": Platt scaling — fits a sigmoid on top of the raw scores
model = CalibratedClassifierCV(
    base_model,
    method="sigmoid",
    cv=5,
)
model.fit(X_train, y_train)

# ── Predictions ───────────────────────────────────────────────────────────────
train["win_prob"] = model.predict_proba(X_train)[:, 1]
test["win_prob"]  = model.predict_proba(X_test)[:, 1]

# Write win_prob back to model_base
model_base = model_base.merge(
    pd.concat([
        train[["race_id", "name", "lap", "win_prob"]],
        test[["race_id", "name", "lap", "win_prob"]],
    ]),
    on=["race_id", "name", "lap"],
    how="left"
)

# ── Metrics ───────────────────────────────────────────────────────────────────
train_auc   = roc_auc_score(y_train, train["win_prob"])
test_auc    = roc_auc_score(y_test,  test["win_prob"])
train_brier = brier_score_loss(y_train, train["win_prob"])
test_brier  = brier_score_loss(y_test,  test["win_prob"])

# ── Log results for comparison ────────────────────────────────────────────────
if "model_results" not in globals():
    model_results = {}
model_results["Model 4 — LR w/ Platt scaling"] = {
    "Train AUC":   round(train_auc, 4),
    "Test AUC":    round(test_auc, 4),
    "Train Brier": round(train_brier, 4),
    "Test Brier":  round(test_brier, 4),
}

# ── Calibration curve (test set) ──────────────────────────────────────────────
frac_pos, mean_pred = calibration_curve(
    y_test, test["win_prob"], n_bins=10, strategy="quantile"
)

# ── Print model summary ───────────────────────────────────────────────────────
print("=" * 60)
print("  Win Probability Model — Logistic Regression + Platt Scaling")
print("  (Model 4)")
print("=" * 60)
print(f"\n  Training set (2024): {len(train):,} observations")
print(f"  Test set     (2025): {len(test):,} observations")
print(f"  Positive rate train: {y_train.mean():.3f}")
print(f"  Positive rate test:  {y_test.mean():.3f}")
print(f"\n  {'Metric':<25} {'Train':>10} {'Test':>10}")
print(f"  {'-'*45}")
print(f"  {'ROC-AUC':<25} {train_auc:>10.4f} {test_auc:>10.4f}")
print(f"  {'Brier Score':<25} {train_brier:>10.4f} {test_brier:>10.4f}")
print(f"  Note: Platt scaling (cv=5) applied to remap probabilities to full 0-1 range")
print(f"  Features: place, behind_time, frac_remaining + two interaction terms\n")

# ── Calibration plot ──────────────────────────────────────────────────────────
fig_cal = go.Figure()
fig_cal.add_trace(go.Scatter(
    x=mean_pred, y=frac_pos,
    mode="lines+markers",
    name="Model 4",
    line=dict(color="#1A7FE8", width=2),
    marker=dict(size=8),
    hovertemplate=(
        "Predicted: %{x:.2f}<br>"
        "Actual: %{y:.2f}<extra></extra>"
    ),
))
fig_cal.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1],
    mode="lines",
    name="Perfect calibration",
    line=dict(color="grey", width=1, dash="dash"),
))
fig_cal.update_layout(
    title="Calibration Plot — Test Set (2025) — Model 4",
    xaxis=dict(title="Mean Predicted Probability", range=[0, 1]),
    yaxis=dict(title="Fraction of Positives",      range=[0, 1]),
    height=450, width=700,
    margin=dict(l=60, r=60, t=60, b=60),
    legend=dict(title=""),
)
display(fig_cal)

print("Model Comparison:")
comparison_df = pd.DataFrame(model_results).T
comparison_df.index.name = "Model"
comparison_df = comparison_df.reset_index()
display(comparison_df.style.hide(axis="index"))

# ── Notes: Interpreting the metrics above ─────────────────────────────────────
print("\n" + "=" * 60)
print("  Notes: Interpreting the metrics above")
print("=" * 60)
print("""
  Positive rate
    Share of lap-rows belonging to the eventual moto winner.
    Roughly tracks 1 / (avg field size), so train and test rates
    being close is a good sanity check on the target.

  ROC-AUC
    P(a random winner-lap ranks above a random non-winner-lap).
    0.5 = random, 1.0 = perfect ranking. Platt scaling is a
    monotonic transform (sigmoid), so for a single fit it wouldn't
    change AUC at all vs. the underlying logistic regression. The
    cv=5 wrapper here actually fits 5 separate base models on
    different folds and averages them, so any AUC difference vs.
    Model 2 reflects that mild ensembling, not the calibration step
    itself.

  Brier score
    Mean squared error between predicted prob and actual 0/1
    outcome. Lower is better, 0 = perfect. This is the metric Platt
    scaling directly targets -- it fits a sigmoid remapping raw
    scores to probabilities that better match observed frequencies.
    Model 2 (no class weighting) was already reasonably well
    calibrated, so the interesting question here is whether Platt
    scaling improves Brier further on top of an already-decent
    baseline, or whether it's mostly redundant in this case.

  Calibration plot
    Test predictions binned into 10 quantile groups; mean predicted
    prob vs. actual win fraction per bin. Above the diagonal =
    underconfident, below = overconfident. Platt scaling has the
    biggest effect when the base model is poorly calibrated (e.g.
    Model 1 with class_weight="balanced"). Applied to Model 2's
    already-reasonable base, expect this to look similar to Model
    2's plot -- a small further improvement here would suggest
    Platt scaling is worth the extra complexity; little change
    suggests it isn't adding much.

  Coefficients (not shown this chunk)
    CalibratedClassifierCV with cv=5 fits 5 separate base logistic
    regressions (one per fold) plus 5 sigmoid calibration params --
    there's no single coef_ to display the way earlier models had.
    The underlying coefficients are accessible via
    model.calibrated_classifiers_[i].estimator.coef_ for each fold
    if needed, but interpreting 5 separate sets is less clean than
    Models 1-3's single table.

  Platt scaling
    A calibration technique: fit a 1D logistic regression (sigmoid)
    mapping a model's raw outputs to probabilities that better match
    true outcome frequencies -- P_calibrated = 1 / (1 + exp(A*s + B)),
    where s is the base model's score and A, B are fit on held-out
    data. Originally developed for SVMs (which don't output
    probabilities naturally), but used generally as a post-processing
    fix for miscalibrated probability outputs. cv=5 here means: split
    training data into 5 folds, fit the base model on each fold's 80%,
    fit the sigmoid on the held-out 20%, then average all 5 calibrated
    models' predictions at inference time -- this avoids calibrating
    on the same data the base model was fit on, which would be
    overly optimistic.

  Models so far
    Model 1: LR, class_weight="balanced"          (poor calibration)
    Model 2: LR, no class weighting                (good baseline)
    Model 3: LR, no class weighting, place-only features
    Model 4: Model 2's setup + Platt scaling (cv=5)
    Model 4 vs Model 2 is the cleanest comparison for isolating
    Platt scaling's effect, since both use the same features and
    no class weighting -- only the calibration wrapper differs.
""")

# ── Interactive moto viewer ───────────────────────────────────────────────────
viewer_base = model_base.copy()
viewer_base["round"] = viewer_base["round"].astype(str)
viewer_base["moto"]  = viewer_base["moto"].astype(str)
viewer_base["year"]  = viewer_base["year"].astype(str)

viewer_years  = sorted(viewer_base["year"].unique())
viewer_rounds = sorted(viewer_base["round"].unique(), key=lambda x: int(x))
viewer_motos  = sorted(viewer_base["moto"].unique(),  key=lambda x: int(x))

year_sel_wp = widgets.Dropdown(
    options=viewer_years, value=viewer_years[-1],
    description="Year:", layout=widgets.Layout(width="200px")
)
class_sel_wp = widgets.Dropdown(
    options=["450", "250", "WMX"], value="450",
    description="Class:", layout=widgets.Layout(width="200px")
)
round_sel_wp = widgets.Dropdown(
    options=viewer_rounds, value=viewer_rounds[0],
    description="Round:", layout=widgets.Layout(width="200px")
)
moto_sel_wp = widgets.Dropdown(
    options=viewer_motos, value=viewer_motos[0],
    description="Moto:", layout=widgets.Layout(width="200px")
)
rider_sel_wp = widgets.Dropdown(
    options=[], value=None,
    description="Rider:", layout=widgets.Layout(width="280px")
)

def refresh_wp_riders(*args):
    mask = (
        (viewer_base["year"]        == year_sel_wp.value) &
        (viewer_base["class_label"] == class_sel_wp.value) &
        (viewer_base["round"]       == round_sel_wp.value) &
        (viewer_base["moto"]        == moto_sel_wp.value)
    )
    riders  = sorted(viewer_base[mask]["name"].dropna().unique().tolist())
    current = rider_sel_wp.value
    rider_sel_wp.options = riders
    rider_sel_wp.value   = current if current in riders else (riders[0] if riders else None)

for w in [year_sel_wp, class_sel_wp, round_sel_wp, moto_sel_wp]:
    w.observe(refresh_wp_riders, names="value")

refresh_wp_riders()

btn_wp    = widgets.Button(description="Update", button_style="primary")
output_wp = widgets.Output()

def plot_win_prob(year_val, class_val, round_val, moto_val, rider):
    if not rider:
        print("No riders found.")
        return

    mask = (
        (viewer_base["year"]        == year_val) &
        (viewer_base["class_label"] == class_val) &
        (viewer_base["round"]       == round_val) &
        (viewer_base["moto"]        == moto_val)
    )
    moto_data  = viewer_base[mask].copy()
    rider_data = moto_data[moto_data["name"] == rider].sort_values("lap")

    if rider_data.empty:
        print(f"No data for {rider}.")
        return

    fig = go.Figure()

    # Grey traces for all other riders
    for other in moto_data["name"].unique():
        if other == rider:
            continue
        od = moto_data[moto_data["name"] == other].sort_values("lap")
        if od["win_prob"].isna().all():
            continue
        fig.add_trace(go.Scatter(
            x=od["lap"], y=od["win_prob"],
            mode="lines",
            line=dict(color="lightgrey", width=1),
            showlegend=False,
            hoverinfo="skip",
        ))

    # Highlighted rider
    fig.add_trace(go.Scatter(
        x=rider_data["lap"],
        y=rider_data["win_prob"],
        mode="lines+markers",
        name=rider,
        line=dict(color="#1A7FE8", width=3),
        marker=dict(size=7),
        customdata=rider_data[[
            "place", "behind_time", "frac_remaining"
        ]].values,
        hovertemplate=(
            f"<b>{rider}</b><br>"
            "Lap: %{x}<br>"
            "Win prob: %{y:.1%}<br>"
            "Place: %{customdata[0]:.0f}<br>"
            "Gap to leader: %{customdata[1]:.2f}s<br>"
            "Race remaining: %{customdata[2]:.1%}"
            "<extra></extra>"
        ),
    ))

    # Actual finish annotation
    actual_finish = int(rider_data["finish_position"].iloc[0])
    suffix = (
        "st" if actual_finish == 1 else
        "nd" if actual_finish == 2 else
        "rd" if actual_finish == 3 else "th"
    )
    fig.add_annotation(
        x=rider_data["lap"].max(),
        y=rider_data["win_prob"].dropna().iloc[-1] if rider_data["win_prob"].notna().any() else 0,
        text=f"  Finished {actual_finish}{suffix}",
        showarrow=False,
        font=dict(color="#1A7FE8", size=11),
        xanchor="left",
    )

    track = rider_data["track"].iloc[0]
    fig.update_layout(
        title=f"Win Probability — {rider} | {class_val} | {year_val} | Rd {round_val} | Moto {moto_val} | {track}",
        xaxis=dict(title="Lap", dtick=1),
        yaxis=dict(
            title="Win Probability",
            range=[0, 1],
            tickformat=".0%",
        ),
        height=500, width=950,
        margin=dict(l=60, r=60, t=60, b=60),
        legend=dict(title=""),
        hovermode="x unified",
    )
    display(fig)

def update_wp(_):
    output_wp.clear_output(wait=True)
    with output_wp:
        plot_win_prob(
            year_sel_wp.value, class_sel_wp.value,
            round_sel_wp.value, moto_sel_wp.value,
            rider_sel_wp.value,
        )

btn_wp.on_click(update_wp)

display(widgets.VBox([
    widgets.HBox([year_sel_wp, class_sel_wp, round_sel_wp, moto_sel_wp]),
    rider_sel_wp,
    btn_wp,
    output_wp,
]))

update_wp(None)

  Win Probability Model — Logistic Regression + Platt Scaling
  (Model 4)

  Training set (2024): 16,217 observations
  Test set     (2025): 15,720 observations
  Positive rate train: 0.036
  Positive rate test:  0.038

  Metric                         Train       Test
  ---------------------------------------------
  ROC-AUC                       0.9900     0.9925
  Brier Score                   0.0132     0.0141
  Note: Platt scaling (cv=5) applied to remap probabilities to full 0-1 range
  Features: place, behind_time, frac_remaining + two interaction terms



Model Comparison:


Model,Train AUC,Test AUC,Train Brier,Test Brier
Model 1 — Logistic Regression,0.990900,0.991800,0.047800,0.050700
Model 2 — LR w/ No class weighting,0.990400,0.992500,0.013500,0.014400
Model 3 — LR w/o behind time,0.986300,0.993100,0.014900,0.015400
Model 4 — LR w/ Platt scaling,0.990000,0.992500,0.013200,0.014100



  Notes: Interpreting the metrics above

  Positive rate
    Share of lap-rows belonging to the eventual moto winner.
    Roughly tracks 1 / (avg field size), so train and test rates
    being close is a good sanity check on the target.

  ROC-AUC
    P(a random winner-lap ranks above a random non-winner-lap).
    0.5 = random, 1.0 = perfect ranking. Platt scaling is a
    monotonic transform (sigmoid), so for a single fit it wouldn't
    change AUC at all vs. the underlying logistic regression. The
    cv=5 wrapper here actually fits 5 separate base models on
    different folds and averages them, so any AUC difference vs.
    Model 2 reflects that mild ensembling, not the calibration step
    itself.

  Brier score
    Mean squared error between predicted prob and actual 0/1
    outcome. Lower is better, 0 = perfect. This is the metric Platt
    scaling directly targets -- it fits a sigmoid remapping raw
    scores to probabilities that better match observed frequencies.
    Model 2

In [47]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import plotly.graph_objects as go
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.calibration import calibration_curve
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
import xgboost as xgb
import warnings

warnings.filterwarnings("ignore")

# ── Build model base ──────────────────────────────────────────────────────────
model_base = (
    df.dropna(subset=["lap", "lap_time", "place",
                      "finish_position", "race_id"])
    .copy()
)
model_base["lap"] = model_base["lap"].astype(float)
model_base["place"] = model_base["place"].astype(float)
model_base["behind_time"] = model_base["behind_time"].fillna(0).astype(float)
model_base["finish_position"] = model_base["finish_position"].astype(float)
model_base["year"] = model_base["year"].astype(str)

# ── Target ────────────────────────────────────────────────────────────────────
model_base["won_moto"] = (model_base["finish_position"] == 1).astype(int)

# ── Fraction of race remaining ────────────────────────────────────────────────
moto_max_laps = (
    model_base.groupby("race_id", observed=True)["lap"]
    .max()
    .rename("max_lap")
    .reset_index()
)
model_base = model_base.merge(moto_max_laps, on="race_id", how="left")
model_base["frac_remaining"] = (
        (model_base["max_lap"] - model_base["lap"]) / model_base["max_lap"]
)

# ── Interaction terms ─────────────────────────────────────────────────────────
model_base["place_x_frac"] = model_base["place"] * model_base["frac_remaining"]
model_base["behind_x_frac"] = model_base["behind_time"] * model_base["frac_remaining"]

# ── Feature set ───────────────────────────────────────────────────────────────
FEATURES = [
    "place",
    "behind_time",
    "frac_remaining",
    "place_x_frac",
    "behind_x_frac",
]

# ── Train / test split ────────────────────────────────────────────────────────
train = model_base[
    model_base["year"] == "2024"
    ].dropna(subset=FEATURES + ["won_moto"]).copy()

test = model_base[
    model_base["year"] == "2025"
    ].dropna(subset=FEATURES + ["won_moto"]).copy()

X_train = train[FEATURES].values
y_train = train["won_moto"].values
X_test = test[FEATURES].values
y_test = test["won_moto"].values

# No scaling needed for XGBoost — tree models are scale invariant
# but we keep it for consistency with prior models
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

# ── Fit XGBoost (no class weighting) ─────────────────────────────────────────
# scale_pos_weight left at default (1) -- no reweighting of the loss.
# Predicted probabilities should track the true ~5-8% base rate directly,
# same logic as Model 1 -> Model 2's class_weight="balanced" removal.
neg_pos_ratio = (y_train == 0).sum() / (y_train == 1).sum()  # printed for reference only

model_xgb = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=4,  # shallow trees — reduces overfitting
    learning_rate=0.05,  # slow learning — reduces overfitting
    subsample=0.8,  # row subsampling — reduces overfitting
    colsample_bytree=0.8,  # feature subsampling — reduces overfitting
    eval_metric="auc",
    random_state=42,
    verbosity=0,
)

# ── Fit with early stopping to quantify overfitting ──────────────────────────
model_xgb.fit(
    X_train_sc, y_train,
    eval_set=[(X_train_sc, y_train), (X_test_sc, y_test)],
    verbose=False,
)

# ── Extract learning curves ───────────────────────────────────────────────────
results = model_xgb.evals_result()
train_auc_curve = results["validation_0"]["auc"]
test_auc_curve = results["validation_1"]["auc"]
n_estimators = list(range(1, len(train_auc_curve) + 1))

# Best test AUC and at which tree
best_test_auc = max(test_auc_curve)
best_tree = test_auc_curve.index(best_test_auc) + 1
final_gap = train_auc_curve[-1] - test_auc_curve[-1]
best_gap = train_auc_curve[best_tree - 1] - best_test_auc

# ── Predictions ───────────────────────────────────────────────────────────────
train["win_prob"] = model_xgb.predict_proba(X_train_sc)[:, 1]
test["win_prob"] = model_xgb.predict_proba(X_test_sc)[:, 1]

# Write win_prob back to model_base
model_base = model_base.merge(
    pd.concat([
        train[["race_id", "name", "lap", "win_prob"]],
        test[["race_id", "name", "lap", "win_prob"]],
    ]),
    on=["race_id", "name", "lap"],
    how="left"
)

# ── Metrics ───────────────────────────────────────────────────────────────────
train_auc = roc_auc_score(y_train, train["win_prob"])
test_auc = roc_auc_score(y_test, test["win_prob"])
train_brier = brier_score_loss(y_train, train["win_prob"])
test_brier = brier_score_loss(y_test, test["win_prob"])

# ── Feature importance ────────────────────────────────────────────────────────
importance_df = pd.DataFrame({
    "Feature": FEATURES,
    "Importance": model_xgb.feature_importances_,
}).sort_values("Importance", ascending=False).reset_index(drop=True)
importance_df["Importance"] = importance_df["Importance"].round(4)
importance_df.index += 1

# ── Log results for comparison ────────────────────────────────────────────────
if "model_results" not in globals():
    model_results = {}
model_results["Model 5 — XGBoost"] = {
    "Train AUC": round(train_auc, 4),
    "Test AUC": round(test_auc, 4),
    "Train Brier": round(train_brier, 4),
    "Test Brier": round(test_brier, 4),
}

# ── Calibration curve ─────────────────────────────────────────────────────────
frac_pos, mean_pred = calibration_curve(
    y_test, test["win_prob"], n_bins=10, strategy="quantile"
)

# ── Print model summary ───────────────────────────────────────────────────────
print("=" * 60)
print("  Win Probability Model — XGBoost (Model 5)")
print("=" * 60)
print(f"\n  Training set (2024): {len(train):,} observations")
print(f"  Test set     (2025): {len(test):,} observations")
print(f"  Positive rate train: {y_train.mean():.3f}")
print(f"  Positive rate test:  {y_test.mean():.3f}")
print(f"\n  {'Metric':<25} {'Train':>10} {'Test':>10}")
print(f"  {'-' * 45}")
print(f"  {'ROC-AUC':<25} {train_auc:>10.4f} {test_auc:>10.4f}")
print(f"  {'Brier Score':<25} {train_brier:>10.4f} {test_brier:>10.4f}")
print(f"\n  Overfitting diagnostics:")
print(f"  Best test AUC: {best_test_auc:.4f} at tree {best_tree} of {len(train_auc_curve)}")
print(f"  AUC gap at best tree:  {best_gap:.4f}  (train - test)")
print(f"  AUC gap at final tree: {final_gap:.4f}  (train - test)")
print(f"  Note: gap > 0.01 suggests meaningful overfitting\n")
print("\n  Feature Importances:\n")
display(importance_df)

# ── Learning curve plot ───────────────────────────────────────────────────────
fig_lc = go.Figure()
fig_lc.add_trace(go.Scatter(
    x=n_estimators, y=train_auc_curve,
    mode="lines",
    name="Train AUC",
    line=dict(color="#1A7FE8", width=2),
))
fig_lc.add_trace(go.Scatter(
    x=n_estimators, y=test_auc_curve,
    mode="lines",
    name="Test AUC",
    line=dict(color="#E8641A", width=2),
))
fig_lc.add_vline(
    x=best_tree,
    line_width=1, line_dash="dash", line_color="grey",
    annotation_text=f"Best test AUC: {best_test_auc:.4f} (tree {best_tree})",
    annotation_position="top right",
)
fig_lc.update_layout(
    title="Learning Curve — XGBoost (Model 5) | Train vs Test AUC by Number of Trees",
    xaxis=dict(title="Number of Trees"),
    yaxis=dict(title="ROC-AUC", range=[0.95, 1.0]),
    height=450, width=950,
    margin=dict(l=60, r=60, t=60, b=60),
    legend=dict(title=""),
    hovermode="x unified",
)
display(fig_lc)

# ── Calibration plot ──────────────────────────────────────────────────────────
fig_cal = go.Figure()
fig_cal.add_trace(go.Scatter(
    x=mean_pred, y=frac_pos,
    mode="lines+markers",
    name="Model 5",
    line=dict(color="#1A7FE8", width=2),
    marker=dict(size=8),
    hovertemplate=(
        "Predicted: %{x:.2f}<br>"
        "Actual: %{y:.2f}<extra></extra>"
    ),
))
fig_cal.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1],
    mode="lines",
    name="Perfect calibration",
    line=dict(color="grey", width=1, dash="dash"),
))
fig_cal.update_layout(
    title="Calibration Plot — Test Set (2025) — Model 5",
    xaxis=dict(title="Mean Predicted Probability", range=[0, 1]),
    yaxis=dict(title="Fraction of Positives", range=[0, 1]),
    height=450, width=700,
    margin=dict(l=60, r=60, t=60, b=60),
    legend=dict(title=""),
)
display(fig_cal)

print("Model Comparison:")
comparison_df = pd.DataFrame(model_results).T
comparison_df.index.name = "Model"
comparison_df = comparison_df.reset_index()
display(comparison_df.style.hide(axis="index"))

# ── Notes: interpreting the metrics above ─────────────────────────────────────
print("\n" + "=" * 60)
print("  Notes: Interpreting the metrics above")
print("=" * 60)
print("""
  Positive rate
    Share of lap-rows belonging to the eventual moto winner.
    Roughly tracks 1 / (avg field size), so train and test rates
    being close is a good sanity check on the target.

  scale_pos_weight (not used in this model)
    XGBoost's scale_pos_weight reweights the loss so winners and
    non-winners contribute equally -- same effect as
    class_weight="balanced" in Model 1. As shown in the Model 1 ->
    Model 2 comparison, that kind of reweighting leaves AUC mostly
    unchanged but pushes predicted probabilities away from the true
    ~5-8% base rate, hurting Brier and calibration. This model
    leaves scale_pos_weight at its default (1, no reweighting), so
    win_prob should track the real base rate directly. The neg/pos
    ratio is printed above for reference only -- it isn't applied
    anywhere.

  ROC-AUC
    P(a random winner-lap ranks above a random non-winner-lap).
    0.5 = random, 1.0 = perfect ranking. Class reweighting mostly
    affects probability *scale*, not ranking, so this should be in
    a similar range to the logistic regression models' test AUC
    (~0.99).

  Brier score
    Mean squared error between predicted prob and actual 0/1
    outcome. Lower is better, 0 = perfect. With no scale_pos_weight,
    predictions are fit to the true ~5-8% win rate directly, so this
    should land in a range comparable to Model 2's test Brier
    (0.0144) rather than the inflated values you'd get with
    reweighting (cf. Model 1's 0.0507).

  Calibration plot
    Test predictions binned into 10 quantile groups; mean predicted
    prob vs. actual win fraction per bin. With no class reweighting,
    this should sit close to the diagonal -- a model trained with
    scale_pos_weight set would tend to show predictions skewed high
    (overconfident) by comparison.

  Feature importances (not coefficients/odds ratios)
    XGBoost importances reflect how often/usefully a feature was
    used to split across all trees -- a relative score summing to 1,
    with no direction (no "increases" vs "decreases" win odds).
    If place_x_frac dominates here, that echoes the logistic
    regression coefficient tables -- current position becoming most
    informative late in the race -- but trees capture that via
    splits rather than an explicit interaction term.

  Learning curve / overfitting diagnostics
    Train AUC climbs toward ~1.0 as trees are added; test AUC rises
    then typically flattens or declines -- the gap is the
    overfitting signal. 'Best tree' is where test AUC peaks. A
    gap > 0.01 suggests meaningful overfitting. If the gap is large,
    re-running with n_estimators = best_tree (or adding
    early_stopping_rounds) would be a natural tightening.

  Comparing to the logistic regression models
    Test AUC/Brier here are the numbers to set alongside Model 2's
    (0.9925 / 0.0144) -- both are "no class reweighting" models, so
    it's a fair comparison of XGBoost vs. logistic regression on
    equal footing. If XGBoost's test AUC is similar but Brier is
    notably better (or worse), that's the headline finding for
    whether the added model complexity is worth it here.
""")

# ── Interactive moto viewer ───────────────────────────────────────────────────
viewer_base = model_base.copy()
viewer_base["round"] = viewer_base["round"].astype(str)
viewer_base["moto"] = viewer_base["moto"].astype(str)
viewer_base["year"] = viewer_base["year"].astype(str)

viewer_years = sorted(viewer_base["year"].unique())
viewer_rounds = sorted(viewer_base["round"].unique(), key=lambda x: int(x))
viewer_motos = sorted(viewer_base["moto"].unique(), key=lambda x: int(x))

year_sel_wp = widgets.Dropdown(
    options=viewer_years, value=viewer_years[-1],
    description="Year:", layout=widgets.Layout(width="200px")
)
class_sel_wp = widgets.Dropdown(
    options=["450", "250", "WMX"], value="450",
    description="Class:", layout=widgets.Layout(width="200px")
)
round_sel_wp = widgets.Dropdown(
    options=viewer_rounds, value=viewer_rounds[0],
    description="Round:", layout=widgets.Layout(width="200px")
)
moto_sel_wp = widgets.Dropdown(
    options=viewer_motos, value=viewer_motos[0],
    description="Moto:", layout=widgets.Layout(width="200px")
)
rider_sel_wp = widgets.Dropdown(
    options=[], value=None,
    description="Rider:", layout=widgets.Layout(width="280px")
)


def refresh_wp_riders(*args):
    mask = (
            (viewer_base["year"] == year_sel_wp.value) &
            (viewer_base["class_label"] == class_sel_wp.value) &
            (viewer_base["round"] == round_sel_wp.value) &
            (viewer_base["moto"] == moto_sel_wp.value)
    )
    riders = sorted(viewer_base[mask]["name"].dropna().unique().tolist())
    current = rider_sel_wp.value
    rider_sel_wp.options = riders
    rider_sel_wp.value = current if current in riders else (riders[0] if riders else None)


for w in [year_sel_wp, class_sel_wp, round_sel_wp, moto_sel_wp]:
    w.observe(refresh_wp_riders, names="value")

refresh_wp_riders()

btn_wp = widgets.Button(description="Update", button_style="primary")
output_wp = widgets.Output()


def plot_win_prob(year_val, class_val, round_val, moto_val, rider):
    if not rider:
        print("No riders found.")
        return

    mask = (
            (viewer_base["year"] == year_val) &
            (viewer_base["class_label"] == class_val) &
            (viewer_base["round"] == round_val) &
            (viewer_base["moto"] == moto_val)
    )
    moto_data = viewer_base[mask].copy()
    rider_data = moto_data[moto_data["name"] == rider].sort_values("lap")

    if rider_data.empty:
        print(f"No data for {rider}.")
        return

    fig = go.Figure()

    for other in moto_data["name"].unique():
        if other == rider:
            continue
        od = moto_data[moto_data["name"] == other].sort_values("lap")
        if od["win_prob"].isna().all():
            continue
        fig.add_trace(go.Scatter(
            x=od["lap"], y=od["win_prob"],
            mode="lines",
            line=dict(color="lightgrey", width=1),
            showlegend=False,
            hoverinfo="skip",
        ))

    fig.add_trace(go.Scatter(
        x=rider_data["lap"],
        y=rider_data["win_prob"],
        mode="lines+markers",
        name=rider,
        line=dict(color="#1A7FE8", width=3),
        marker=dict(size=7),
        customdata=rider_data[[
            "place", "behind_time", "frac_remaining"
        ]].values,
        hovertemplate=(
            f"<b>{rider}</b><br>"
            "Lap: %{x}<br>"
            "Win prob: %{y:.1%}<br>"
            "Place: %{customdata[0]:.0f}<br>"
            "Gap to leader: %{customdata[1]:.2f}s<br>"
            "Race remaining: %{customdata[2]:.1%}"
            "<extra></extra>"
        ),
    ))

    actual_finish = int(rider_data["finish_position"].iloc[0])
    suffix = (
        "st" if actual_finish == 1 else
        "nd" if actual_finish == 2 else
        "rd" if actual_finish == 3 else "th"
    )
    fig.add_annotation(
        x=rider_data["lap"].max(),
        y=rider_data["win_prob"].dropna().iloc[-1] if rider_data["win_prob"].notna().any() else 0,
        text=f"  Finished {actual_finish}{suffix}",
        showarrow=False,
        font=dict(color="#1A7FE8", size=11),
        xanchor="left",
    )

    track = rider_data["track"].iloc[0]
    fig.update_layout(
        title=f"Win Probability — {rider} | {class_val} | {year_val} | Rd {round_val} | Moto {moto_val} | {track}",
        xaxis=dict(title="Lap", dtick=1),
        yaxis=dict(
            title="Win Probability",
            range=[0, 1],
            tickformat=".0%",
        ),
        height=500, width=950,
        margin=dict(l=60, r=60, t=60, b=60),
        legend=dict(title=""),
        hovermode="x unified",
    )
    display(fig)


def update_wp(_):
    output_wp.clear_output(wait=True)
    with output_wp:
        plot_win_prob(
            year_sel_wp.value, class_sel_wp.value,
            round_sel_wp.value, moto_sel_wp.value,
            rider_sel_wp.value,
        )


btn_wp.on_click(update_wp)

display(widgets.VBox([
    widgets.HBox([year_sel_wp, class_sel_wp, round_sel_wp, moto_sel_wp]),
    rider_sel_wp,
    btn_wp,
    output_wp,
]))

update_wp(None)

  Win Probability Model — XGBoost (Model 5)

  Training set (2024): 16,217 observations
  Test set     (2025): 15,720 observations
  Positive rate train: 0.036
  Positive rate test:  0.038

  Metric                         Train       Test
  ---------------------------------------------
  ROC-AUC                       0.9968     0.9911
  Brier Score                   0.0085     0.0130

  Overfitting diagnostics:
  Best test AUC: 0.9921 at tree 7 of 300
  AUC gap at best tree:  0.0014  (train - test)
  AUC gap at final tree: 0.0057  (train - test)
  Note: gap > 0.01 suggests meaningful overfitting


  Feature Importances:



,Feature,Importance
1,place,0.6537
2,behind_time,0.2600
3,behind_x_frac,0.0401
4,frac_remaining,0.0247
5,place_x_frac,0.0215


Model Comparison:


Model,Train AUC,Test AUC,Train Brier,Test Brier
Model 1 — Logistic Regression,0.990900,0.991800,0.047800,0.050700
Model 2 — LR w/ No class weighting,0.990400,0.992500,0.013500,0.014400
Model 3 — LR w/o behind time,0.986300,0.993100,0.014900,0.015400
Model 4 — LR w/ Platt scaling,0.990000,0.992500,0.013200,0.014100
Model 5 — XGBoost,0.996800,0.991100,0.008500,0.013000



  Notes: Interpreting the metrics above

  Positive rate
    Share of lap-rows belonging to the eventual moto winner.
    Roughly tracks 1 / (avg field size), so train and test rates
    being close is a good sanity check on the target.

  scale_pos_weight (not used in this model)
    XGBoost's scale_pos_weight reweights the loss so winners and
    non-winners contribute equally -- same effect as
    class_weight="balanced" in Model 1. As shown in the Model 1 ->
    Model 2 comparison, that kind of reweighting leaves AUC mostly
    unchanged but pushes predicted probabilities away from the true
    ~5-8% base rate, hurting Brier and calibration. This model
    leaves scale_pos_weight at its default (1, no reweighting), so
    win_prob should track the real base rate directly. The neg/pos
    ratio is printed above for reference only -- it isn't applied
    anywhere.

  ROC-AUC
    P(a random winner-lap ranks above a random non-winner-lap).
    0.5 = random, 1.0 = perfect ranking. Clas

In [48]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import plotly.graph_objects as go
import pandas as pd
import numpy as np
from itertools import combinations
import warnings
warnings.filterwarnings("ignore")

# ── Build pairwise comparisons ────────────────────────────────────────────────
bt_base = (
    df.dropna(subset=["lap", "place", "finish_position", "race_id"])
    .copy()
)
bt_base["lap"]   = bt_base["lap"].astype(float)
bt_base["place"] = bt_base["place"].astype(float)
bt_base["year"]  = bt_base["year"].astype(str)

def build_pairwise(data):
    rows = []
    for (race_id, lap), group in data.groupby(["race_id", "lap"], observed=True):
        riders = group[["name", "place"]].dropna().values.tolist()
        if len(riders) < 2:
            continue
        for (name1, place1), (name2, place2) in combinations(riders, 2):
            if place1 < place2:
                rows.append({"winner": name1, "loser": name2})
            elif place2 < place1:
                rows.append({"winner": name2, "loser": name1})
    return pd.DataFrame(rows)

# ── Fit Bradley-Terry via iterative scaling ───────────────────────────────────
def fit_bradley_terry(pairs, n_iter=1000, tol=1e-8):
    riders  = sorted(set(pairs["winner"]) | set(pairs["loser"]))
    n       = len(riders)
    idx     = {r: i for i, r in enumerate(riders)}
    wins    = np.zeros(n)
    for _, row in pairs.iterrows():
        wins[idx[row["winner"]]] += 1

    comp_counts = np.zeros((n, n))
    for _, row in pairs.iterrows():
        i = idx[row["winner"]]
        j = idx[row["loser"]]
        comp_counts[i, j] += 1
        comp_counts[j, i] += 1

    strengths = np.ones(n)
    for iteration in range(n_iter):
        old = strengths.copy()
        for i in range(n):
            denom = 0.0
            for j in range(n):
                if i != j and comp_counts[i, j] > 0:
                    denom += comp_counts[i, j] / (strengths[i] + strengths[j])
            if denom > 0:
                strengths[i] = wins[i] / denom
        strengths = strengths / strengths.sum() * n
        if np.max(np.abs(strengths - old)) < tol:
            break

    return {r: strengths[idx[r]] for r in riders}

# ── Fit models for both years ─────────────────────────────────────────────────
strengths_by_year = {}
pairs_by_year     = {}

for year in ["2024", "2025"]:
    print(f"Building pairwise comparisons for {year}...")
    pairs = build_pairwise(bt_base[bt_base["year"] == year])
    pairs_by_year[year] = pairs
    print(f"  {len(pairs):,} pairwise comparisons")
    strengths_by_year[year] = fit_bradley_terry(pairs)
    print(f"  {len(strengths_by_year[year])} riders fitted\n")

# ── Build strength tables ─────────────────────────────────────────────────────
def build_strength_table(year):
    strengths  = strengths_by_year[year]
    year_data  = bt_base[bt_base["year"] == year]

    strength_df = pd.DataFrame([
        {"name": rider, "bt_strength": round(s, 4)}
        for rider, s in strengths.items()
    ]).sort_values("bt_strength", ascending=False).reset_index(drop=True)

    win_rate = (
        year_data.drop_duplicates(subset=["race_id", "name"])
        .groupby("name", observed=True)
        .agg(
            motos=("race_id", "count"),
            wins=("finish_position", lambda x: (x == 1).sum()),
        )
        .reset_index()
    )
    win_rate["win_rate"] = (win_rate["wins"] / win_rate["motos"]).round(3)

    strength_df = strength_df.merge(
        win_rate[["name", "motos", "wins", "win_rate"]],
        on="name", how="left"
    )
    strength_df.index = range(1, len(strength_df) + 1)
    return strength_df

# ── Filter options ─────────────────────────────────────────────────────────────
bt_classes = ["450", "250", "WMX"]

# ── Widgets ───────────────────────────────────────────────────────────────────
year_sel_bt = widgets.Dropdown(
    options=["2024", "2025"], value="2024",
    description="Year:", layout=widgets.Layout(width="200px")
)
class_sel_bt = widgets.Dropdown(
    options=bt_classes, value="450",
    description="Class:", layout=widgets.Layout(width="200px")
)

btn_bt    = widgets.Button(description="Update", button_style="primary")
output_bt = widgets.Output()

# ── Plot function ─────────────────────────────────────────────────────────────
def plot_strength(year_val, class_val):
    strength_df = build_strength_table(year_val)
    year_data   = bt_base[bt_base["year"] == year_val]

    class_riders = set(
        year_data[year_data["class_label"] == class_val]["name"].unique()
    )
    subset = strength_df[
        strength_df["name"].isin(class_riders)
    ].head(30).copy().reset_index(drop=True)
    subset.index += 1

    if subset.empty:
        print("No data for selected filters.")
        return

    fig = go.Figure()
    fig.add_trace(go.Bar(
        y=subset["name"],
        x=subset["bt_strength"],
        orientation="h",
        marker=dict(
            color=subset["bt_strength"],
            colorscale="Blues",
            showscale=False,
        ),
        customdata=subset[["win_rate", "wins", "motos"]].values,
        hovertemplate=(
            "<b>%{y}</b><br>"
            "BT Strength: %{x:.4f}<br>"
            "Win Rate: %{customdata[0]:.1%}<br>"
            "Wins: %{customdata[1]:.0f} / %{customdata[2]:.0f} motos"
            "<extra></extra>"
        ),
    ))

    fig.update_layout(
        title=f"Bradley-Terry Strength Rankings — {class_val} | {year_val}",
        xaxis=dict(title="BT Strength (higher = stronger)"),
        yaxis=dict(
            title="",
            categoryorder="total ascending",
        ),
        height=max(400, len(subset) * 22),
        width=950,
        margin=dict(l=180, r=60, t=60, b=60),
        hovermode="closest",
    )
    display(fig)


# ── Callback ──────────────────────────────────────────────────────────────────
def update_bt(_):
    output_bt.clear_output(wait=True)
    with output_bt:
        plot_strength(year_sel_bt.value, class_sel_bt.value)

btn_bt.on_click(update_bt)

display(widgets.VBox([
    widgets.HBox([year_sel_bt, class_sel_bt, btn_bt]),
    output_bt,
]))

update_bt(None)

# ── Notes: Interpreting the rankings above ─────────────────────────────────────
print("\n" + "=" * 60)
print("  Notes: Interpreting the Bradley-Terry rankings above")
print("=" * 60)
print("""
  What's being compared
    For every lap of every race, every pair of riders present is
    compared: whoever has the lower 'place' value "wins" that
    pairwise comparison. This happens at EVERY lap, not just the
    finish -- so a rider who leads laps 1-10 before fading to 5th
    still racks up "wins" against the field for those laps. BT
    strength measures "how often was this rider running ahead of
    others, at any point in the season" -- closer to an on-track
    pace/position metric than a pure race-results metric.

  Bradley-Terry model
    A classic pairwise-comparison model: P(i beats j) =
    strength_i / (strength_i + strength_j). Strengths are fit via
    iterative scaling -- an iterative MLE that converges on the
    strengths best explaining the observed win/loss pattern.
    Strengths are normalized to sum to n (rider count), so the
    average rider has strength 1.0; >1 = stronger than average,
    <1 = weaker.

  Reading the strength values
    Strength is a ratio scale, not a percentage. A rider with
    strength 2.0 is expected to win a pairwise comparison against a
    strength-1.0 rider roughly 2/(2+1) = 67% of the time -- not
    "twice as good" in any absolute sense.

  BT strength vs. win rate
    These can diverge. A rider who's consistently 2nd-3rd (often
    ahead of most of the field, rarely first) can show high BT
    strength but modest win_rate. A rider who's mid-pack most laps
    but occasionally wins from a late charge could show the reverse.
    Both are in the table so you can spot riders where these tell
    different stories.

  *** Cross-class comparisons -- read with caution ***
    Pairwise comparisons only happen within a single race_id/lap,
    and riders only share a race_id with others in their own class
    (450/250/WMX riders never appear in the same race). So the
    comparison graph is really 3 separate, disconnected groups per
    year. The sum-to-n normalization puts every rider on the same
    numeric scale, but there's no actual head-to-head data linking
    e.g. the top 450 rider's strength to the top 250 rider's -- so
    "450 rider X (2.1) > 250 rider Y (1.8)" isn't backed by any real
    comparison between X and Y. Within-class rankings ARE meaningful;
    cross-class strength comparisons are not.

  Sample size / field size differences
    Larger fields produce more pairwise comparisons per lap (a
    20-rider field yields C(20,2)=190 comparisons per lap vs. 45 for
    a 10-rider field), so larger classes likely have more stable
    estimates. Riders with very few motos will have noisier strength
    values regardless of class.

  A note on independence
    Comparisons from consecutive laps in the same race are highly
    correlated -- if rider A is ahead of B on lap 1, they're usually
    still ahead on lap 2. The raw comparison count overstates how
    much independent information is really there, so treat strength
    differences as directional/relative rather than precise.

  2024 vs 2025
    Fit completely separately, with no shared scale between years --
    same disconnected-graph reasoning as the cross-class note above.
    A rider's 2024 and 2025 strengths aren't directly comparable to
    each other, only their relative position within each year's
    rankings.

  No train/test split or comparison table here
    Unlike the win-probability models (1-5), this isn't a predictive
    model validated against held-out data -- it's a descriptive
    ranking of on-track competitiveness for a given year/class. AUC,
    Brier, and the model_results comparison table don't apply.
""")

Building pairwise comparisons for 2024...
  239,319 pairwise comparisons
  242 riders fitted

Building pairwise comparisons for 2025...
  220,886 pairwise comparisons
  232 riders fitted




  Notes: Interpreting the Bradley-Terry rankings above

  What's being compared
    For every lap of every race, every pair of riders present is
    compared: whoever has the lower 'place' value "wins" that
    pairwise comparison. This happens at EVERY lap, not just the
    finish -- so a rider who leads laps 1-10 before fading to 5th
    still racks up "wins" against the field for those laps. BT
    strength measures "how often was this rider running ahead of
    others, at any point in the season" -- closer to an on-track
    pace/position metric than a pure race-results metric.

  Bradley-Terry model
    A classic pairwise-comparison model: P(i beats j) =
    strength_i / (strength_i + strength_j). Strengths are fit via
    iterative scaling -- an iterative MLE that converges on the
    strengths best explaining the observed win/loss pattern.
    Strengths are normalized to sum to n (rider count), so the
    average rider has strength 1.0; >1 = stronger than average,
    <1 = weak

<a id="rider-profiles"></a>
## 18. Rider Profiles


In [57]:
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display
import plotly.graph_objects as go

# ── Build rider profiles base from df ────────────────────────────────────────
df_prof = df.copy()
df_prof['frac_bin'] = ((1 - df_prof['frac_remaining']) * 10).clip(0, 9.9999).fillna(0).astype(int)

BIN_LABELS = ['0–10%', '10–20%', '20–30%', '30–40%', '40–50%',
              '50–60%', '60–70%', '70–80%', '80–90%', '90–100%']

# ── Pre-compute moto-level results & standings ────────────────────────────────
moto_results = (
    df.groupby(['name', 'class_label', 'year', 'round', 'moto'], observed=True)
    [['finish_position', 'points']].first()
    .reset_index()
)
moto_results = moto_results.sort_values(['round', 'moto'])
moto_results['cum_points'] = (
    moto_results.groupby(['name', 'class_label', 'year'], observed=True)['points']
    .cumsum()
)

standing_rows = []
for (cl, yr), grp in moto_results.groupby(['class_label', 'year'], observed=True):
    grp = grp.sort_values(['round', 'moto']).copy()
    grp['round_int'] = grp['round'].astype(int)
    grp['moto_int'] = grp['moto'].astype(int)
    for (rnd, mto), _ in grp.groupby(['round_int', 'moto_int'], observed=True):
        snapshot = grp[
            (grp['round_int'] < rnd) | ((grp['round_int'] == rnd) & (grp['moto_int'] <= mto))
            ]
        latest = snapshot.groupby('name', observed=True)['cum_points'].max().reset_index()
        latest['standing'] = latest['cum_points'].rank(ascending=False, method='min').astype(int)
        latest['class_label'] = cl
        latest['year'] = yr
        latest['round'] = rnd
        latest['moto'] = mto
        standing_rows.append(latest[['name', 'class_label', 'year', 'round', 'moto', 'standing']])

standings_df = pd.concat(standing_rows, ignore_index=True)
moto_results = moto_results.merge(
    standings_df, on=['name', 'class_label', 'year', 'round', 'moto'], how='left'
)

# ── Build rider list ──────────────────────────────────────────────────────────
eligible_names = sorted(moto_results['name'].dropna().unique().tolist())

# ── Widgets ───────────────────────────────────────────────────────────────────
rider_dd_rp = widgets.Dropdown(
    options=eligible_names,
    description='Rider:',
    layout=widgets.Layout(width='280px')
)
year_dd_rp = widgets.Dropdown(
    description='Year:',
    layout=widgets.Layout(width='160px')
)
class_dd_rp = widgets.Dropdown(
    description='Class:',
    layout=widgets.Layout(width='170px')
)
header_html = widgets.HTML()
charts_out = widgets.Output()

CLASS_COLORS = {'450': '#1A7FE8', '250': '#1AE87F', 'WMX': '#E8641A'}
CLASS_BG = {'450': '#E6F1FB', '250': '#EAF3DE', 'WMX': '#FBEAF0'}
CLASS_TEXT = {'450': '#0C447C', '250': '#27500A', 'WMX': '#72243E'}

TRAFFIC_STATE_COLORS = {
    'Sandwiched': '#9B27AE',
    'Chasing': '#E8641A',
    'Being Chased': '#E8C11A',
    'Clear Air': '#1A7FE8',
}


# ── Cascade helpers ───────────────────────────────────────────────────────────
def _pause(dd):
    try:
        dd.unobserve(render_profile_rp, names='value')
    except ValueError:
        pass


def _resume(dd):
    dd.observe(render_profile_rp, names='value')


def update_year_options_rp(change=None):
    name = rider_dd_rp.value
    avail_years = sorted(df_prof[df_prof['name'] == name]['year'].unique().tolist())
    _pause(year_dd_rp)
    _pause(class_dd_rp)
    year_dd_rp.options = avail_years
    if year_dd_rp.value not in avail_years:
        year_dd_rp.value = avail_years[-1]
    _resume(year_dd_rp)
    update_class_options_rp()


def update_class_options_rp(change=None):
    name = rider_dd_rp.value
    yr = year_dd_rp.value
    if yr is None:
        return
    avail_classes = sorted(
        df_prof[
            (df_prof['name'] == name) &
            (df_prof['year'] == yr)
            ]['class_label'].unique().tolist()
    )
    _pause(class_dd_rp)
    class_dd_rp.options = avail_classes
    if class_dd_rp.value not in avail_classes:
        class_dd_rp.value = avail_classes[0]
    _resume(class_dd_rp)
    render_profile_rp()


# ── HTML card builder ─────────────────────────────────────────────────────────
def build_header_html(name, cl, mfr, yr, avg_finish, avg_z, avg_pct_off,
                      podiums, n_motos, traffic_pcts):
    col = CLASS_COLORS.get(cl, '#888')
    bg = CLASS_BG.get(cl, '#eee')
    txt = CLASS_TEXT.get(cl, '#333')

    traf_tot = sum(traffic_pcts.values()) or 1
    traf_widths = {k: v / traf_tot * 100 for k, v in traffic_pcts.items()}

    metrics = [
        ('Avg finish', f'{avg_finish:.1f}', f'{n_motos} motos'),
        ('Avg z-score', f'{avg_z:+.2f}', 'vs field'),
        ('% off best', f'{avg_pct_off:.1f}%', 'avg lap'),
        ('Podiums', str(podiums), f'of {n_motos} motos'),
    ]

    cards_html = ''.join(f'''
        <div style="flex:1;background:#f7f7f5;border:1px solid #e5e5e3;
                    border-radius:8px;padding:12px 16px;">
            <div style="font-size:11px;color:#888;margin-bottom:6px;">{lbl}</div>
            <div style="font-size:22px;font-weight:600;color:#1a1a1a;
                        line-height:1.1;">{val}</div>
            <div style="font-size:11px;color:#aaa;margin-top:4px;">{sub}</div>
        </div>''' for lbl, val, sub in metrics)

    strip_segments = ''.join(
        f'<div style="flex:{traf_widths.get(state, 0)};background:{color};"></div>'
        for state, color in TRAFFIC_STATE_COLORS.items()
    )
    legend_items = ''.join(
        f'''<span style="font-size:11px;color:#555;">
                <span style="color:{color};font-size:13px;">■</span>
                {traffic_pcts.get(state, 0):.0f}% {state.lower()}
            </span>'''
        for state, color in TRAFFIC_STATE_COLORS.items()
    )

    return f'''
    <div style="background:#ffffff;padding:4px;border-radius:13px;">
    <div style="font-family:Arial,sans-serif;border:1px solid #e5e5e3;
                border-radius:12px;overflow:hidden;margin-bottom:6px;">

        <!-- Header -->
        <div style="padding:14px 20px 10px;border-bottom:1px solid #e5e5e3;
                    display:flex;justify-content:space-between;align-items:center;
                    background:#ffffff;">
            <div>
                <span style="font-size:20px;font-weight:700;color:#1a1a1a;">{name}</span>
                <span style="display:inline-block;font-size:11px;font-weight:600;
                             background:{bg};color:{txt};border-radius:4px;
                             padding:2px 8px;margin-left:8px;">{cl}</span>
                <span style="display:inline-block;font-size:11px;
                             background:#f0f0ee;color:#666;border-radius:4px;
                             padding:2px 8px;margin-left:4px;">{mfr}</span>
            </div>
        </div>

        <!-- Metric cards -->
        <div style="display:flex;gap:10px;padding:14px 20px;
                    border-bottom:1px solid #e5e5e3;background:#ffffff;">
            {cards_html}
        </div>

        <!-- Traffic strip -->
        <div style="padding:10px 20px 14px;background:#ffffff;">
            <div style="font-size:11px;color:#888;margin-bottom:6px;">
                Traffic profile
            </div>
            <div style="display:flex;height:6px;border-radius:3px;
                        overflow:hidden;background:#eee;margin-bottom:8px;">
                {strip_segments}
            </div>
            <div style="display:flex;gap:18px;">
                {legend_items}
            </div>
        </div>

    </div>
    </div>'''


# ── Main render ───────────────────────────────────────────────────────────────
def render_profile_rp(change=None):
    charts_out.clear_output(wait=True)
    name = rider_dd_rp.value
    yr = year_dd_rp.value
    cl = class_dd_rp.value
    if yr is None or cl is None:
        return

    col = CLASS_COLORS.get(cl, '#888888')

    mask = (
            (df_prof['name'] == name) &
            (df_prof['class_label'] == cl) &
            (df_prof['year'] == yr)
    )
    sub = df_prof[mask]
    yr_moto = moto_results[
        (moto_results['name'] == name) &
        (moto_results['class_label'] == cl) &
        (moto_results['year'] == yr)
        ].sort_values(['round', 'moto']).copy()

    if len(sub) == 0 or len(yr_moto) == 0:
        header_html.value = ''
        with charts_out:
            print("No data for this selection.")
        return

    avg_finish = sub.groupby(['round', 'moto'], observed=True)['finish_position'].first().mean()
    avg_z = sub['z_score'].mean()
    avg_pct_off = sub['pct_off_best'].mean()
    podiums = int((yr_moto['finish_position'] <= 3).sum())
    n_motos = len(yr_moto)
    mfr = sub['manufacturer'].iloc[0]

    traffic_counts = sub['traffic_state'].value_counts()
    traffic_pcts = {
        state: traffic_counts.get(state, 0) / len(sub) * 100
        for state in TRAFFIC_STATE_COLORS.keys()
    }

    header_html.value = (
            '<div style="max-width:840px;">'
            + build_header_html(name, cl, mfr, yr, avg_finish, avg_z, avg_pct_off,
                                podiums, n_motos, traffic_pcts)
            + '</div>'
    )

    arc = (
        sub.groupby('frac_bin', observed=True)['z_score']
        .mean()
        .reindex(range(10))
    )

    all_motos = (
        moto_results[
            (moto_results['class_label'] == cl) &
            (moto_results['year'] == yr)
            ][['round', 'moto']]
        .drop_duplicates()
        .copy()
    )
    all_motos['round_int'] = all_motos['round'].astype(int)
    all_motos['moto_int'] = all_motos['moto'].astype(int)
    all_motos = all_motos.sort_values(['round_int', 'moto_int'])
    all_motos['x_label'] = all_motos.apply(
        lambda r: f"R{int(r['round_int'])}M{int(r['moto_int'])}", axis=1
    )

    yr_moto['round_int'] = yr_moto['round'].astype(int)
    yr_moto['moto_int'] = yr_moto['moto'].astype(int)
    yr_moto['x_label'] = yr_moto.apply(
        lambda r: f"R{int(r['round_int'])}M{int(r['moto_int'])}", axis=1
    )

    full_index = all_motos[['x_label']].merge(
        yr_moto[['x_label', 'finish_position', 'standing']],
        on='x_label', how='left'
    )
    full_index['standing_filled'] = full_index['standing'].ffill()

    x_labels = full_index['x_label'].tolist()
    finish_vals = full_index['finish_position'].tolist()
    stand_vals_filled = full_index['standing_filled'].tolist()

    max_finish = max([v for v in finish_vals if pd.notna(v)] + [1])
    max_stand = max([v for v in stand_vals_filled if pd.notna(v)] + [1])
    actual_max = max(max_finish, max_stand)
    combined_tick = max(1, round(actual_max / 5))

    AXIS_STYLE = dict(gridcolor='rgba(0,0,0,0.06)', linecolor='rgba(0,0,0,0.15)')

    fig_arc = go.Figure()
    fig_arc.add_trace(go.Scatter(
        x=BIN_LABELS, y=arc.values,
        mode='lines+markers',
        line=dict(color=col, width=2.5),
        marker=dict(size=7, color=col),
        fill='tozeroy',
        fillcolor=hex_to_rgba(col, 0.12),
        showlegend=False,
        hovertemplate='%{x}<br>Z-score: %{y:.2f}<extra></extra>',
    ))
    fig_arc.update_layout(
        title=dict(text='Pace Arc — Avg z-score by race %',
                   font=dict(size=16, color='#1a1a1a'), x=0, xanchor='left'),
        xaxis=dict(**AXIS_STYLE,
                   title=dict(text='Fraction of race completed',
                              font=dict(size=15, color='#1a1a1a')),
                   tickfont=dict(size=13, color='#1a1a1a'), tickangle=30, showgrid=True),
        yaxis=dict(**AXIS_STYLE,
                   title=dict(text='Avg z-score', font=dict(size=15, color='#1a1a1a')),
                   tickfont=dict(size=13, color='#1a1a1a'), showgrid=True),
        height=380, margin=dict(t=50, b=60, l=80, r=30),
        plot_bgcolor='white', paper_bgcolor='white',
        font=dict(family='Arial, sans-serif', color='#1a1a1a'),
    )

    fig_res = go.Figure()
    fig_res.add_trace(go.Scatter(
        x=x_labels, y=finish_vals,
        mode='lines+markers',
        line=dict(color=col, width=2.5),
        marker=dict(size=7, color=col),
        name='Finish Position', connectgaps=False,
        hovertemplate='%{x}<br>Finish: %{y}<extra></extra>',
    ))
    fig_res.add_trace(go.Scatter(
        x=x_labels, y=stand_vals_filled,
        mode='lines+markers',
        line=dict(color='#E8C01A', width=2.5, dash='dash'),
        marker=dict(size=5, color='#E8C01A'),
        name='Championship Standing', connectgaps=True,
        hovertemplate='%{x}<br>Standing: %{y}<extra></extra>',
    ))
    fig_res.update_layout(
        title=dict(text='Results & Championship Standing',
                   font=dict(size=16, color='#1a1a1a'), x=0, xanchor='left'),
        xaxis=dict(**AXIS_STYLE, tickangle=35,
                   tickfont=dict(size=13, color='#1a1a1a'), showgrid=True),
        yaxis=dict(**AXIS_STYLE,
                   title=dict(text='Position', font=dict(size=15, color='#1a1a1a')),
                   tickfont=dict(size=13, color='#1a1a1a'),
                   range=[actual_max + 2, 0.5], dtick=combined_tick, tick0=1, showgrid=True),
        legend=dict(orientation='h', x=0.5, xanchor='center', y=-0.22,
                    font=dict(size=13, color='#1a1a1a')),
        height=380, margin=dict(t=50, b=90, l=80, r=40),
        plot_bgcolor='white', paper_bgcolor='white',
        font=dict(family='Arial, sans-serif', color='#1a1a1a'),
    )

    with charts_out:
        display(fig_arc)
        display(fig_res)


# ── Wire observers & initial render ──────────────────────────────────────────
rider_dd_rp.observe(update_year_options_rp, names='value')
year_dd_rp.observe(update_class_options_rp, names='value')
class_dd_rp.observe(render_profile_rp, names='value')

update_year_options_rp()

display(widgets.HBox([rider_dd_rp, year_dd_rp, class_dd_rp]))
display(header_html)
display(charts_out)

HTML(value='<div style="max-width:840px;">\n    <div style="background:#ffffff;padding:4px;border-radius:13px;…

Output()

<a id="extensions"></a>
## 19. Extensions
- Believe it or not, there is still Canadian motocross data available from years prior to 2024. That can always be pulled for a more complete data picture.
- There is more than just lap time and place data to source. With more attention, throttle percentages, O2 sensors (optimizing the catalytic converter through ideal air-fuel ratios), and further granularity of individual rider data using [LitPro](https://litprolive.com/) could be explored.